# Resonator Coil Characterisation — a real coil, outside any trap

**Standalone.** No imports from any other notebook, so it runs on a bare board.

Every stage from 8 upward measured a resonator whose answer was chosen in software. This one
measures a physical coil, where nothing is known in advance. Three safeguards replace that:

| Replacement | Catches |
|---|---|
| Self-test against the emulator, before and after | Any fault in the instrument. The emulator is a register bit, not a cable, so it can be switched in at any moment |
| The same width measured two ways | A fit that is wrong for a reason the fit cannot see — sloping baseline, stray tone, compression |
| A signal budget computed *and* measured | Coupling and loading problems, which on a bench look exactly like physics |

**Nothing is carried over between runs.** The amplifier's gain, the usable drive, the decimation,
the coarse search step, every plot limit and every feedback sweep span are measured or derived at
run time — a resonator behaves differently in the trap and out of it, warm and cold, so a
constant copied from a previous session is a wrong answer waiting to be believed.

**It cannot tell you anything that needs an ion**, and nothing from Stage 17.

**Before running:** Stage 12 first, while the emulator is still the authority. Then this once with
`COIL_CONNECTED = False`, to catalogue the board's own stray signals — searching a band you have
not surveyed is how a stray signal becomes a discovery. As root: everything reads `/dev/mem`.

| Step | Question |
|---|---|
| 0 | Is the right bitstream loaded, and does everything answer? |
| 1 | Does the instrument still return a known answer? |
| 2 | Are the levels sane, and what else is in the band? |
| 3 | Where is the resonance, and how sharp? |
| 4a–4d | How much gain, how much drive, how much signal, and does it match L and C? |
| 5 | Does a second, independent method agree? |
| 6 | Is the measurement linear? |
| 7 | Does anything move while I measure? |
| 8 | Does feedback change Q and centre as it should? |
| 9 | Summary, and the constants later notebooks need |


---
# Wiring

```
                 Red Pitaya STEMlab 125-14
      +--------------------------------------------+
      |   OUT1  (SMA)                   IN1  (SMA) |
      +-----|--------------------------------|-----+
            |                                |
       [8 dB pad]                            |  short coax
            |                                |
      [ZFL-500LN+]  28 dB, +15 V             |
            |                                |
       [6 dB pad]    high-gain campaign       |
            |                                |
       [ZFL-750+]   18 dB, +15 V             |
            |                                |
      [termination tee]---[50 ohm load]      |
            |  short coax                    |
         C_drive                          C_pick
         0.3 pF                           0.3 pF
            |    +----------------------+    |
            +----|   the coil / tank    |----+
                 +----------------------+
                            |
                     C_parallel 39 pF
                            |
                     board ground (SMA shells)
```

**OUT1** carries the drive and the feedback, already summed, through the amplifier; **IN1**
carries everything measured. **OUT2 copies IN1 at unity** for a spectrum analyser or scope;
**IN2** is unused. OUT2 is an output: never connect another source to it. The **E1/E2 headers** must never see the
coil — 3.3 V logic, no protection.

One output cable, because the loop closes inside the chip: `dac_a_sum = asg_dat[0] + pid_dat[0]`
sums the drive and the phase-shifted copy before the converter, saturating rather than wrapping.
The emulator is a multiplexer in front of the phase shifter, so Step 1's self-test works with the
coil still attached.

Input jumper **LV** (±1 V). The input is ~1 MΩ — do not terminate it, and keep the coax short:
every picofarad lands in `C_STRAY_EST`.

## The current limit — the reason this is safe

At 41 MHz a 0.3 pF capacitor is 12.9 kΩ, and OUT1 delivers at most ±1 V, so the power it can put
into *any* load — matched, mismatched, shorted, open — is bounded by

    P_max = V² / (4 X_c) = 1 / (4 × 12850) ≈ 19 µW

No setting of `DRIVE_FRAC`, no feedback gain and no software fault gets past that, because the
limit is the capacitor and not the code. The software guards protect the *measurement* — a
clipped front end still looks vaguely like a resonance. The coil is protected by 0.3 pF.

> **The one change that removes this.** Replace either capacitor with a direct connection, or
> anything in the nanofarad range, and OUT1 sees 50 Ω instead of 13 kΩ: 18 mA and 8 mW, about
> 500× more, and a fine-wire coil can care about that. Recompute `DRIVE_MAX` from the coil's own
> dissipation limit *before* the first run.

## The amplifier

This campaign drive chain is **8 dB → ZFL-500LN+ → 6 dB → ZFL-750+ → tee**.
Both amplifiers use +15 V and share RF ground. The tee's other arm must contain a real 50 Ω
termination rated at least 0.25 W before either amplifier is powered. A through attenuator is
not a termination. After oscilloscope and spectrum-analyser level, harmonic, and compression
checks pass, the interstage pad may be changed to **6 dB**. Keep the OUT1 pad at 8 dB; never
reduce both pads during this campaign.

Two limits, and conflating them is a mistake: **damage** is set by the amplifier's *input* power
(derated 20 dB into an open load) and caps `DRIVE_MAX`; **compression** is set by its *output*
power and caps `DRIVE_LINEAR_MAX`. They differ by the gain — a factor of nine here.
`EXT_GAIN_DB` is a starting guess; Step 4a measures the gain and adopts it.


---
# Configuration

**The only cell meant to be edited**, and these are starting points — the notebook measures and
replaces most of them as it runs. `SEARCH_LO`, `SEARCH_HI` and `DRIVE_FRAC` decide whether Step 3
finds anything at all.


In [ ]:
# ============================ CONFIGURATION ==================================
#  The only cell meant to be edited. These are STARTING POINTS - the notebook
#  measures and replaces most of them as it runs.

COIL_CONNECTED = True     # False = input terminated. Step 2 becomes a survey of
                          #   the board's OWN stray signals and Steps 3-8 are
                          #   skipped. Run it that way ONCE before ever attaching
                          #   the coil.
OUT2_MONITOR_INPUT = True # unity copy of IN1 for scope/spectrum analyser;
                          #   never connect OUT2 to another output
OUT1_PAD_LABEL = '8 dB'
INTERSTAGE_PAD_LABEL = '6 dB'   # this file is only for the verified 6 dB chain
SOURCE_NOTE    = ('bench coil; OUT1 -> %s -> ZFL-500LN+ -> %s -> ZFL-750+ '
                  '-> terminated tee -> 0.3 pF; 0.3 pF pickup; room temperature'
                  % (OUT1_PAD_LABEL, INTERSTAGE_PAD_LABEL))

AUTO_CALIBRATE = True     # Step 4a adopts the measured amplifier gain; Step 4b
                          #   adopts its recommended drive and re-measures the
                          #   resonance at it. False = print advice only, which
                          #   means every later number belongs to a drive that
                          #   was chosen rather than measured.
QUALIFICATION_ONLY = True # diagnose extreme points; skip full map and legacy sweeps
ENVIRONMENT_PROFILE = 'RTP' # 'RTP' or '4K'; only seeds the search, measurements remain authoritative
EXTERNAL_MEASUREMENT = {     # optional manual VNA/signal-generator comparison; never a gate
    'instrument': '', 'f0_hz': float('nan'), 'f0_unc_hz': float('nan'),
    'q': float('nan'), 'q_unc': float('nan'), 'temperature_k': float('nan'),
    'hardware': ''}
HIGH_Q_POINTS_PER_LW = 25
HIGH_Q_WING_LW = 4.0
HIGH_Q_RESONATOR_TAU = 7.0
FIR_TAPS = 309
FIR_SETTLE_GROUP_DELAYS = 4.0
SETTLING_CONVERGENCE_TOL = 0.01
HIGH_K_PAIR_OPTIONAL = 0.60
HIGH_Q_COMPLEX_WIDTH_TOL = 0.05
RUN_HIGH_Q_EMULATOR_VALIDATION = True
HIGH_Q_TEST_CENTRES_HZ = (25.0e6, 40.0e6, 55.0e6)
HIGH_Q_TEST_VALUES = (250.0, 1000.0, 5000.0)
HIGH_Q_TEST_DECIMATIONS = (10, 11, 12)

# ---- where to look for the resonance ---------------------------------------
SEARCH_LO      = 20.0e6   # [Hz] bottom of the search band
SEARCH_HI      = 55.0e6   # [Hz] top. Must stay below 62.5 MHz (Nyquist)
SEARCH_STEP    = (25e3 if ENVIRONMENT_PROFILE == '4K' else 100e3)
                          # [Hz] adaptive starting grid; 4 K starts finer
                          #   whether Step 3 finds anything. It MUST be finer
                          #   than the narrowest plausible linewidth or the
                          #   coarse pass steps straight over the line.
                          #   HALVED AUTOMATICALLY and retried if nothing is
                          #   found, so a colder, higher-Q resonator needs no
                          #   edit here.
SEARCH_RETRIES = 6        # halvings permit sub-10-kHz cryogenic discovery
SEARCH_MIN_DB  = 6.0      # contrast a candidate must clear. NOT sufficient on
                          #   its own - see confirm_candidate() in Step 3.
SEARCH_MIN_CORR = 0.50    # two scans of the candidate must correlate this well.
                          #   THIS is what separates a resonance from noise; the
                          #   contrast figure above cannot.
SEARCH_FIT_MAX_RESID = 0.10
                          # Step 3 refuses a fit worse than this rather than
                          #   handing a non-Lorentzian down to Steps 4-8.

# ---- drive ------------------------------------------------------------------
DRIVE_FRAC     = 0.0032   # generator amplitude, 0..1 of full scale. A starting
                          #   value only: Step 4b measures the usable window and
                          #   replaces this.
DRIVE_LIN_FRACS = [0.5, 0.71, 1.0]     # Step 6 ladder, as FRACTIONS of
                          #   DRIVE_FRAC. Absolute values are a trap: once an
                          #   amplifier is fitted DRIVE_FRAC drops by 40 dB and a
                          #   fixed list would drive it past its rating. Only
                          #   6 dB of span, because a rung 12 dB below an already
                          #   low working drive sits in the noise floor, where the
                          #   fitted "peak" is floor rather than signal.

# ---- the hard ceiling -------------------------------------------------------
DRIVE_MAX_BASE = 0.9      # ceiling with NOTHING fitted. Every amplitude write is
                          #   checked against the derived DRIVE_MAX and refused if
                          #   larger. Must be in (0, 1.0].
assert all(0.0 < x <= 1.0 for x in DRIVE_LIN_FRACS), \
    'DRIVE_LIN_FRACS are fractions of DRIVE_FRAC and must be in (0, 1]'

# ---- receiver ---------------------------------------------------------------
DEC_SEARCH     = 8        # log2(R) during the coarse search. R = 256, +-49 kHz
                          #   clean band, 488 kHz output rate. Wide and quick.
DEC_FIT        = 10       # log2(R) for the careful sweeps. Only a starting point:
                          #   choose_dec() picks the decimation from the measured
                          #   linewidth every time a fit is refined.
N_AVG          = 64       # PEEK register pairs averaged per measurement point

# ---- external amplifier in the DRIVE path -----------------------------------
G_PASSIVE      = 0.0158   # round-trip transfer with NO amplifier fitted. Step 4a
                          #   infers the amplifier's OUTPUT level from what
                          #   arrives at the input, so this must be right. With
                          #   no amplifier declared, Step 4a re-measures it and
                          #   writes it back.
EXT_GAIN_DB    = 36.0     # [dB] initial estimate: 15 -> 6 dB adds nominally 9 dB
                          #   interstage pad. Step 4a replaces this estimate.
EXT_GAIN_SAFETY_DB = 49.2 # [dB] conservative 15 dB-chain bound plus 9 dB
                          #   derive hardware drive limits; never relaxed by a run.
#  TWO SEPARATE LIMITS, and conflating them is a mistake:
#    DAMAGE is set by the amplifier's INPUT power  -> caps DRIVE_MAX
#    COMPRESSION is set by its OUTPUT power        -> caps DRIVE_LINEAR_MAX
#  For this cascade, compression is set by the final ZFL-750+ output.
#  Steps that only need the drive to EXIST (the Step 1 self-test, where the
#  emulator replaces the input) may use the damage limit; steps that measure the
#  coil must stay under the other.
EXT_PAD_DB      = 8.0     # [dB] protection directly between OUT1 and ZFL-500LN+.
INTERSTAGE_PAD_DB = 6.0   # [dB] high-gain value between amplifiers. Included
                          #   in EXT_GAIN_DB, so do not subtract it twice.
SOURCE_NOTE = ('bench coil; OUT1 -> %.0f dB -> ZFL-500LN+ -> %.0f dB -> '
               'ZFL-750+ -> terminated tee -> 0.3 pF; 0.3 pF pickup; room temperature'
               % (EXT_PAD_DB, INTERSTAGE_PAD_DB))
                          # EXT_PAD_DB decides whether self-oscillation can be explored:
                          #   an oscillating loop puts FULL SCALE (+10 dBm) into
                          #   the DAC and gen.amp()'s ceiling does not bound the
                          #   feedback path, so the amplifier sees 10 - EXT_PAD_DB
                          #   dBm, which must stay under AMP_IN_MAX_DBM.
AMP_IN_MAX_DBM  = 5.0     # [dBm] first-stage max input; valid with 50 ohm load.
FIRST_P1DB_DBM  = 7.9     # [dBm] ZFL-500LN+ output compression point.
SECOND_IN_MAX_DBM = 5.0   # [dBm] ZFL-750+ maximum input.
FINAL_P1DB_DBM  = 9.0     # [dBm] ZFL-750+ output 1 dB compression point.
AMP_OUT_MAX_DBM = 6.0     # [dBm] operating ceiling, 3 dB below final P1dB.
# Enter externally measured worst cases before enabling feedback. NaN is a
# deliberate fail-closed state; these cannot be inferred from OUT2.
MEASURED_GAIN_INCREASE_DB = float('nan') # 6 dB versus the 15 dB control
MEASURED_SECOND_INPUT_DBM = float('nan')
MEASURED_FINAL_OUTPUT_DBM = float('nan')
MEASURED_WORST_HARMONIC_DBC = float('nan')
MEASURED_BROADBAND_RISE_DB = float('nan')
SCOPE_NO_CLIPPING_CONFIRMED = False
OSC_PROTECTION_CONFIRMED = False # confirm full-scale OUT1 remains safe
ONSET_SEARCH_ARMED = False       # separately enable only after calibration
TERMINATED_OUTPUT = True  # Must describe a fitted tee AND 50 ohm terminator.
AMP_TEST_START = 0.001    # first drive of the ladder. Safe for any gain up to
                          #   40 dB: OUT1 is -44 dBm here.
AMP_TEST_STEP  = 1.6      # multiplier per rung, about 4 dB
FB_BASE_DRIVE_MAX = 0.0087 # conservative feedback baseline; Step 4b may
                          #   select less, but feedback never adopts more.

# ---- Step 4b: finding the drive window --------------------------------------
DRIVE_SCAN_N   = 7        # rungs in the drive-window scan
FIT_TRUST_TOL  = 0.02     # the emulator's Q must still be recovered this well for
                          #   a signal level to count as fittable

GUARD_NSAMP    = 256      # scope samples per input-level check. The guard runs at
                          #   every swept point, so it dominates a sweep's time.
SETTLE         = 0.02     # [s] wait after any retune, for the filter pipeline

# ---- the coupling and the tank ----------------------------------------------
C_COUPLE       = 0.3e-12  # [F] the coupling capacitors actually fitted
#  A transfer measurement determines (C_COUPLE + C_stray) and nothing else, so
#  the split below rests on C_COUPLE being its marked value and scales with it.
#  Step 4d prints the measurement that would settle it.
C_BOARD_IN     = 11.5e-12 # [F] the board's own input capacitance. Derived, not
                          #   measured - see above.
C_CABLE_PICKUP = 13.0e-12 # [F] the pickup cable. From the two-length experiment:
                          #   doubling it moved the transfer by -3.7 dB, which
                          #   requires 13.0 pF per length. ~13 cm at 100 pF/m.
C_STRAY_EST    = C_BOARD_IN + C_CABLE_PICKUP
                          # [F] the sum is what the transfer constrains.
L_COIL         = 423.5e-9   # [H] coil inductance
L_COIL_ERR     = 11.8e-9    # [H] its uncertainty
C_COIL         = 1.59e-12   # [F] the coil's own self-capacitance
C_COIL_ERR     = 0.06e-12   # [F] its uncertainty

C_PARALLEL     = 39.0e-12   # [F] the capacitor fitted ACROSS the tank to emulate
                            #   the trap's parasitic capacitance. Without it the
                            #   coil sits near 165 MHz, above Nyquist, and is
                            #   invisible to this board.
C_PARALLEL_ERR = 2.0e-12    # [F] its tolerance. A 5% part on 39 pF is +-1.95 pF
                            #   and it DOMINATES the predicted centre frequency -
                            #   25x the coil's own capacitance. If you know the
                            #   part is 1% or 2%, put that in: Step 4d's agreement
                            #   test is only as tight as this.

L_EST          = L_COIL   # [H] used by Step 4c's thermal-noise estimate. None to
                          #   skip that block rather than guess.
R_IN_BOARD     = 1.0e6    # [ohm] the board's input resistance
V_FULLSCALE    = 1.0      # [V] input full scale. 1.0 on LV, 20.0 on HV. Used only
                          #   to print volts instead of counts.
T_COIL         = 293.0    # [K] coil temperature, for the thermal-noise estimate

# ---- Step 7: drift -----------------------------------------------------------
DRIFT_MINUTES  = 5.0      # qualification run; final mapping retains 20 minutes
DRIFT_PERIOD   = 20.0     # [s] between revisits

# ---- Step 8: feedback --------------------------------------------------------
FB_PHASES      = list(range(0, 360, 20))   # [deg] phase axis. Must stay
                          #   symmetric - every phase needs its opposite present,
                          #   because Step 8 subtracts the two.
FB_GAIN_MAX    = 1.90     # ceiling on the feedback coefficient. The register is
                          #   Q2.14 and A x 16384 must fit in int16: at A = 2.0 the
                          #   coefficient wraps to -32768 and the feedback SILENTLY
                          #   INVERTS. PhaseShift.set() refuses 1.99 and above.
FB_GAIN_PROBE  = 1.50     # gain used to FIND the two special phases. High on
                          #   purpose; the input level guard is still checked.
FB_GAIN_FRACS  = [0.10, 0.30, 0.50]
                          # gains for Step 8b as FRACTIONS of the usable ceiling.
                          #   Absolute values get filtered out entirely whenever
                          #   the coupling improves and the ceiling drops.
FB_NAVG        = 96       # averages per point in Step 8. Higher than anywhere
                          #   else: the signature is a few percent of Q, so the
                          #   fit noise has to go below it.

# ---- Step 8c: the gain/phase map --------------------------------------------
FB_MAP_PHASES  = 24       # phase steps across 360 degrees
FB_MAP_GAINS   = 12       # gain steps from zero to the usable ceiling
FB_MAP_NPTS    = 15       # frequency points per pixel
FB_MAP_REPEATS = 3        # used only by the later full-map notebook
                          #   whether the gain/phase sweep is reproducible
FB_MAP_NAVG    = 32       # averages per point. Lower than the 1D sweep: on the
                          #   map the effect is tens of percent, not one percent.
FB_MAP_SETTLE  = 0.010    # [s] per point
FB_OSC_LEVEL   = 0.10     # input fraction, DRIVE OFF, above which the loop is
                          #   judged to be self-oscillating. Those pixels are left
                          #   blank and drawn black, as in the paper.

# ---- Step 8d: the lineshape family, positive and negative feedback ----------
FB_SPEC_FRACS  = [0.0, 0.40] # seed traces only; extremes are tested in Step 8g
                          # fractions of the usable ceiling. NEGATIVE means the
                          #   opposite phase - the coefficient register cannot go
                          #   negative, but phi + 180 is the same thing.
FB_SPEC_NPTS   = 61       # points per lineshape
FB_SPEC_HALFSPAN = 3.5    # sweep half-span in linewidths OF THAT SETTING, not of
                          #   the unfed line. Feedback narrows the line by 1/(1-K),
                          #   so a fixed span puts three points across the
                          #   narrowest curve and the Lorentzian stops looking like
                          #   one. Also sets panel (a)'s x-limits.
FB_SPEC_ZOOM_LW = 2.5     # panel (b)'s half-width, in linewidths of the UNFED
                          #   line. That curve is the reference the whole family
                          #   is quoted against, so the zoom is built around it
                          #   rather than around whichever setting happened to
                          #   come out narrowest.

# ---- Step 8e: how far and how accurately the resonance can be moved ---------
FB_SHIFT_FRACS = [0.2, 0.4, 0.6, 0.8, 0.95]
FB_SHIFT_TESTS = 3        # target shifts to request and then verify

FB_LEVEL_GUARD = 0.80     # abort the point if the input exceeds this fraction of
                          #   full scale. THE REAL PATH HAS NO OVERLOAD FLAG - the
                          #   emulator's sticky CLIPPED bit has quietly been doing
                          #   that job for ten stages and it is gone here.

# ---- Step 1: instrument self-test -------------------------------------------
SELFTEST_F0    = 37.0e6   # deliberately NOT 40 MHz, so a hard-coded 40 MHz
SELFTEST_Q     = 250.0    #   anywhere in the analysis shows up as a failure
SELFTEST_TOL_F = 0.20     # tolerance: fraction of a linewidth on the centre
SELFTEST_TOL_Q = 0.05     # tolerance: fractional error on Q

RESULTS_JSON   = 'bench_coil_6db_results.json' # written by Step 9

FS_CLK         = 125e6    # the one clock everything on the board runs from [Hz]

# ---- derived ceilings -------------------------------------------------------
#  External gain sits outside anything the software can see, so fold it into the
#  ceiling here. With an amplifier fitted, DRIVE_MAX is set by what the AMPLIFIER
#  can survive, not by what the converter can. Step 4a recomputes both from the
#  MEASURED gain.
def derive_limits(gain_db, pad_db, in_max_dbm, out_max_dbm, base=None):
    """(DRIVE_MAX, DRIVE_LINEAR_MAX) for a declared external chain.

    A pad in front of the amplifier raises BOTH limits by its value, and it is
    the only way to give the converter a decent number of counts: the output
    ceiling caps the delivered power either way, so without a pad the DAC ends up
    running at a couple of dozen counts."""
    base = DRIVE_MAX_BASE if base is None else base
    if gain_db <= 0.0:
        return base, base
    dmax = min(base, 10.0 ** ((in_max_dbm + pad_db - 10.0) / 20.0))
    dlin = 10.0 ** ((out_max_dbm - gain_db + pad_db - 10.0) / 20.0)
    return dmax, dlin


DRIVE_MAX, DRIVE_LINEAR_MAX = derive_limits(
    EXT_GAIN_SAFETY_DB, EXT_PAD_DB, AMP_IN_MAX_DBM, AMP_OUT_MAX_DBM)
if EXT_GAIN_DB > 0.0:
    print('external gain %.1f dB expected; %.1f dB safety model: DRIVE_MAX %.4f '
          '(damage), linear limit %.5f (compression)'
          % (EXT_GAIN_DB, EXT_GAIN_SAFETY_DB, DRIVE_MAX, DRIVE_LINEAR_MAX))
    if DRIVE_FRAC > DRIVE_LINEAR_MAX:
        print('  NOTE: DRIVE_FRAC %.5f is above the linear limit. Safe, but the'
              % DRIVE_FRAC)
        print('  amplifier will compress. Step 4b measures the usable window and')
        print('  replaces it, so this is a starting value rather than a problem.')

# The Step 1 self-test drives the EMULATOR, which replaces the digitised input,
# so the amplifier plays no part in what it measures - only the damage limit
# applies. It needs a large signal to hit its 5% tolerance on Q.
SELFTEST_DRIVE = min(0.3, 0.9 * DRIVE_MAX)

assert 0.0 < DRIVE_MAX <= 1.0, 'DRIVE_MAX must be in (0, 1.0]'
assert 0.0 <= DRIVE_FRAC <= DRIVE_MAX, (
    'DRIVE_FRAC %.5f exceeds DRIVE_MAX %.5f. With %.1f dB of external gain the '
    'amplifier would be driven far past its rating. Use about %.5f.'
    % (DRIVE_FRAC, DRIVE_MAX, EXT_GAIN_DB, 0.5 * DRIVE_MAX))
assert TERMINATED_OUTPUT, (
    'STOP: fit the termination tee and 50 ohm load before using this configuration')
assert EXT_PAD_DB >= 6.0, 'OUT1 pad below 6 dB can exceed the first amplifier input limit'
assert INTERSTAGE_PAD_DB >= 3.0, (
    'interstage pad below 3 dB can exceed the ZFL-750+ input limit')
assert AMP_OUT_MAX_DBM <= FINAL_P1DB_DBM - 3.0, (
    'final output ceiling must retain 3 dB compression margin')
assert 0.0 < FB_GAIN_MAX < 1.99, 'FB_GAIN_MAX must be below the 1.99 register limit'
assert 0.0 <= FB_GAIN_PROBE <= FB_GAIN_MAX, 'FB_GAIN_PROBE exceeds FB_GAIN_MAX'
assert all(0.0 < fr <= 1.0 for fr in FB_GAIN_FRACS), \
    'FB_GAIN_FRACS are fractions of the usable ceiling and must be in (0, 1]'
assert len(FB_PHASES) % 2 == 0 and all(((p + 180) % 360) in FB_PHASES for p in FB_PHASES), \
    'FB_PHASES must contain the opposite of every phase it contains'

print('configuration loaded. coil connected: %s' % COIL_CONNECTED)
print('search band %.1f - %.1f MHz, coarse step %.0f kHz (halved up to %d times '
      'if nothing is found)'
      % (SEARCH_LO / 1e6, SEARCH_HI / 1e6, SEARCH_STEP / 1e3, SEARCH_RETRIES))
print('drive chain: %.0f dB OUT1 pad, %.0f dB interstage, terminated output'
      % (EXT_PAD_DB, INTERSTAGE_PAD_DB))
print('final amplifier operating ceiling: %+.1f dBm (3 dB below P1dB)'
      % AMP_OUT_MAX_DBM)
print('auto-calibration: %s' % ('ON - measured gain and drive are adopted'
                                if AUTO_CALIBRATE else 'OFF - advice only'))


---
# Imports, and the one rule that must not be broken

**Every register access is a single 32-bit read or write.** A bulk `mmap` slice becomes a burst
the bridge to the register bus cannot acknowledge: it locks the bus and reboots the board — no
traceback, no log. The loops below look inefficient on purpose.

DDR is different — bulk reads *there* are fine and fast, which is why the capture routine reads
its records in one slice while the register drivers never do.

The two shims exist because this board carries older numpy and matplotlib than a desktop.


In [ ]:
import os, mmap, struct, time, json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt


def s32(v):
    """Reinterpret a 32-bit register value as a signed integer."""
    return v - (1 << 32) if v & 0x80000000 else v


class Bus(object):
    """One register block, accessed strictly one 32-bit word at a time."""

    def __init__(self, base, span=0x1000):
        self.base = base
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m = mmap.mmap(self.fd, span, offset=base)

    def w(self, off, val):
        self.m[off:off + 4] = struct.pack('<I', val & 0xFFFFFFFF)

    def r(self, off):
        return struct.unpack('<I', self.m[off:off + 4])[0]

    def rs(self, off):
        return s32(self.r(off))


# =============================================================================
#  PLOTTING COMPATIBILITY  -  this board runs matplotlib 1.x
# =============================================================================
#  The 'C0'..'C9' colour shorthand arrived in matplotlib 2.0. On 1.x it is not
#  rejected when you pass it - it is accepted, stored, and then raises
#
#      ValueError: to_rgba: Invalid rgba arg "C3"
#
#  from inside the draw, at which point the figure is already built. What you see
#  is a traceback followed by a bare <matplotlib.figure.Figure at 0x...> instead
#  of a plot, and every OTHER cell still runs, so it looks like a plotting
#  nuisance rather than a version problem. Every colour in this notebook is
#  therefore written out in full.
CLR_BLUE   = '#1f77b4'
CLR_ORANGE = '#ff7f0e'
CLR_GREEN  = '#2ca02c'
CLR_RED    = '#d62728'
CLR_GREY   = '0.5'

# The paper's figures use the modern perceptual colour maps; matplotlib 1.x has
# none of them, so viridis is rebuilt here from its own anchors - identical
# output on any version.
VIRIDIS_ANCHORS = [
    (0.2670, 0.0049, 0.3294), (0.2823, 0.0950, 0.4173), (0.2788, 0.1755, 0.4834),
    (0.2590, 0.2515, 0.5247), (0.2297, 0.3224, 0.5457), (0.1994, 0.3876, 0.5546),
    (0.1727, 0.4488, 0.5579), (0.1490, 0.5081, 0.5573), (0.1276, 0.5669, 0.5506),
    (0.1206, 0.6258, 0.5335), (0.1579, 0.6838, 0.5017), (0.2461, 0.7389, 0.4520),
    (0.3692, 0.7889, 0.3829), (0.5160, 0.8312, 0.2943), (0.6785, 0.8637, 0.1895),
    (0.8456, 0.8873, 0.0997), (0.9932, 0.9062, 0.1439)]
CMAP_SEQ = matplotlib.colors.LinearSegmentedColormap.from_list(
    'viridis_local', VIRIDIS_ANCHORS, N=256)
CMAP_DIV = matplotlib.colors.LinearSegmentedColormap.from_list(
    'coolwarm_local',
    [(0.230, 0.299, 0.754), (0.553, 0.690, 0.996), (0.866, 0.866, 0.866),
     (0.968, 0.718, 0.612), (0.706, 0.016, 0.150)], N=256)
CMAP_SEQ.set_bad('black')
CMAP_DIV.set_bad('black')

# One place for the look of every figure. Setting it here means no cell needs to
# repeat grid, font and layout calls.
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 110, 'figure.autolayout': True,
    'font.size': 9, 'axes.titlesize': 10, 'axes.labelsize': 9,
    'legend.fontsize': 8, 'legend.framealpha': 0.9,
    'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'axes.grid': True, 'grid.color': '0.88', 'grid.linewidth': 0.6,
    'axes.axisbelow': True, 'lines.linewidth': 1.2, 'lines.markersize': 4,
})

try:                       # inline figures when running under Jupyter...
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:          # ...and a file-only backend when running as a script
    matplotlib.use('Agg')

print('numpy %s, matplotlib %s' % (np.__version__, matplotlib.__version__))
if int(matplotlib.__version__.split('.')[0]) < 2:
    print('  matplotlib 1.x detected - using explicit hex colours (see above)')


---
# The four hardware blocks

| Block | Base | Role here |
|---|---|---|
| **Acquisition** | `0x40600000` | The receiver: a narrowband voltmeter (registers) and a recorder (memory) |
| **Generator** | `0x40200000` | The stimulus: a stored sine replayed at a chosen rate |
| **Phase shifter** | `0x40300000` | The feedback controller, Step 8 only |
| **Digital resonator** | `0x40400000` | The emulator — a known answer, in Steps 1, 4b and 9a |
| **Scope** | `0x40100000` | Full-rate capture of the raw input, for level checks |

Full register maps are in `docs/acquisition.md` and in each Verilog header.

**The trick that makes every sweep simple:** tune the receiver's local oscillator to *exactly*
the drive frequency, and the signal lands at 0 Hz where the in-phase and quadrature values are
steady numbers readable from two registers at leisure. At any other offset they rotate, and two
register reads land on different samples — an artefact that once convincingly imitated filter
droop in this project. "Exactly" is doable rather than approximate: the generator's frequency
word has 30 bits, the oscillator's 32, and both count the same 125 MHz clock, so an oscillator
word of `4 × step` is the same frequency identically, not to within a rounding error.


In [ ]:
# =============================================================================
#  ACQUISITION SYSTEM - base 0x40600000
# =============================================================================
#  VOLTMETER: tune the local oscillator to the drive, read PEEK_I/PEEK_Q.
#  RECORDER : stream to memory for spectra.
#  ORDER MATTERS: disable -> write settings -> reset -> enable. The decimator
#  holds a running total that assumes a fixed R. Retuning the oscillator is one
#  register write and is safe at any time.
# =============================================================================
class Acq(Bus):

    def __init__(self):
        Bus.__init__(self, 0x40600000)
        self.dec = None

    # ---- setup ----
    def start(self, dec_log2, src=0):
        """Bring the chain up at a given decimation. Safe to call repeatedly."""
        self.w(0x00, 0)                 # disable while settings change
        self.w(0x08, dec_log2)          # decimation R = 2**dec_log2
        self.w(0x0C, src)               # 0 = IN1, 1 = IN2
        self.w(0x00, 1 | 2)             # enable + soft reset (self-clearing)
        self.w(0x00, 1)                 # running
        self.dec = dec_log2

    def stop(self):
        self.w(0x00, 0)

    # ---- tuning ----
    def tune_word(self, finc):
        """Set the local oscillator by its raw 32-bit frequency word."""
        self.w(0x04, int(finc) & 0xFFFFFFFF)

    def tune(self, f):
        """Set the local oscillator to the nearest word to f hertz."""
        self.tune_word(int(round(f / FS_CLK * (1 << 32))))

    # ---- voltmeter ----
    def measure(self, navg=None):
        """Average navg in-phase/quadrature pairs; return one complex number.

        At DC both are steady, so this is coherent averaging: a real signal adds
        linearly while noise averages down as the square root of navg."""
        navg = N_AVG if navg is None else navg
        zi = 0.0
        zq = 0.0
        for _ in range(navg):
            zi += self.rs(0x18)
            zq += self.rs(0x1C)
        return complex(zi / navg, zq / navg)

    # ---- housekeeping ----
    def f_out(self):
        """Output sample rate at the current decimation [Hz]."""
        return FS_CLK / (1 << self.dec)

    def window_hz(self):
        """Half-width of the CLEAN part of the receiver window [Hz].

        The decimator rejects by better than 75 dB only over the middle fifth of
        its output band, so the usable half-width is 0.1 x the output rate."""
        return 0.10 * self.f_out()

    def build_id(self):
        return self.r(0x48)

    def status2(self):
        return self.r(0x38)


acq = Acq()
print('acquisition mapped. BUILD_ID = 0x%08X' % acq.build_id())


In [ ]:
# =============================================================================
#  SIGNAL GENERATOR - base 0x40200000, stock Red Pitaya module
# =============================================================================
#  Replays a 16384-entry table; frequency is one register write.
#      f = step x 125e6 / 2**30      (about 0.116 Hz per step)
#  Registers: 0x04 amplitude (0x2000 = x1.0), 0x08 table size, 0x0C start,
#  0x10 step, table at 0x10000. Control 0x00: 0x11 free-run, 0xC0 held at zero.
# =============================================================================
class Generator(Bus):

    def __init__(self):
        Bus.__init__(self, 0x40200000, span=0x20000)
        self._step = None

    def load_sine(self):
        """Write one cycle of a sine into the table. Takes a second or two:
        16384 single-word writes, and single-word is not negotiable here."""
        for i in range(16384):
            v = int(round(8191.0 * np.sin(2.0 * np.pi * i / 16384.0)))
            self.w(0x10000 + 4 * i, v & 0x3FFF)
        self.w(0x08, (16384 << 16) - 1)     # table size, fixed point, inclusive
        self.w(0x0C, 0)                     # start replaying from entry 0

    def freq_word(self, f):
        """Nearest 30-bit frequency word to f hertz."""
        return int(round(f / FS_CLK * (1 << 30)))

    def set_word(self, step):
        """Set frequency by raw word; return the frequency actually produced."""
        self._step = int(step)
        self.w(0x10, self._step)
        return self._step * FS_CLK / float(1 << 30)

    def freq(self, f):
        """Set frequency in hertz; return the frequency actually produced."""
        return self.set_word(self.freq_word(f))

    def lo_word(self):
        """The 32-bit local-oscillator word that lands this signal EXACTLY at
        0 Hz. The generator counts in units of 2**-30 of the clock and the local
        oscillator in units of 2**-32, so the conversion is a factor of four and
        carries no rounding error at all."""
        return 4 * self._step

    def amp(self, frac):
        """Output amplitude as a fraction of full scale. CLAMPED, deliberately.

        The register is Q1.13 with 0x2000 = x1.0, and it happily accepts up to
        0x3FFF = x1.9999. The write below is masked with & 0x3FFF, so before this
        guard existed the mask turned mistakes into silent, WRONG drive levels:

            frac = -0.5  ->  0x3000  ->  x1.5   (a sign slip drives 50% HARDER)
            frac =  2.0  ->  0x0000  ->  x0.0   (silence, and no complaint)
            frac =  1.5  ->  0x3000  ->  x1.5   (clipped square wave, not a sine)

        Every one of those is silent on the bench. Anything outside
        [0, DRIVE_MAX] is now refused instead, so the only way to drive the coil
        harder is to raise DRIVE_MAX in the configuration cell on purpose."""
        frac = float(frac)
        if not np.isfinite(frac):
            raise ValueError('drive amplitude is not a finite number: %r' % frac)
        if frac < 0.0:
            raise ValueError(
                'negative drive amplitude (%.4f). A sign has no meaning here, and '
                'the register would silently wrap it to x%.4f.'
                % (frac, (int(round(frac * 8192)) & 0x3FFF) / 8192.0))
        if frac > DRIVE_MAX:
            raise ValueError(
                'drive %.4f exceeds DRIVE_MAX = %.4f. Nothing in this notebook '
                'may drive harder than the ceiling; raise it in the configuration '
                'cell if that is really what you want.' % (frac, DRIVE_MAX))
        self.w(0x04, int(round(frac * 8192)) & 0x3FFF)

    def _control_a_b_zero(self, a_control):
        """Write channel-A control while forcing generator channel B to zero.

        OUT2 is the phase-shifter monitor plus ASG channel B. A plain channel-A
        write clears B's control bits, so stale B state from another program could
        corrupt an external measurement. Bits 23:22 hold B at zero/reset."""
        self.w(0x24, 0)                    # B amplitude and DC offset both zero
        self.w(0x00, 0x00C00000 | int(a_control))

    def assert_b_zero(self):
        """Require channel B amplitude/offset zero and its engine held reset."""
        packed, control = self.r(0x24), self.r(0x00)
        if packed != 0 or ((control >> 22) & 3) != 3:
            raise RuntimeError('generator B is not zero: AMP/DC=0x%08X CTRL=0x%08X'
                               % (packed, control))
        return {'b_amp_dc': packed, 'control': control}

    def run(self):
        self._control_a_b_zero(0x40)       # reset the A read pointer
        self._control_a_b_zero(0x11)       # A free-runs; B remains forced zero

    def off(self):
        self.amp(0.0)
        self._control_a_b_zero(0xC0)       # both outputs held at zero/reset


gen = Generator()
gen.off()
gen.load_sine()
print('generator ready, table loaded, output off')


In [ ]:
# =============================================================================
#  PHASE SHIFTER - base 0x40300000   (Step 8 only)
# =============================================================================
#  Output = A cos(phi) x I + A sin(phi) x Q, Q being the input delayed a quarter
#  period. Two corrections, both from the module header:
#   1. The quarter-period delay is under one clock at 40 MHz, so add whole signal
#      periods and pick the k whose fraction sits nearest 0 or 1.
#   2. Fixed THREE-sample latency (L = 3 since FIX 6; one sample is 115 deg at
#      40 MHz, so a stale value rotates the whole axis). Subtracted from the
#      request so the delivered phase is the requested one.
#  Both describe the DIGITAL path only. Step 8 finds its phases by measurement.
#  "Feedback off" is coefficients (0, 0), NOT the bypass bit.
# =============================================================================
class PhaseShift(Bus):

    LATENCY = 3      # datapath latency in samples (L = 3 since FIX 6; see above)

    def __init__(self):
        Bus.__init__(self, 0x40300000)
        self.f_ref = None

    def config_delay(self, f):
        """Programme the quarter-period delay line for frequency f."""
        best = None
        for k in range(0, 9):
            D = FS_CLK / (4.0 * f) + k * FS_CLK / f
            if D < 1.0 or D > 1000.0:
                continue
            frac = D - np.floor(D)
            score = min(frac, 1.0 - frac)       # interpolator error peaks at 0.5
            if best is None or score < best[0]:
                best = (score, int(np.floor(D)), frac)
        if best is None:
            raise ValueError('no usable quarter-period delay at %.3f MHz' % (f / 1e6))
        self.w(0x08, best[1])                       # whole clocks
        self.w(0x0C, int(round(best[2] * 16384)))   # fraction, Q0.14
        self.w(0x10, 0)                             # bypass OFF
        self.f_ref = f
        return best[1], best[2]

    def set(self, A, phi_deg):
        """Apply gain/phase and prove the exact CI/CQ words were accepted."""
        if self.f_ref is None:
            raise RuntimeError('call config_delay() first')
        if not (0.0 <= A < 1.99):
            # A x 16384 must stay inside int16. At A = 2.0 the coefficient is
            # 32768, which wraps to -32768 and silently INVERTS the feedback.
            raise ValueError('gain must be in [0, 1.99)')
        phi = phi_deg - 360.0 * self.LATENCY * self.f_ref / FS_CLK
        ci = int(round(A * np.cos(np.radians(phi)) * 16384)) & 0xFFFF
        cq = int(round(A * np.sin(np.radians(phi)) * 16384)) & 0xFFFF
        self.w(0x00, ci); self.w(0x04, cq)
        if self.readback()['ci'] != ci or self.readback()['cq'] != cq:
            raise RuntimeError('phase/gain coefficient readback failed')
        return ci, cq

    def readback(self):
        """All state that can alter feedback or the OUT2 observation."""
        return {'ci': self.r(0x00) & 0xFFFF, 'cq': self.r(0x04) & 0xFFFF,
                'delay': self.r(0x08), 'delta': self.r(0x0C),
                'bypass': self.r(0x10) & 1, 'monitor': self.r(0x18) & 0x3F}

    def assert_held(self, expected):
        got = self.readback()
        changed = {k: (expected[k], got[k]) for k in expected if got[k] != expected[k]}
        if changed:
            raise RuntimeError('feedback/OUT2 settings changed during replay: %r' % changed)
        return got

    def off(self):
        self.w(0x00, 0)
        self.w(0x04, 0)

    def monitor(self, source='input', shift=0):
        """Route a signal to OUT2 and verify the register accepted it.

        source='input' observes the coil/ADC signal used by feedback; 'output'
        observes the phase-shifted feedback contribution. shift is a saturating
        left shift (0..7). Keep it at zero until the external level is known."""
        sources = {'off': 0, 'input': 1, 'output': 2}
        if source not in sources:
            raise ValueError("monitor source must be 'off', 'input' or 'output'")
        shift = int(shift)
        if not (0 <= shift <= 7):
            raise ValueError('monitor shift must be in [0, 7]')
        value = sources[source] | (shift << 3)
        self.w(0x18, value)
        got = self.r(0x18)
        if (got & 0x3F) != value:
            raise RuntimeError('OUT2 monitor readback failed: wrote 0x%02X, read 0x%08X; check the bitstream' % (value, got))
        return got

    def monitor_clipped(self):
        """True if OUT2 monitor scaling has clipped since configuration."""
        return bool(self.r(0x18) & (1 << 8))

    def monitor_off(self):
        self.monitor('off', 0)


ps = PhaseShift()
ps.off()
if OUT2_MONITOR_INPUT:
    ps.monitor('input', shift=0)
    print('phase shifter mapped, feedback off; OUT2 copies IN1 at unity')
else:
    ps.monitor_off()
    print('phase shifter mapped, feedback off; OUT2 disabled')


In [ ]:
# =============================================================================
#  DIGITAL RESONATOR (THE EMULATOR) - base 0x40400000
# =============================================================================
#  Used only in Step 1. Enabled, it REPLACES the digitised input, so the coil can
#  stay connected and simply be ignored - a known answer is available at any
#  moment without recabling.
#  Coefficients c = r cos(w0) x 2**17, s = r sin(w0) x 2**17, w0 = 2 pi f0/125e6,
#  r = 1 - pi (f0/125e6)/Q. Rounding moves the pole, so the QUANTISED values
#  recovered below are the real known answer.
# =============================================================================
class Emulator(Bus):

    def __init__(self):
        Bus.__init__(self, 0x40400000)
        self.f0_q = None
        self.q_q = None

    def config(self, f0, Q, delay=12, oshift=7):
        """Set the emulated resonance. FOUR coefficients, since 2026-08-07.

        The recursion advances two samples per update - two-way look-ahead, see
        the module header - so the pole it actually realises is the one in
        COEF_C2/COEF_S2: r squared, turned by 2*w0. COEF_C and COEF_S are still
        required, because they carry the drive term, so all four are written.

        The quantised answer returned therefore comes from c2 and s2, since that
        is the pole the hardware has. Comparing the Step 1 self-test against
        anything else would build a permanent error into its tolerance.

        `delay` is the EXTRA delay, on top of the module's own pipeline. That
        pipeline is three clocks since 2026-08-07 - one from the drive leg, one
        from the state output, one from the output-stage register added to close
        the last of the timing - so 12 + 3 = the board's real 15. It was 15 + 0
        before any of this, then 13 + 2. If a future build adds another stage,
        this number moves again, and Step 1 is what will notice."""
        w0 = 2.0 * np.pi * f0 / FS_CLK
        r = 1.0 - np.pi * (f0 / FS_CLK) / Q
        ci = int(round(r * np.cos(w0) * 131072))
        cs = int(round(r * np.sin(w0) * 131072))
        c2 = int(round(r * r * np.cos(2.0 * w0) * 131072))
        s2 = int(round(r * r * np.sin(2.0 * w0) * 131072))
        self.w(0x04, ci & 0x3FFFF)
        self.w(0x08, cs & 0x3FFFF)
        self.w(0x28, c2 & 0x3FFFF)
        self.w(0x2C, s2 & 0x3FFFF)
        self.w(0x0C, delay)
        self.w(0x10, oshift)
        # arctan2 gives 2*w0 modulo 2*pi, and 2*w0 passes pi above 31.25 MHz,
        # so lift it onto the branch nearest the frequency asked for - which is
        # known here, because it is what produced the coefficients.
        r2 = np.hypot(c2, s2) / 131072.0
        two_w0 = np.arctan2(s2, c2)
        two_w0 += 2.0 * np.pi * round((2.0 * w0 - two_w0) / (2.0 * np.pi))
        self.f0_q = (two_w0 / 2.0) / (2.0 * np.pi) * FS_CLK
        self.q_q = np.pi * (self.f0_q / FS_CLK) / (1.0 - np.sqrt(r2))
        #  Prove the write took. COEF_C2 exists only in the four-coefficient
        #  bitstream; an older one does not decode 0x28 and returns zero, and
        #  then the recursion has no pole at all - the emulator becomes a
        #  pass-through with a flat response, which a Lorentzian fitter turns
        #  into a plausible-looking wrong answer rather than an error.
        if ((self.r(0x28) & 0x3FFFF) != (c2 & 0x3FFFF) or
                (self.r(0x2C) & 0x3FFFF) != (s2 & 0x3FFFF)):
            raise RuntimeError(
                'COEF_C2/COEF_S2 did not read back. This notebook writes the '
                'four-coefficient emulator (two-way look-ahead, 2026-08-07); '
                'the bitstream on the board does not have it. Rebuild and '
                'reload. NOTE the opposite mistake is SILENT: an old notebook '
                'on new hardware leaves c2 = s2 = 0, the recursion has no pole, '
                'and Step 1 reports a fitted Q near 1 with a centre outside the '
                'swept band.')
        return self.f0_q, self.q_q

    def enable(self):
        self.w(0x00, 1)

    def disable(self):
        self.w(0x00, 0)

    def clear(self):
        self.w(0x00, 3)          # zero the state and the sticky clip flag

    def enabled(self):
        return bool(self.r(0x00) & 1)

    def clipped(self):
        return bool(self.r(0x14) & 1)


emu = Emulator()
emu.disable()
print('emulator mapped and DISABLED (the board is listening to its real input)')


In [ ]:
# =============================================================================
#  INTERNAL SCOPE - base 0x40100000
# =============================================================================
#  16384 raw samples at up to 125 MSa/s. The only overload guard on the real
#  path: the emulator's sticky clip flag is gone once a real coil is measured.
#  Sees the same signal as the rest of the chain, so with the emulator enabled it
#  shows the emulator.
# =============================================================================
SC_BASE = 0x40100000
SC_SPAN = 0x40000
SC_BUF_A = 0x10000
SC_BUF_B = 0x20000
SC_NMAX = 1 << 14


class Scope(Bus):

    def __init__(self):
        Bus.__init__(self, SC_BASE, span=SC_SPAN)

    def _buf(self, off, n):
        """Read n samples, one 32-bit word at a time (this is a register block,
        not DDR). The buffer holds 14-bit two's-complement values."""
        ba = bytearray(4 * n)
        m = self.m
        for i in range(n):
            o = off + 4 * i
            ba[4 * i:4 * i + 4] = m[o:o + 4]
        v = np.frombuffer(bytes(ba), dtype='<u4').astype(np.int64) & 0x3FFF
        return np.where(v & 0x2000, v - 0x4000, v).astype(float)

    def acquire(self, dec=1, n=SC_NMAX, chan='A'):
        """Capture and return (samples, sample_rate). dec = 1 is every sample."""
        self.w(0x14, dec)
        self.w(0x10, n - 1)
        self.w(0x00, 2)        # reset the capture engine
        self.w(0x00, 1)        # arm
        self.w(0x04, 1)        # trigger now, from software
        t0 = time.time()
        while self.r(0x04) != 0:
            if time.time() - t0 > 1.0:
                break
        time.sleep(0.005)
        off = SC_BUF_A if chan == 'A' else SC_BUF_B
        return self._buf(off, n), FS_CLK / float(dec)

    def peak_frac(self, n=4096):
        """TOTAL peak excursion as a fraction of full scale. 1.0 means clipping.

        Deliberately includes everything - the converter's DC offset, broadband
        noise, out-of-band pickup - because all of it counts towards clipping.
        That makes it the right guard and the WRONG way to measure the size of a
        tone. Use tone_frac() for that."""
        x, _ = self.acquire(dec=1, n=n)
        return float(np.max(np.abs(x))) / 8191.0

    def tone_frac(self, f, n=4096):
        """Amplitude of the component at f alone, as a fraction of full scale.

        WHY THIS EXISTS. peak_frac() answers "how close am I to clipping". It was
        also being used to answer "how much 40 MHz is arriving", and those are
        not the same question. max|x| is inflated by the converter's DC offset
        and by every other thing in the record, so on this board it read about
        four times the true tone amplitude - which then over-predicted the
        feedback loop gain by the same factor.

        A least-squares projection onto cos and sin at the known frequency fixes
        it: the mean is removed first, so DC cannot contribute, and anything not
        at f is orthogonal to both basis vectors and averages away. No window and
        no scalloping loss, because f is known exactly rather than searched."""
        x, fs = self.acquire(dec=1, n=n)
        x = x - np.mean(x)                       # the converter's DC offset
        t = np.arange(len(x)) / fs
        basis = np.column_stack([np.cos(2.0 * np.pi * f * t),
                                 np.sin(2.0 * np.pi * f * t)])
        coef, _, _, _ = np.linalg.lstsq(basis, x, rcond=-1)
        return float(np.hypot(coef[0], coef[1])) / 8191.0


sc = Scope()
print('scope mapped, %d samples per capture' % SC_NMAX)


# =============================================================================
#  ALL_OFF  -  the one call that makes the hardware harmless
# =============================================================================
#  Every step below that raises the drive does so inside a try/finally that ends
#  here, so an exception, a failed fit or an interrupted cell cannot leave the
#  coil being driven with the feedback loop closed. It is also registered to run
#  when the kernel shuts down.
#
#  Safe to call at any time, including twice, including when nothing is running.
#  If anything ever looks wrong on the bench, run all_off() in a fresh cell.
# =============================================================================
def all_off():
    """Silence the drive, open the feedback loop, disable the emulator."""
    for what, fn in (('generator', lambda: gen.off()),
                     ('feedback', lambda: ps.off()),
                     ('OUT2 monitor', lambda: ps.monitor_off()),
                     ('acquisition', lambda: acq.stop()),
                     ('emulator', lambda: emu.disable())):
        try:
            fn()
        except Exception as exc:
            # One block failing to switch off must never stop the others being
            # switched off - the generator is the one that matters.
            print('all_off: could not switch off the %s (%s)' % (what, exc))


import atexit
atexit.register(all_off)
all_off()
if OUT2_MONITOR_INPUT:
    ps.monitor('input', shift=0)
print('all_off() defined and armed - drive, feedback and emulator are all off')
print('OUT2 monitor: %s' % ('IN1 copy at unity' if OUT2_MONITOR_INPUT else 'off'))


---
# Recording to memory

The voltmeter reports one number per tuning; spectra need time records, and those need memory.
This is the Stage 6/7 transport with every safeguard kept:

- **The region comes from the device tree**, not from a constant.
- **The bounds are read back before arming.** The writer is a DMA master: if the stop bound did
  not take, it runs upward through the memory the operating system is using. That is not an
  exception with a traceback — it is a dead board. Two register reads turn the worst failure
  mode here into a sentence.
- **The region is pre-filled with a poison value**, so a torn record cannot masquerade as data.
- **Settle first, record second.** Arming and then sleeping lets the opening milliseconds catch
  the pipeline still flushing the *previous* tuning, which poisons each sweep point with its
  predecessor's power — most visibly the first point, which has maximum leverage on the fit.
- **Wait for the writer.** The sample counter reports words handed *to* it, not words landed.


In [ ]:
# =============================================================================
#  MEMORY CAPTURE
# =============================================================================
def _be32(b, i=0):
    return struct.unpack('>I', b[i * 4:i * 4 + 4])[0]


def _dt_read(path):
    for root in ('/sys/firmware/devicetree/base', '/proc/device-tree'):
        try:
            with open(os.path.join(root, path), 'rb') as fh:
                return fh.read()
        except (IOError, OSError):
            pass
    return None


_blob = _dt_read('memory/reg') or _dt_read('memory@0/reg')
assert _blob and len(_blob) >= 8, 'cannot read the device tree memory node'
_declared_end = _be32(_blob, 0) + _be32(_blob, 1)
assert _declared_end < 512 * 1024 ** 2, \
    'no memory above the declared end - see Test Stage 6 Step 1'

MEM_START = (_declared_end + 0xFFF) & ~0xFFF
MEM_SIZE = 0x00800000                        # 8 MB is far more than needed
MEM_STOP = MEM_START + (MEM_SIZE & ~7) - 8   # LAST address, INCLUSIVE
POISON = 0xDEADBEEF

print('capture region 0x%08X .. 0x%08X inclusive (%d MB)'
      % (MEM_START, MEM_STOP + 7, MEM_SIZE // (1024 * 1024)))


def _poison(nbytes):
    fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
    mm = mmap.mmap(fd, ((nbytes + 4095) // 4096) * 4096, offset=MEM_START)
    try:
        mm[0:nbytes] = np.full(nbytes // 4, POISON, dtype='<u4').tobytes()
    finally:
        mm.close()
        os.close(fd)


def _read_region(nbytes):
    """Bulk read of DDR. try/finally because everything here can raise, and a
    leaked mapping inside a scan loop surfaces much later as an unrelated-
    looking out-of-memory error."""
    fd = mm = None
    try:
        fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        mm = mmap.mmap(fd, ((nbytes + 8191) // 4096) * 4096, offset=MEM_START)
        return mm[0:nbytes]
    finally:
        if mm is not None:
            mm.close()
        if fd is not None:
            os.close(fd)


def capture(lo_word, dec_log2, n_samples, settle=None):
    """Record n_samples of the complex stream. Returns a complex numpy array.

    lo_word is the RAW 32-bit local-oscillator word, so a caller that wants an
    exact zero-frequency landing can pass gen.lo_word() and get it. Raises on
    any transport fault rather than returning plausible garbage."""
    settle = SETTLE if settle is None else settle
    if n_samples * 8 > (MEM_STOP + 8 - MEM_START):
        raise ValueError('record of %d samples exceeds the capture region' % n_samples)

    _poison(n_samples * 8)

    acq.w(0x00, 0)                       # disable while settings change
    acq.w(0x08, dec_log2)
    acq.w(0x0C, 0)
    acq.dec = dec_log2
    acq.tune_word(lo_word)
    acq.w(0x28, MEM_START)
    acq.w(0x2C, MEM_STOP)

    lo_rb, hi_rb = acq.r(0x28), acq.r(0x2C)
    if lo_rb != MEM_START or hi_rb != MEM_STOP:
        acq.w(0x00, 0)
        raise RuntimeError(
            'capture bounds did not take: wrote 0x%08X..0x%08X, read back '
            '0x%08X..0x%08X. Refusing to arm an unbounded memory writer.'
            % (MEM_START, MEM_STOP, lo_rb, hi_rb))

    acq.w(0x00, 1 | 2)                   # run at the new tuning...
    acq.w(0x00, 1)
    time.sleep(settle)                   # ...let the pipeline flush...
    acq.w(0x00, 1 | 16 | 128)            # ...then arm with a fresh writer
    acq.w(0x00, 1 | 16)

    t_record = n_samples * (1 << dec_log2) / FS_CLK
    deadline = time.time() + t_record + 0.5
    while int(acq.r(0x34)) < n_samples and time.time() < deadline:
        time.sleep(0.001)
    acq.w(0x00, 1)                       # disarm
    time.sleep(0.002)                    # let the last burst land

    st2 = acq.r(0x38)
    if st2 & 8:
        raise RuntimeError('memory refused writes - run check_system.py')
    got = min(int(acq.r(0x34)), n_samples)
    if got < n_samples:
        raise RuntimeError('short record: asked %d, got %d' % (n_samples, got))

    buf = _read_region(got * 8)
    u32 = np.frombuffer(buf, dtype='<u4')
    limit = max(8, 0.001 * u32.size)
    if int(np.count_nonzero(u32 == POISON)) > limit:
        # A word still inside the writer's burst buffer is not a lost word. If
        # it is genuinely lost it will still be poison in 20 ms.
        time.sleep(0.02)
        u32 = np.frombuffer(_read_region(got * 8), dtype='<u4')
        if int(np.count_nonzero(u32 == POISON)) > limit:
            raise RuntimeError('unwritten words in the record - transport fault')

    iq = u32.astype(np.int64)
    i = np.where(iq[0::2] & 0x80000000, iq[0::2] - (1 << 32), iq[0::2])
    q = np.where(iq[1::2] & 0x80000000, iq[1::2] - (1 << 32), iq[1::2])
    return i.astype(float) + 1j * q.astype(float)


---
# Analysis

**`sweep()`** steps the generator across a list of frequencies, keeps the receiver pinned exactly
on the drive, and reads a complex number at each point. **Only the magnitude is used** — see
Step 5 for why the phase between points is not a measurement on this board.

**`fit_lorentzian()`** turns `|z|²` against frequency into a centre and a width, with no fitting
library: the *reciprocal* of a Lorentzian power curve is a parabola, so a weighted quadratic fit
recovers everything. It also fits an **additive power floor**, found by search rather than
assumed — a floor absorbed into the parabola inflates the width, and the floor under a coil
behind 0.3 pF is not small.

**`measure_transfer()`** returns both the total input excursion (the clipping guard) and the
component at the drive frequency alone. Different questions; only the second is a transfer.

**`refine_and_fit()`** picks a decimation from a linewidth estimate, sweeps densely and fits.
Step 3 calls it, and Step 4b calls it again once the drive is adopted, so the headline numbers
always belong to the drive the rest of the notebook uses.


In [ ]:
# =============================================================================
#  ANALYSIS
# =============================================================================
def sweep(freqs, drive_frac=None, navg=None, settle=None, level_guard=None):
    """Drive at each frequency and read the complex response at DC.

    Returns (f_actual, z): f_actual is what the generator really produced (its
    word is quantised) and z is complex. Only |z| is a measurement - see Step 5.

    The receiver is tuned by WORD, not by hertz, so the signal lands at exactly
    zero frequency and the two register reads cannot land on different phases of
    a rotating vector."""
    drive_frac = DRIVE_FRAC if drive_frac is None else drive_frac
    navg = N_AVG if navg is None else navg
    settle = SETTLE if settle is None else settle

    gen.amp(drive_frac)                  # refused outright if above DRIVE_MAX
    gen.run()
    f_act = np.zeros(len(freqs))
    z = np.zeros(len(freqs), dtype=complex)
    try:
        for k, f in enumerate(freqs):
            f_act[k] = gen.freq(f)
            acq.tune_word(gen.lo_word())
            time.sleep(settle)
            if level_guard is not None and sc.peak_frac(GUARD_NSAMP) > level_guard:
                raise RuntimeError('input exceeded %.0f%% of full scale at %.6f MHz'
                                   % (100 * level_guard, f_act[k] / 1e6))
            z[k] = acq.measure(navg)
    finally:
        # Whatever ended this sweep - the guard above, a bad register read, a
        # Ctrl-C halfway through a 350-point pass - the drive comes off. Every
        # measurement goes through here, so no failure path can walk away
        # leaving the coil driven.
        gen.amp(0.0)
    return f_act, z


def preflight_drive(f_probe, target=None):
    """Approach the working drive in two steps instead of jumping to it.

    THE FAILSAFE THAT MATTERS. Every other guard assumes EXT_GAIN_DB is right; if
    it is understated, DRIVE_MAX is too permissive and the first full-drive write
    is the one that does the damage. So measure at a tenth of the drive,
    extrapolate, and refuse before the converter ever sees the full value."""
    target = DRIVE_FRAC if target is None else target
    lo = max(target / 10.0, 2.0e-4)
    try:
        gen.freq(f_probe); gen.amp(lo); gen.run(); time.sleep(0.05)
        small = sc.peak_frac(GUARD_NSAMP)
    finally:
        gen.off()
    pred = small * target / lo
    print('preflight: %.4f of full scale at drive %.5f -> %.3f predicted at %.5f'
          % (small, lo, pred, target))
    if pred > 0.90:
        raise RuntimeError(
            'a drive of %.5f would put %.2f of full scale into the input. '
            'Something delivers far more than expected - check EXT_GAIN_DB '
            '(declared %.1f dB) and that the amplifier is in the DRIVE path, '
            'not the pickup path.' % (target, pred, EXT_GAIN_DB))
    if small > 0.60:
        raise RuntimeError('input already at %.2f of full scale at a tenth of '
                           'the working drive - stop and investigate.' % small)
    return pred


def fit_lorentzian(f, p, guard=0.05, minpts=5):
    """Centre, linewidth, peak and baseline from power p against frequency f.

    1/p is a parabola in f for a PURE Lorentzian. Any additive floor is absorbed
    into that parabola and widens the answer, so the floor is found by search:
    for each trial floor, subtract it, fit, rebuild the full model, and keep the
    trial that reproduces the measured points best. Two passes, coarse then fine.

    Points below `guard` of the peak are excluded - their reciprocals are all
    noise - and the rest are weighted by their power, so the well-measured points
    near the top dominate."""
    f = np.asarray(f, dtype=float)
    p = np.asarray(p, dtype=float)
    pmax = float(np.max(p))
    u = f - f.mean()

    def _try(b):
        q = p - b
        keep = q > guard * q.max()
        if int(np.count_nonzero(keep)) < minpts:
            return None
        A, B, C = np.polyfit(u[keep], 1.0 / q[keep], 2, w=q[keep])
        if A <= 0.0:
            return None                              # parabola opens the wrong way
        u0 = -B / (2.0 * A)
        lw = 2.0 * np.sqrt(max(C / A - u0 ** 2, 1e-30))
        p0 = 1.0 / max(C - A * u0 ** 2, 1e-300)
        f0 = f.mean() + u0
        model = p0 / (1.0 + (2.0 * (f - f0) / lw) ** 2) + b
        rms = float(np.sqrt(np.mean(((p - model) / pmax) ** 2)))
        return rms, (f0, lw, p0, b)

    best, best_b = _try(0.0), 0.0
    hi = 0.999 * float(np.min(p))
    if hi > 0.0:
        for b in np.linspace(0.0, hi, 17):
            got = _try(b)
            if got is not None and (best is None or got[0] < best[0]):
                best, best_b = got, b
        w = hi / 16.0
        for b in np.linspace(max(best_b - w, 0.0), min(best_b + w, hi), 17):
            got = _try(b)
            if got is not None and (best is None or got[0] < best[0]):
                best, best_b = got, b
    if best is None:
        _n = int(np.count_nonzero(p > guard * p.max()))
        if _n < minpts:
            raise ValueError('fit failed: only %d points above the %.0f%% guard'
                             % (_n, 100 * guard))
        raise ValueError('fit failed: %d points available but no trial baseline '
                         'gave a peak-shaped curve - this is not a resonance '
                         '(peak %.1f counts)' % (_n, np.sqrt(max(p.max(), 0.0))))
    f0, lw, p0, base = best[1]
    return dict(f0=f0, lw=lw, peak=p0, base=base, resid=best[0], Q=f0 / lw)


def lorentzian(f, fit):
    """The fitted model evaluated anywhere - used by every plot, so no cell has
    to rebuild it and none can rebuild it differently."""
    return fit['peak'] / (1.0 + (2.0 * (np.asarray(f, dtype=float) - fit['f0'])
                                 / fit['lw']) ** 2) + fit['base']


def receiver_policy(lw, weak=False, high_q=False, requested=None):
    """Choose a zero-IF receiver for noise/runtime, not apparent linewidth.

    Generator and LO use the same word at every point, so the wanted tone is
    always at DC: CIC/FIR passband width does not convolve the swept resonance.
    Higher decimation buys coherent SNR but costs FIR settling time."""
    if requested is not None:
        dec = int(requested)
    elif weak or high_q or (np.isfinite(lw) and lw < 0.5*globals().get('LW', lw)):
        dec = 12
    else:
        dec = 10
    return int(min(12, max(8, dec)))


def choose_dec(lw):
    """Compatibility wrapper; new code should call receiver_policy()."""
    return receiver_policy(lw, high_q=(np.isfinite(lw) and lw < 0.5*globals().get('LW', lw)))


def adaptive_settle(lw, dec_log2, multiplier=1.0):
    """Conservative zero-IF filter plus resonator settling time."""
    fout = FS_CLK / float(1 << int(dec_log2))
    fir = FIR_SETTLE_GROUP_DELAYS * 0.5*(FIR_TAPS-1) / fout
    resonator = HIGH_Q_RESONATOR_TAU / (np.pi*max(float(lw), 1.0))
    return float(multiplier * max(SETTLE, fir, resonator))


def adaptive_frequency_grid(f0, lw, points_per_lw=None, wing_lw=None):
    """Dense nonuniform centre plus sparse wings for a stable background fit."""
    ppl = int(HIGH_Q_POINTS_PER_LW if points_per_lw is None else points_per_lw)
    wing = float(HIGH_Q_WING_LW if wing_lw is None else wing_lw)
    lw = max(float(lw), 1.0)
    centre = np.linspace(-1.25, 1.25, int(np.ceil(2.5*ppl))+1)
    left = np.linspace(-wing, -1.25, max(10, ppl//2), endpoint=False)
    right = -left[::-1]
    return float(f0) + np.unique(np.r_[left, centre, right])*lw


def receiver_metadata(dec_log2, lw, settle_s, f=None, fit=None):
    status = int(acq.status2())
    points = 0
    if f is not None and fit is not None:
        points = int(np.sum(np.abs(np.asarray(f)-fit['f0']) <= 0.5*fit['lw']))
    return {'dec_log2': int(dec_log2), 'decimation': int(1 << int(dec_log2)),
            'output_rate_hz': float(FS_CLK/(1 << int(dec_log2))),
            'fir_bypassed': bool(status & 0x4), 'fir_overrun': bool(status & 0x2),
            'dma_overflow': bool(status & 0x1), 'write_error': bool(status & 0x8),
            'status2': status, 'settle_s': float(settle_s),
            'points_per_linewidth': points, 'zero_if_tracked': True}


def half_power_width(f, p, base=None):
    """Interpolated FWHM using a supplied fit floor or the sweep edges."""
    f = np.asarray(f, dtype=float); p = np.asarray(p, dtype=float)
    order = np.argsort(f); f, p = f[order], p[order]
    edge_n = max(2, len(p)//10)
    if base is None:
        base = float(np.median(np.r_[p[:edge_n], p[-edge_n:]]))
    peak_i = int(np.argmax(p)); half = float(base) + 0.5*(p[peak_i]-float(base))
    def _cross(indices):
        for i, j in indices:
            if (p[i]-half)*(p[j]-half) <= 0 and p[j] != p[i]:
                return float(f[i] + (half-p[i])*(f[j]-f[i])/(p[j]-p[i]))
        return None
    left = _cross((i, i+1) for i in range(peak_i-1, -1, -1))
    right = _cross((i, i+1) for i in range(peak_i, len(f)-1))
    return None if left is None or right is None else float(right-left)


def fit_complex_resonator(f, z, seed):
    """Complex pole plus linear complex background, using NumPy only."""
    f = np.asarray(f, dtype=float); z = np.asarray(z, dtype=complex)
    f0, lw = float(seed['f0']), float(seed['lw'])
    span = max(float(np.ptp(f)), lw); best = None
    for frac in (0.30, 0.10, 0.03):
        fgrid = np.linspace(f0-frac*lw, f0+frac*lw, 13)
        lwgrid = np.linspace(max(0.2*lw, (1-frac)*lw), (1+frac)*lw, 13)
        for fc in fgrid:
            x = (f-fc)/span
            for width in lwgrid:
                for sign in (-1.0, 1.0):
                    pole = 1.0/(1.0 + sign*2.0j*(f-fc)/max(width, 1e-30))
                    design = np.column_stack([np.ones(len(f)), x, pole])
                    coef, _, _, _ = np.linalg.lstsq(design, z, rcond=-1)
                    model = np.dot(design, coef)
                    rms = float(np.sqrt(np.mean(np.abs(z-model)**2)) /
                                max(np.max(np.abs(z)), 1e-30))
                    if best is None or rms < best[0]:
                        best = (rms, float(fc), float(width), coef, model, sign)
        _, f0, lw, _, _, _ = best
    return {'f0': f0, 'lw': lw, 'Q': f0/lw, 'resid': best[0],
            'background_constant': best[3][0],
            'background_slope': best[3][1], 'resonance_amplitude': best[3][2],
            'model': best[4], 'pole_sign': best[5]}


def trace_fit_diagnostics(f, p, fit, z=None, groups=5):
    """Independent width, residual vector and delete-group fit uncertainty."""
    f = np.asarray(f, dtype=float); p = np.asarray(p, dtype=float)
    estimates = []
    for group in range(groups):
        keep = (np.arange(len(f)) % groups) != group
        try:
            estimates.append(fit_lorentzian(f[keep], p[keep]))
        except Exception:
            pass
    def _unc(key):
        vals = np.asarray([x[key] for x in estimates], dtype=float)
        return float(np.std(vals, ddof=1)) if len(vals) >= 2 else float('nan')
    scale = max(float(np.max(p)), 1e-30)
    hp = half_power_width(f, p, base=fit.get('base'))
    cfit = fit_complex_resonator(f, z, fit) if z is not None else None
    residual = (p - lorentzian(f, fit)) / scale
    kernel = np.ones(5, dtype=float) / 5.0
    smooth = np.convolve(residual, kernel, mode='valid') if len(residual) >= 5 else residual
    return {'half_power_lw': hp,
            'half_power_disagreement': (abs(hp/fit['lw']-1.0)
                                        if hp is not None else float('inf')),
            'residual': residual,
            'residual_rms': float(np.sqrt(np.mean(residual**2))),
            'residual_max': float(np.max(np.abs(residual))),
            'structured_residual': float(np.sqrt(np.mean(smooth**2))),
            'floor_peak_amp': float(np.sqrt(max(fit['base'], 0.0))/
                                    max(np.sqrt(max(fit['peak'], 0.0)), 1e-30)),
            'complex_fit': cfit,
            'complex_lw_disagreement': (abs(cfit['lw']/fit['lw']-1.0)
                                        if cfit is not None else float('inf')),
            'f0_uncertainty': _unc('f0'), 'lw_uncertainty': _unc('lw'),
            'q_uncertainty': _unc('Q'), 'jackknife_fits': len(estimates)}


def refine_and_fit(f_centre, lw_hint, drive, npts=61, span_lw=2.5, verbose=True):
    """Dense sweep at a decimation chosen from lw_hint, then fit.

    Returns (dec_log2, f, p, fit). Called by Step 3 and AGAIN by Step 4b once the
    drive has been adopted, so the headline numbers always belong to the drive the
    rest of the notebook uses. Nothing about it is specific to this coil, which is
    the point: the same call works warm, cold, in the trap or out of it."""
    dec = receiver_policy(lw_hint, high_q=(lw_hint < 0.5*globals().get('LW', lw_hint)))
    acq.start(dec)
    if verbose:
        print('  zero-IF receiver R = %d, output rate %.1f kHz, FIR corrected; '
              'tracked tone remains at DC' % (1 << dec, acq.f_out() / 1e3))
    grid = adaptive_frequency_grid(f_centre, lw_hint,
                                   points_per_lw=max(HIGH_Q_POINTS_PER_LW, npts//3),
                                   wing_lw=max(HIGH_Q_WING_LW, span_lw))
    use_settle = adaptive_settle(lw_hint, dec)
    f, z = sweep(grid, drive_frac=drive, navg=N_AVG, settle=use_settle, level_guard=0.95)
    gen.amp(0.0)
    p = np.abs(z) ** 2
    return dec, f, p, fit_lorentzian(f, p)


def emulator_high_q_qualification():
    """Exercise zero-IF tracking, FIR/CIC status, density and settling over Q/f0."""
    rows = []
    try:
        ps.off()
        for requested_f0 in HIGH_Q_TEST_CENTRES_HZ:
            for requested_q in HIGH_Q_TEST_VALUES:
                f0q, qq = emu.config(requested_f0, requested_q, delay=12, oshift=7)
                lwq = f0q/qq; emu.clear(); emu.enable()
                for dec in HIGH_Q_TEST_DECIMATIONS:
                    acq.start(dec); settle = adaptive_settle(lwq, dec)
                    grid = adaptive_frequency_grid(f0q, lwq)
                    f,z = sweep(grid, drive_frac=SELFTEST_DRIVE, navg=N_AVG,
                                settle=settle, level_guard=0.95)
                    fit = fit_lorentzian(f,np.abs(z)**2)
                    meta = receiver_metadata(dec,fit['lw'],settle,f=f,fit=fit)
                    row={'f0_known':float(f0q),'q_known':float(qq),'dec_log2':int(dec),
                         'f0_meas':float(fit['f0']),'q_meas':float(fit['Q']),
                         'centre_error_lw':float((fit['f0']-f0q)/lwq),
                         'q_error_frac':float(fit['Q']/qq-1.0),'receiver':meta}
                    if dec == 12:
                        f2,z2=sweep(grid,drive_frac=SELFTEST_DRIVE,navg=N_AVG,
                                    settle=2.0*settle,level_guard=0.95)
                        fit2=fit_lorentzian(f2,np.abs(z2)**2)
                        row['double_settle_q_change']=float(abs(fit2['Q']/fit['Q']-1.0))
                        row['double_settle_f0_change_lw']=float(abs(fit2['f0']-fit['f0'])/lwq)
                    row['ok']=bool(meta['points_per_linewidth']>=HIGH_Q_POINTS_PER_LW and
                        not meta['fir_bypassed'] and not meta['fir_overrun'] and
                        abs(row['centre_error_lw'])<=0.02 and abs(row['q_error_frac'])<=0.03 and
                        row.get('double_settle_q_change',0)<=SETTLING_CONVERGENCE_TOL and
                        row.get('double_settle_f0_change_lw',0)<=SETTLING_CONVERGENCE_TOL)
                    rows.append(row)
    finally:
        emu.disable(); gen.off(); ps.off()
    # Cross-decimation agreement is evaluated per emulated resonance.
    for f0q in sorted(set(round(r['f0_known'],3) for r in rows)):
        group=[r for r in rows if round(r['f0_known'],3)==f0q]
        for qq in sorted(set(round(r['q_known'],3) for r in group)):
            same=[r for r in group if round(r['q_known'],3)==qq]
            spread=np.ptp([r['q_meas'] for r in same])/max(np.mean([r['q_meas'] for r in same]),1e-30)
            for r in same: r['decimation_spread_frac']=float(spread); r['ok']=bool(r['ok'] and spread<=0.03)
    return rows


def measure_transfer(f0, drive):
    """One-way DAC -> coil -> ADC transfer at f0.

    Returns (raw_peak, tone, G_fwd). raw_peak is the TOTAL excursion, inflated by
    the converter's DC offset and everything else in the record - that is the
    right clipping guard and the WRONG way to size a tone. tone is the component
    at f0 alone, and only that is a transfer."""
    try:
        gen.freq(f0); gen.amp(drive); gen.run(); time.sleep(0.05)
        raw = sc.peak_frac(SC_NMAX)
        tone = sc.tone_frac(f0)
    finally:
        gen.off()
    return raw, tone, tone / max(drive, 1e-12)


print('analysis helpers defined')


---
# Step 0 — Is the right thing loaded, and does everything answer?

**Loading an old bitstream looks exactly like success:** the light blinks, registers answer,
measurements look plausible. The build identifier at `0x48` is a hash of the Verilog it was built
from, and zero means `project.ps1 build` did not produce it.

The emulator probe matters for a subtler reason — an older bitstream has a *stub* at that address
that answers every read with zero and looks politely dead, and Step 1 would then compare against
nothing at all and pass.


In [ ]:
# =============================================================================
#  STEP 0 - PREFLIGHT
# =============================================================================
print('=' * 68)
print('STEP 0 - preflight')
print('=' * 68)

fail = []

# --- 0a. build identifier ----------------------------------------------------
bid = acq.build_id()
print('build id                 : 0x%08X' % bid)
if bid == 0:
    fail.append('BUILD_ID is zero - this bitstream was not built by project.ps1 build')
else:
    print('  compare this against bitstream_id.txt, or run check_system.py')

# --- 0b. do the acquisition registers respond? -------------------------------
probe = 0x5A5A5A5A
acq.w(0x04, probe)
back = acq.r(0x04)
print('local oscillator readback: wrote 0x%08X, read 0x%08X' % (probe, back))
if back != probe:
    fail.append('acquisition registers do not read back')

# --- 0c. is the emulator present, or is this a stub? -------------------------
probe18 = 0x15555                     # alternating bits, positive in 18 bits
emu.w(0x04, probe18)
back18 = emu.r(0x04) & 0x3FFFF
print('emulator readback        : wrote 0x%05X, read 0x%05X' % (probe18, back18))
if back18 != probe18:
    fail.append('the emulator is absent or stubbed - Step 1 cannot run')

#  Does it have the FOUR-coefficient map? COEF_C2 at 0x28 exists only in the
#  two-way look-ahead build (2026-08-07). Without it the recursion has no pole
#  and the emulator passes the drive straight through - a silent failure that
#  looks like a broken fit rather than a mismatched bitstream.
emu.w(0x28, probe18)
back28 = emu.r(0x28) & 0x3FFFF
print('emulator COEF_C2 readback: wrote 0x%05X, read 0x%05X' % (probe18, back28))
if back28 != probe18:
    fail.append('no COEF_C2 at 0x28 - this bitstream predates the two-way '
                'look-ahead resonator, but this notebook writes four '
                'coefficients. Rebuild and reload.')
emu.w(0x04, 0)
emu.w(0x28, 0)                 # leave the coefficients as they were found

# --- 0d. is the chain actually producing samples? ----------------------------
acq.start(DEC_FIT)
n0 = acq.r(0x20)
time.sleep(0.10)
n1 = acq.r(0x20)
rate = (n1 - n0) / 0.10
expect = FS_CLK / (1 << DEC_FIT)
print('output rate              : %.0f samples/s (expected %.0f, %+.1f%%)'
      % (rate, expect, 100.0 * (rate / expect - 1.0)))
if not (0.8 * expect < rate < 1.25 * expect):
    fail.append('output sample rate is wrong - check DEC_LOG2 and the clock')

st2 = acq.status2()
print('status2                  : 0x%02X  (overflow=%d corr_overrun=%d '
      'corr_bypassed=%d write_err=%d)'
      % (st2, st2 & 1, (st2 >> 1) & 1, (st2 >> 2) & 1, (st2 >> 3) & 1))
if (st2 >> 2) & 1:
    print('  NOTE: droop corrector bypassed - expected below R = 256 only')

print('-' * 68)
if fail:
    for f in fail:
        print('FAIL: %s' % f)
    raise SystemExit('Step 0 failed - fix these before continuing')
print('PASS - hardware answers and the chain is running')


---
# Step 1 — Does the instrument still return a known answer?

Enabled, the emulator **replaces the digitised input entirely**, so the coil can stay connected
and irrelevant. This check is therefore not a one-way door: it is available at any moment,
including between two real measurements that disagree.

The test sets a centre and a Q the analysis is never told, sweeps, fits, and demands them back.
The comparison is against the **quantised** values — the coefficients are 18-bit integers, so the
emulator's true centre and Q are the ones recovered from the rounded coefficients, and comparing
against the requested numbers would build a permanent error into the tolerance.

`SELFTEST_F0` is deliberately not 40 MHz: a 40 MHz constant baked into the analysis would fail
here rather than quietly agree.


In [ ]:
# =============================================================================
#  STEP 1 - INSTRUMENT SELF-TEST AGAINST A KNOWN ANSWER
# =============================================================================
def instrument_selftest(verbose=True):
    """Switch the emulator in, measure it, and demand the answer it was given.

    Returns a dict. Leaves the emulator DISABLED whatever happens."""
    try:
        f0_q, q_q = emu.config(SELFTEST_F0, SELFTEST_Q, delay=12, oshift=7)
        lw_q = f0_q / q_q
        emu.clear()
        emu.enable()

        acq.start(DEC_FIT)
        ps.off()

        # +-2.5 linewidths is the whole curve, not just its top. SELFTEST_DRIVE
        # is capped at 0.3 to keep the emulator's 25-bit state clear of
        # saturation, and at the amplifier's damage limit when one is fitted.
        grid = f0_q + np.linspace(-2.5, 2.5, 25) * lw_q
        f, z = sweep(grid, drive_frac=SELFTEST_DRIVE)
        gen.amp(0.0)

        if emu.clipped():
            raise RuntimeError('the emulator saturated - lower the self-test drive')

        fit = fit_lorentzian(f, np.abs(z) ** 2)
        err_f = (fit['f0'] - f0_q) / lw_q            # in linewidths
        err_q = fit['Q'] / q_q - 1.0

        ok = (abs(err_f) <= SELFTEST_TOL_F) and (abs(err_q) <= SELFTEST_TOL_Q)
        if verbose:
            print('known answer (quantised) : f0 = %.6f MHz, Q = %.1f, lw = %.1f kHz'
                  % (f0_q / 1e6, q_q, lw_q / 1e3))
            print('measured                 : f0 = %.6f MHz, Q = %.1f'
                  % (fit['f0'] / 1e6, fit['Q']))
            print('error                    : centre %+.3f linewidths, Q %+.2f%%'
                  % (err_f, 100 * err_q))
            print('fit residual             : %.4f of peak' % fit['resid'])
            print('-' * 68)
            print('%s - the instrument %s reproduce an answer it was given'
                  % ('PASS' if ok else 'FAIL', 'can' if ok else 'CANNOT'))
        return dict(ok=ok, f0_known=f0_q, q_known=q_q, f0_meas=fit['f0'],
                    q_meas=fit['Q'], err_f_lw=err_f, err_q_frac=err_q,
                    resid=fit['resid'], f=f, z=z, fit=fit)
    finally:
        emu.disable()
        gen.off()


print('=' * 68)
print('STEP 1 - instrument self-test (emulator in, coil ignored)')
print('=' * 68)
st1_before = instrument_selftest()
print('emulator disabled - the board is listening to its real input again')
HIGH_Q_FILTER_ROWS = (emulator_high_q_qualification()
                      if RUN_HIGH_Q_EMULATOR_VALIDATION else [])
HIGH_Q_FILTER_VALID = bool(HIGH_Q_FILTER_ROWS and all(r['ok'] for r in HIGH_Q_FILTER_ROWS))
if HIGH_Q_FILTER_ROWS:
    print('high-Q zero-IF/FIR/CIC qualification: %s (%d/%d cases)' %
          ('PASS' if HIGH_Q_FILTER_VALID else 'DIAGNOSTIC FAIL',
           sum(r['ok'] for r in HIGH_Q_FILTER_ROWS),len(HIGH_Q_FILTER_ROWS)))
else:
    print('high-Q zero-IF/FIR/CIC qualification: not run')


In [ ]:
# --- Step 1 picture: the known answer and what came back ---------------------
_f = st1_before['f'] / 1e6
_p = np.abs(st1_before['z']) ** 2
_ft = st1_before['fit']
_fine = np.linspace(_f.min(), _f.max(), 400)
_model = _ft['peak'] / (1.0 + (2.0 * (_fine * 1e6 - _ft['f0']) / _ft['lw']) ** 2) + _ft['base']

fig, ax = plt.subplots(figsize=(7.0, 3.6))
ax.plot(_f, _p, 'o', ms=4, label='measured')
ax.plot(_fine, _model, '-', lw=1.4, label='fit')
ax.axvline(st1_before['f0_known'] / 1e6, ls='--', lw=1.0, color='0.4',
           label='known centre')
ax.set_xlabel('drive frequency / MHz')
ax.set_ylabel('power (arb.)')
ax.set_title('Step 1 - emulator with a known answer  (Q asked %.0f, got %.0f)'
             % (st1_before['q_known'], st1_before['q_meas']))
ax.legend(fontsize=8)
plt.show()


---
# Step 2 — Levels, and what else is in the band

| | Why it matters |
|---|---|
| Are the levels sane? | An overloaded input produces a distorted shape that still looks vaguely like a resonance. This project's signature failure |
| What else is in the band? | A clock harmonic near the coil would be found by Step 3 and fitted as a spectacular high-Q resonance |
| How quiet is the input? | The floor decides whether a resonance is findable at all |

**Run this with the input terminated first.** That catalogues the board's *own* stray signals;
run it again with the coil attached and anything new belongs to the coil or the room. Without the
first run there is no way to tell those apart.


In [ ]:
# =============================================================================
#  STEP 2a - INPUT LEVELS (the only overload guard on the real path)
# =============================================================================
print('=' * 68)
print('STEP 2 - input health   [%s]' % SOURCE_NOTE)
print('=' * 68)

gen.off()
ps.off()
ps.monitor_off()
emu.disable()

quiet_frac = sc.peak_frac(SC_NMAX)
print('input peak, generator OFF : %.4f of full scale (%.0f counts)'
      % (quiet_frac, quiet_frac * 8191))

if COIL_CONNECTED:
    try:
        gen.freq(0.5 * (SEARCH_LO + SEARCH_HI))
        gen.amp(DRIVE_FRAC)
        gen.run()
        time.sleep(0.05)
        drive_frac_seen = sc.peak_frac(SC_NMAX)
    finally:
        gen.off()
    print('input peak, generator ON  : %.4f of full scale (%.0f counts)'
          % (drive_frac_seen, drive_frac_seen * 8191))
else:
    drive_frac_seen = float('nan')

print('-' * 68)
if quiet_frac > 0.5:
    print('WARNING: the input is more than half full scale with NO drive.')
    print('         Something large is arriving. Find it before going on -')
    print('         everything downstream assumes a linear front end.')
elif quiet_frac > 0.05:
    print('NOTE: %.1f%% of full scale with no drive. Not dangerous, but the'
          % (100 * quiet_frac))
    print('      survey below should show what it is.')
else:
    print('levels look sane')


In [ ]:
# =============================================================================
#  STEP 2b - BAND SURVEY, generator off
# =============================================================================
#  Step the receiver across the search band and record the power in each
#  window. The step is a fraction of the clean window width so nothing can hide
#  in a gap between windows - the same discipline the adaptive sweep uses.
# =============================================================================
acq.start(DEC_SEARCH)
f_out_s = acq.f_out()
step_s = 0.15 * f_out_s                       # windows overlap generously
survey_f = np.arange(SEARCH_LO, SEARCH_HI + step_s, step_s)

gen.off()
survey_p = np.zeros(len(survey_f))
t0 = time.time()
for k, f in enumerate(survey_f):
    z = capture(int(round(f / FS_CLK * (1 << 32))), DEC_SEARCH, 1024, settle=0.004)
    survey_p[k] = float(np.mean(np.abs(z) ** 2))
print('surveyed %d windows across %.1f - %.1f MHz in %.0f s'
      % (len(survey_f), SEARCH_LO / 1e6, SEARCH_HI / 1e6, time.time() - t0))

# --- report the strongest features, as decibels above the median window ------
med = float(np.median(survey_p))
excess_db = 10.0 * np.log10(np.maximum(survey_p, 1e-300) / max(med, 1e-300))
order = np.argsort(excess_db)[::-1]

print('')
print('median window power      : %.3e  (this is the noise floor)' % med)
print('strongest windows (dB above the median):')
print('    frequency / MHz    excess / dB')
shown = 0
for idx in order:
    if excess_db[idx] < 6.0 or shown >= 10:
        break
    print('    %14.4f     %+8.1f' % (survey_f[idx] / 1e6, excess_db[idx]))
    shown += 1
if shown == 0:
    print('    (nothing more than 6 dB above the median - a clean band)')

fig, ax = plt.subplots(figsize=(9.0, 3.2))
ax.plot(survey_f / 1e6, excess_db, '-', lw=1.0)
ax.axhline(6.0, ls='--', lw=0.9, color='0.5', label='6 dB reporting threshold')
ax.set_xlabel('receiver centre / MHz')
ax.set_ylabel('dB above median')
ax.set_title('Step 2 - band survey, generator off   [%s]' % SOURCE_NOTE)
ax.legend(fontsize=8)
plt.show()

print('')
print('KEEP THIS PLOT. With the input terminated it is the board\'s own spectrum;')
print('with the coil attached, anything NEW is the coil or the room.')


---
# Step 3 — Find the resonance without being told where it is

| Pass | What it does | Why |
|---|---|---|
| **Look around** | Step across the band, few averages, no fitting | It only has to find the neighbourhood, and nothing is fitted, so there is nothing to be fooled by |
| **Confirm** | Scan the candidate's neighbourhood twice and correlate | Contrast alone cannot tell a resonance from noise — see below |
| **Bracket** | Widen until the curve comes back down on *both* sides | A fit that interpolates between points is trustworthy; one that extrapolates past them is not |
| **Refine** | Dense tracked zero-IF sweep around the estimated width | Grid density and settling, not receiver passband, set linewidth accuracy |

**The coarse pass cannot find a line narrower than its own step** — it walks straight over it. So
`SEARCH_STEP` is **halved automatically** and the pass repeated until a candidate confirms, which
is what a colder, higher-Q resonator needs and is exactly the parameter nobody remembers to
change when the coil moves into a cryostat.

**Why contrast is not enough to trigger that.** `|z|²` on an empty band is exponentially
distributed, so the largest of a few hundred points sits about 8 dB above the median *whatever*
the averaging — the ratio is scale-free. A 6 dB threshold therefore passes pure noise, the
halving never fires, and the search reports the biggest noise excursion in the band as a
discovery. The confirmation step scans the neighbourhood twice and **correlates the two scans**: a
resonance puts the same shape on the grid both times, while two realisations of noise are
independent and correlate at zero give or take 1/√N. Correlation rather than "do the two maxima
agree", because the top of a broad line is flat and its argmax wanders by a good fraction of a
linewidth while a narrow line's does not — one threshold cannot serve both. Correlation is
scale-free and works at any Q.

The fit at the end is checked against `SEARCH_FIT_MAX_RESID`, but only warns: it is taken at the
starting drive, and a poor residual there is exactly what Step 4b exists to fix. **Step 4b
enforces it** once the drive has been chosen by measurement.


In [ ]:
# =============================================================================
#  STEP 3 - FIND THE RESONANCE
# =============================================================================
if not COIL_CONNECTED:
    raise SystemExit('COIL_CONNECTED is False - Steps 3 onwards need the coil. '
                     'Stop here; the survey above is the deliverable for this run.')

print('=' * 68)
print('STEP 3 - find the resonance')
print('=' * 68)

emu.disable()
ps.off()
preflight_drive(0.5 * (SEARCH_LO + SEARCH_HI))


def confirm_candidate(f_cand, step, drive, band_floor):
    """Does the coarse pass's best point survive a second look?

    THE CONTRAST TEST ALONE IS NOT ENOUGH, and this is the trap it walks into:
    |z|^2 on an empty band is exponentially distributed, so the largest of a few
    hundred points sits roughly 8 dB above the median whatever the averaging -
    the ratio is scale-free, so more averaging does not help. A 6 dB threshold
    therefore passes pure noise, and the search reports the biggest noise
    excursion in the band as a discovery. That is exactly what a narrow line
    looks like from outside: the coarse grid steps over the real resonance,
    finds nothing, and the noise wins by default.

    So scan the neighbourhood TWICE and CORRELATE the two scans. A resonance
    puts the same shape on the grid both times; two realisations of noise are
    independent, and their correlation sits at zero give or take 1/sqrt(N).
    Correlation is used rather than "do the two maxima agree", because the top
    of a broad line is flat and its argmax wanders by a good fraction of a
    linewidth, while a narrow line's does not - the same threshold cannot serve
    both. Correlation is scale-free and works at any Q.

    `band_floor` is the median power of the COARSE pass, not of this zoomed
    window: on a line wider than the window the local median is most of the way
    up the peak, and a perfectly good resonance then measures 2 dB of "contrast"
    against itself.
    """
    span = 2.0 * step
    fine = np.linspace(f_cand - span, f_cand + span, 81)
    fine = fine[(fine > 1.0e6) & (fine < 0.49 * FS_CLK)]
    ps = []
    for _ in range(2):
        f_, z_ = sweep(fine, drive_frac=drive, navg=32, settle=0.006,
                       level_guard=0.95)
        gen.amp(0.0)
        ps.append(np.abs(z_) ** 2)
    a = ps[0] - ps[0].mean()
    b = ps[1] - ps[1].mean()
    denom = np.sqrt(float(np.dot(a, a)) * float(np.dot(b, b)))
    corr = float(np.dot(a, b) / denom) if denom > 0 else 0.0
    p = 0.5 * (ps[0] + ps[1])
    contrast = 10.0 * np.log10(p.max() / max(band_floor, 1e-300))
    lw = half_power_width(fine, p)
    ok = bool(corr >= SEARCH_MIN_CORR and contrast >= SEARCH_MIN_DB and lw)
    return dict(ok=ok, f0=float(fine[int(np.argmax(p))]), lw=lw, corr=corr,
                contrast=contrast, f=fine, p=p)


# --- pass 1: look around, then confirm --------------------------------------
#  The coarse pass cannot find a line narrower than its own step, so the step is
#  halved and the pass repeated until a candidate CONFIRMS. That is the one
#  parameter that has to change when the coil gets colder and narrower, and it
#  is the one nobody remembers to change.
acq.start(DEC_SEARCH)
SEARCH_STEP_USED = float(SEARCH_STEP)
FOUND = None
for _attempt in range(SEARCH_RETRIES + 1):
    grid1 = np.arange(SEARCH_LO, SEARCH_HI + SEARCH_STEP_USED, SEARCH_STEP_USED)
    t0 = time.time()
    f1, z1 = sweep(grid1, drive_frac=DRIVE_FRAC, navg=16, settle=0.006,
                   level_guard=0.95)
    gen.amp(0.0)
    p1 = np.abs(z1) ** 2
    k1 = int(np.argmax(p1))
    contrast_db = 10.0 * np.log10(p1[k1] / max(np.median(p1), 1e-300))
    print('pass 1, step %6.1f kHz : %4d points in %3.0f s, best %.4f MHz, '
          '%.1f dB above the median, %.0f counts'
          % (SEARCH_STEP_USED / 1e3, len(grid1), time.time() - t0,
             f1[k1] / 1e6, contrast_db, np.sqrt(p1[k1])))

    conf = confirm_candidate(f1[k1], SEARCH_STEP_USED, DRIVE_FRAC,
                             float(np.median(p1)))
    print('        confirming : two scans correlate %+.3f (need %.2f), contrast '
          '%.1f dB, width %s'
          % (conf['corr'], SEARCH_MIN_CORR, conf['contrast'],
             '%.1f kHz' % (conf['lw'] / 1e3) if conf['lw'] else 'unresolved'))
    if conf['ok']:
        FOUND = conf
        print('        CONFIRMED at %.4f MHz' % (conf['f0'] / 1e6))
        break
    if _attempt == SEARCH_RETRIES:
        raise RuntimeError(
            'no candidate survived a second look after %d halvings, down to a '
            '%.2f kHz step. The coarse pass keeps landing on noise, which means '
            'either the resonance is narrower than %.2f kHz, or it is outside '
            '%.0f-%.0f MHz, or the coil is not coupled at this drive (%.5f). '
            'Raise SEARCH_RETRIES, widen the band, or check the coupling.'
            % (SEARCH_RETRIES, SEARCH_STEP_USED / 1e3, SEARCH_STEP_USED / 1e3,
               SEARCH_LO / 1e6, SEARCH_HI / 1e6, DRIVE_FRAC))
    SEARCH_STEP_USED *= 0.5
    print('        NOT confirmed - halving the step to %.2f kHz and retrying'
          % (SEARCH_STEP_USED / 1e3))

if np.sqrt(p1[k1]) < 20.0:
    print('        WARNING: under 20 counts at the coarse peak. Check the drive,')
    print('        the coupling, and whether an external amplifier is declared in')
    print('        EXT_GAIN_DB but not actually fitted.')


In [ ]:
# --- pass 2: bracket ---------------------------------------------------------
#  Widen until the curve comes down on BOTH sides. Stopping only once both
#  flanks are below half the peak is what makes the later fit an interpolation
#  rather than an extrapolation.
span = max(6.0 * FOUND['lw'], 0.5 * SEARCH_STEP_USED)
f_c = FOUND['f0']
lw_guess = FOUND['lw']
for attempt in range(6):
    grid2 = f_c + np.linspace(-span, span, 41)
    grid2 = grid2[(grid2 > 1.0e6) & (grid2 < 0.49 * FS_CLK)]
    f2, z2 = sweep(grid2, drive_frac=DRIVE_FRAC, navg=24, settle=0.008,
                   level_guard=0.95)
    gen.amp(0.0)
    p2 = np.abs(z2) ** 2
    base2 = float(np.median(p2))
    half = base2 + 0.5 * (float(np.max(p2)) - base2)
    down_both = (p2[0] < half) and (p2[-1] < half)
    f_c = float(f2[int(np.argmax(p2))])
    lw2 = half_power_width(f2, p2)
    print('pass 2 attempt %d: span +-%8.2f kHz, centre %.4f MHz, flanks down '
          'both sides: %s' % (attempt + 1, span / 1e3, f_c / 1e6, down_both))
    if down_both and lw2:
        lw_guess = lw2
        break
    span *= 2.0

print('        rough linewidth %.2f kHz  ->  rough Q %.0f'
      % (lw_guess / 1e3, f_c / lw_guess))

# --- pass 3: refine and fit --------------------------------------------------
print('pass 3: refine')
DEC_USE, f3, p3, FIT = refine_and_fit(f_c, lw_guess, DRIVE_FRAC)
F0 = FIT['f0']
LW = FIT['lw']
Q_SWEEP = FIT['Q']
FLOOR_FRAC = float(np.sqrt(FIT['base'] / max(FIT['peak'], 1e-300)))

print('')
print('  centre frequency       : %.6f MHz' % (F0 / 1e6))
print('  linewidth              : %.2f kHz' % (LW / 1e3))
print('  quality factor         : %.1f' % Q_SWEEP)
print('  off-resonance floor    : %.1f%% of the peak amplitude' % (100 * FLOOR_FRAC))
print('  fit residual           : %.4f of peak' % FIT['resid'])

#  A fit that does not describe the data is not a resonance. It is only a
#  WARNING here, because this fit is taken at the starting drive and Step 4b has
#  not yet had its say - a poor residual at a drive that turns out to be
#  noise-limited is exactly what Step 4b exists to fix. Step 4b enforces it.
if FIT['resid'] > SEARCH_FIT_MAX_RESID:
    print('  WARNING: residual above the %.2f limit - this is not yet a clean'
          % SEARCH_FIT_MAX_RESID)
    print('  Lorentzian. Step 4b will re-measure at a drive chosen by')
    print('  measurement, and will refuse to continue if it is still this bad.')
if FLOOR_FRAC > 0.5:
    print('  NOTE: a large floor costs precision - it is what the fit has to')
    print('  separate the line from - but it does not bias the width, provided it')
    print('  adds in POWER rather than coherently. Step 5 tests that.')
print('')
print('  Provisional: Step 4b measures the usable drive window and re-runs this')
print('  fit at the drive it recommends.')


In [ ]:
# --- Step 3 picture ----------------------------------------------------------
#  Three panels, all scaled from this run's own data: the whole band, the
#  confirmation scan that decided the candidate was real, and the fitted line.
_fine = np.linspace(f3.min(), f3.max(), 600)

fig, (a1, a2, a3) = plt.subplots(3, 1, figsize=(8.0, 7.4))

a1.plot(f1 / 1e6, 10 * np.log10(np.maximum(p1, 1e-300) / max(np.median(p1), 1e-300)),
        '-', lw=0.9, color=CLR_BLUE)
a1.axvline(F0 / 1e6, ls='--', lw=1.0, color=CLR_RED)
a1.axhline(SEARCH_MIN_DB, ls=':', lw=0.9, color=CLR_GREY)
a1.set_xlabel('drive frequency / MHz')
a1.set_ylabel('dB above median')
a1.set_title('pass 1 - the whole search band at a %.2f kHz step'
             % (SEARCH_STEP_USED / 1e3))

a2.plot((FOUND['f'] - F0) / 1e3, FOUND['p'], '-', lw=1.0, color=CLR_GREEN)
a2.set_xlabel('offset from the fitted centre / kHz')
a2.set_ylabel('power (arb.)')
a2.set_title('confirmation - the two scans correlate %+.3f, %.1f dB contrast'
             % (FOUND['corr'], FOUND['contrast']))

a3.plot((f3 - F0) / 1e3, p3, 'o', ms=3.5, color=CLR_BLUE, label='measured')
a3.plot((_fine - F0) / 1e3, lorentzian(_fine, FIT), '-', lw=1.4, color=CLR_ORANGE,
        label='Lorentzian fit')
a3.axhline(FIT['base'], ls=':', lw=1.0, color=CLR_GREY, label='fitted floor')
a3.set_xlabel('offset from centre / kHz')
a3.set_ylabel('power (arb.)')
a3.set_title('pass 3 - f0 = %.6f MHz, linewidth = %.2f kHz, Q = %.1f, '
             'residual %.4f' % (F0 / 1e6, LW / 1e3, Q_SWEEP, FIT['resid']))
a3.legend(fontsize=8)

plt.show()


---
# Step 4a — How much gain is the amplifier actually giving?

A ladder of drive levels, climbing until the amplifier is near its rating. **It never needs to
know the gain** — it infers the output level from what arrives at the input and the passive
transfer measured without it, so each rung refuses the next if that would cross
`AMP_OUT_MAX_DBM`. Safe whatever is in the box.

**The measured gain is then adopted**, so the ceilings the rest of the notebook obeys come from a
measurement rather than a declaration. The one case that still aborts is a measured gain well
*above* the declared one: `DRIVE_MAX` was derived from the declaration, so every measurement
already taken ran under a ceiling that was too permissive, and that has to be re-run rather than
patched. With no amplifier declared, the ladder's round trip *is* `G_PASSIVE` and is adopted as
such — which is how that calibration gets measured instead of remembered.


In [ ]:
# =============================================================================
#  STEP 4a - HOW MUCH GAIN IS THE AMPLIFIER ACTUALLY GIVING?
# =============================================================================
print('=' * 68)
print('STEP 4a - amplifier gain ladder')
print('=' * 68)
print('passive transfer without the amplifier : %.5f' % G_PASSIVE)
print('stopping when the amplifier would pass : %+.1f dBm' % AMP_OUT_MAX_DBM)
print('')
print('%9s %8s %10s %9s %10s' % ('drive', 'tone', 'G total', 'gain dB', 'amp dBm'))

amp_D, amp_G, amp_dbm = [], [], []
AMP_STOP_REASON = 'reached the amplifier rating'
try:
    D = AMP_TEST_START
    while D <= DRIVE_MAX:
        _raw, tone, Gt = measure_transfer(F0, D)
        if tone <= 0:
            AMP_STOP_REASON = 'no signal at the input'; break
        out = 10.0 + 20.0 * np.log10(max(tone / G_PASSIVE, 1e-12))
        amp_D.append(D); amp_G.append(Gt); amp_dbm.append(out)
        print('%9.5f %8.5f %10.4f %+9.1f %+10.1f'
              % (D, tone, Gt, 20 * np.log10(Gt / G_PASSIVE), out))
        if tone > 0.5:
            AMP_STOP_REASON = 'input approaching full scale'; break
        if len(amp_G) > 2 and Gt < 0.79 * max(amp_G):
            AMP_STOP_REASON = 'gain has fallen 2 dB - the amplifier is compressing'
            break
        if out + 20.0 * np.log10(AMP_TEST_STEP) > AMP_OUT_MAX_DBM:
            break
        D *= AMP_TEST_STEP
        if D > DRIVE_MAX:
            AMP_STOP_REASON = 'DRIVE_MAX reached before the amplifier rating'
finally:
    gen.off()

print('')
print('stopped: %s' % AMP_STOP_REASON)
if len(amp_G) < 2:
    raise RuntimeError('ladder too short to measure a gain - check the wiring')

amp_D = np.array(amp_D); amp_G = np.array(amp_G); amp_dbm = np.array(amp_dbm)
# The linear region: rungs within 10% of the best transfer. Never fewer than two
# - a "gain" from a single rung is a reading, not a measurement.
_lin_mask = amp_G > 0.90 * amp_G.max()
if int(np.count_nonzero(_lin_mask)) < 2:
    _lin_mask = np.zeros_like(amp_G, dtype=bool)
    _lin_mask[np.argsort(amp_G)[-2:]] = True
    print('  (only one rung stayed within 10% - the gain is provisional)')
G_WITH_AMP = float(np.mean(amp_G[_lin_mask]))
AMP_GAIN_DB = float(20 * np.log10(G_WITH_AMP / G_PASSIVE))
DRIVE_SUGGEST = float(min(amp_D[int(np.argmax(amp_G))],
                          amp_D[-1] * 10 ** ((AMP_OUT_MAX_DBM - 3.0 - amp_dbm[-1]) / 20.0)))

print('  amplifier gain          : %+.1f dB  (%.0fx), from %d linear rungs'
      % (AMP_GAIN_DB, 10 ** (AMP_GAIN_DB / 20), int(np.count_nonzero(_lin_mask))))
print('  round-trip transfer     : %.4f  (was %.4f)' % (G_WITH_AMP, G_PASSIVE))
print('  self-oscillation onset  : A = %.2f   (register limit 1.99)'
      % (1.0 / G_WITH_AMP))
print('-' * 68)
if 1.0 / G_WITH_AMP < 1.99:
    print('  REACHABLE - the instability region can appear in the Step 8c map.')
else:
    print('  NOT reachable: %.0f dB short. A second amplifier in the same path'
          % (20 * np.log10((1.0 / 1.99) / G_WITH_AMP)))
    print('  would close it; the map will otherwise be complete but with no')
    print('  black region.')

# ---- the one case that must abort, not adopt --------------------------------
#  DRIVE_MAX was derived from the DECLARED gain. If the measured gain is well
#  above it, the ceiling every measurement so far ran under was too permissive,
#  and that has to be re-run rather than patched. The protection against an
#  amplifier that is fitted but NOT declared is preflight_drive(), which measures
#  the actual input level and does not care what anyone declared.
if EXT_GAIN_DB > 0.0 and AMP_GAIN_DB > EXT_GAIN_DB + 3.0:
    raise RuntimeError(
        'EXT_GAIN_DB declares %.1f dB but the ladder measured %.1f dB. DRIVE_MAX '
        'was derived from the declared figure, so it is %.1f dB too permissive '
        'and every step above ran under a ceiling that was too high. Put %.1f in '
        'the configuration cell and re-run from the top.'
        % (EXT_GAIN_DB, AMP_GAIN_DB, AMP_GAIN_DB - EXT_GAIN_DB, AMP_GAIN_DB))

# ---- adopt --------------------------------------------------------------
#  The measured gain controls transfer predictions. Hardware ceilings remain
#  derived from EXT_GAIN_SAFETY_DB, so adopting a lower healthy measurement
#  can never loosen the fault or compression protection.
print('')
if not AUTO_CALIBRATE:
    print('  AUTO_CALIBRATE is off: nothing adopted. EXT_GAIN_DB stays at %.1f dB'
          % EXT_GAIN_DB)
elif EXT_GAIN_DB <= 0.0:
    if abs(AMP_GAIN_DB) <= 3.0:
        #  Nothing is fitted, so the ladder's round trip IS the passive transfer.
        #  This is how that calibration gets measured instead of remembered.
        G_PASSIVE = G_WITH_AMP
        print('  no amplifier declared and none measured: G_PASSIVE re-measured')
        print('  as %.5f and adopted.' % G_PASSIVE)
    else:
        print('  *** %.1f dB of gain measured but EXT_GAIN_DB says none is fitted.'
              % AMP_GAIN_DB)
        print('  Either G_PASSIVE (%.5f) is stale, or an amplifier is fitted and'
              % G_PASSIVE)
        print('  undeclared - in which case DRIVE_MAX is far too permissive.')
        print('  Resolve this before running any further step.')
else:
    EXT_GAIN_DB = AMP_GAIN_DB
    DRIVE_MAX, DRIVE_LINEAR_MAX = derive_limits(
        EXT_GAIN_SAFETY_DB, EXT_PAD_DB, AMP_IN_MAX_DBM, AMP_OUT_MAX_DBM)
    print('  ADOPTED: measured EXT_GAIN_DB = %.1f dB for transfer predictions.'
          % EXT_GAIN_DB)
    print('           Safety limits retain %.1f dB worst-case gain; DRIVE_MAX = %.5f,'
          % (EXT_GAIN_SAFETY_DB, DRIVE_MAX))
    print('           DRIVE_LINEAR_MAX = %.5f. Step 4b scans inside these.'
          % DRIVE_LINEAR_MAX)
    if DRIVE_FRAC > DRIVE_MAX:
        DRIVE_FRAC = 0.5 * DRIVE_MAX
        print('  DRIVE_FRAC exceeded the new ceiling and has been cut to %.5f.'
              % DRIVE_FRAC)


In [ ]:
# --- Step 4a picture: the ladder --------------------------------------------
#  Axis limits come from the ladder itself, so the same cell is readable whatever
#  is in the amplifier box.
fig, (a1, a2) = plt.subplots(2, 1, figsize=(7.4, 5.4), sharex=True)

a1.loglog(amp_D, amp_G, 'o-', color=CLR_BLUE, label='measured transfer')
a1.loglog(amp_D[_lin_mask], amp_G[_lin_mask], 'o', ms=8, mfc='none',
          mec=CLR_RED, mew=1.4, label='linear region, averaged')
a1.axhline(G_PASSIVE, ls=':', lw=1.0, color=CLR_GREY, label='passive (no amplifier)')
a1.set_ylabel('round-trip transfer')
a1.set_title('Step 4a - gain %+.1f dB, onset A = %.2f  (%s)'
             % (AMP_GAIN_DB, 1.0 / G_WITH_AMP, AMP_STOP_REASON))
a1.legend(fontsize=8)

a2.semilogx(amp_D, amp_dbm, 's-', color=CLR_ORANGE, label='inferred amplifier output')
a2.axhline(AMP_OUT_MAX_DBM, ls='--', lw=1.1, color=CLR_RED,
           label='AMP_OUT_MAX_DBM (%+.1f dBm)' % AMP_OUT_MAX_DBM)
a2.set_xlabel('DRIVE_FRAC')
a2.set_ylabel('amplifier output / dBm')
a2.legend(fontsize=8)
plt.show()

print('Top: the transfer is flat while the amplifier is linear and falls when it')
print('compresses. Bottom: the ladder stops before the next rung would cross the')
print('rating - it infers the output from what arrives at the input, so it never')
print('needs to be told the gain.')


---
# Step 4b — The drive window

Steps 5 to 8 all assume the fit is describing the coil. Two things break that, pulling opposite
ways: **too little drive** leaves the fitted peak mostly receiver noise floor with the DAC out of
counts, and **too much** compresses the amplifier so the centre and width move. Nudging cannot
find the window between them, because from inside each failure looks like the other.

**The emulator settles it.** Enabled, it replaces the input with a resonator of known Q driven
through the same chain as the coil, so the drive at which the recovered Q leaves its known value
is the smallest signal at which *this* receiver and *this* fitter can be trusted — a property of
the instrument, not of the coil. The same ladder then runs on the coil: rungs clearing that
threshold are noise-safe, rungs where Q has left its plateau are compressed, and the window is
what is left.

**The recommendation is applied, not printed.** `DRIVE_FRAC` becomes the top of the window and
Step 3's refine-and-fit is re-run at it, so `F0`, the linewidth and `Q` all belong to the drive
Steps 5 to 8 actually use. A recommendation that has to be typed back into the configuration cell
is a recommendation that gets forgotten.


In [ ]:
# =============================================================================
#  STEP 4b - THE DRIVE WINDOW, AND ADOPTING IT
# =============================================================================
print('=' * 68)
print('STEP 4b - where is the fit describing the coil and not the noise?')
print('=' * 68)

_hi = min(DRIVE_LINEAR_MAX, DRIVE_MAX)
scan_D = _hi * np.logspace(-1.5, 0.0, DRIVE_SCAN_N)

# ---- part 1: what does a trustworthy fit need, in receiver counts? ----------
#  The emulator is a known answer measured through the identical chain, so this
#  threshold is a property of the instrument and not of the coil.
print('part 1 - calibrating the fitter against the emulator (known Q = %.0f)'
      % SELFTEST_Q)
print('%10s %8s %11s %10s' % ('drive', 'set_amp', 'amplitude', 'Q error'))
cal_amp, cal_err = [], []
try:
    f0q, qq = emu.config(SELFTEST_F0, SELFTEST_Q, delay=12, oshift=7)
    lwq = f0q / qq
    emu.clear(); emu.enable()
    acq.start(DEC_USE)
    gcal = f0q + np.linspace(-2.5, 2.5, 25) * lwq
    for D in scan_D:
        try:
            fc, zc = sweep(gcal, drive_frac=D, navg=N_AVG, level_guard=0.95)
            ft = fit_lorentzian(fc, np.abs(zc) ** 2)
            amp = float(np.sqrt(max(ft['peak'], 0.0)))
            err = abs(ft['Q'] / qq - 1.0)
            cal_amp.append(amp); cal_err.append(err)
            print('%10.5f %8d %11.4g %9.2f%%'
                  % (D, int(round(D * 8192)), amp, 100 * err))
        except Exception as exc:
            print('%10.5f %8d   fit failed (%s)'
                  % (D, int(round(D * 8192)), str(exc)[:34]))
finally:
    emu.disable(); gen.off()

_ok = [a for a, e in zip(cal_amp, cal_err) if e <= FIT_TRUST_TOL]
if _ok:
    MIN_AMP_FOR_FIT = float(min(_ok))
    print('  -> a fit is good to %.0f%% once the peak reaches %.4g counts'
          % (100 * FIT_TRUST_TOL, MIN_AMP_FOR_FIT))
else:
    MIN_AMP_FOR_FIT = float('inf')
    print('  -> the emulator was never recovered to %.0f%%. The receiver cannot'
          % (100 * FIT_TRUST_TOL))
    print('     fit anything at these drives; nothing below is meaningful.')

# ---- part 2: the same ladder on the coil -----------------------------------
print('')
print('part 2 - the coil')
print('%10s %8s %11s %9s %10s %8s'
      % ('drive', 'set_amp', 'amplitude', 'Q', 'floor/peak', 'resid'))
sc_D, sc_amp, sc_Q, sc_fl = [], [], [], []
try:
    acq.start(DEC_USE)
    gco = F0 + np.linspace(-2.5, 2.5, 41) * LW
    for D in scan_D:
        try:
            fk, zk = sweep(gco, drive_frac=D, navg=N_AVG, level_guard=0.95)
            ft = fit_lorentzian(fk, np.abs(zk) ** 2)
            amp = float(np.sqrt(max(ft['peak'], 0.0)))
            fl = float(np.sqrt(max(ft['base'], 0.0)) / max(amp, 1e-30))
            sc_D.append(D); sc_amp.append(amp); sc_Q.append(ft['Q']); sc_fl.append(fl)
            print('%10.5f %8d %11.4g %9.1f %10.3f %8.4f'
                  % (D, int(round(D * 8192)), amp, ft['Q'], fl, ft['resid']))
        except Exception as exc:
            print('%10.5f %8d   fit failed (%s)'
                  % (D, int(round(D * 8192)), str(exc)[:34]))
finally:
    gen.off()

if len(sc_D) < 3:
    #  Say WHY, because the two causes need opposite responses and the message
    #  is the only thing standing between them.
    _thr = ('the %.4g counts a trustworthy fit needs' % MIN_AMP_FOR_FIT
            if np.isfinite(MIN_AMP_FOR_FIT)
            else 'a threshold the emulator never reached either')
    raise RuntimeError(
        'only %d of %d rungs produced a fit, so there is no window to find. '
        'Largest peak %.4g counts, against %s. The receiver is not seeing the '
        'coil at any drive this chain allows: DRIVE_MAX is %.5f with %.1f dB of '
        'external gain declared, and Step 4a measured %+.1f dB. Fix the chain or '
        'the coupling; there is nothing here to fit.'
        % (len(sc_D), len(scan_D), max(sc_amp) if sc_amp else 0.0, _thr,
           DRIVE_MAX, EXT_GAIN_DB, AMP_GAIN_DB))
sc_D = np.array(sc_D); sc_amp = np.array(sc_amp); sc_Q = np.array(sc_Q)

# ---- the window ------------------------------------------------------------
_noise_ok = sc_amp >= MIN_AMP_FOR_FIT
_reg_ok = np.array([int(round(d * 8192)) >= 30 for d in sc_D])
DRIVE_BEST = float('nan')
if not np.any(_noise_ok & _reg_ok):
    DRIVE_WINDOW = (float('nan'), float('nan'))
    print('')
    print('  NO USABLE WINDOW at any drive this amplifier allows.')
    print('  The largest peak reached was %.4g against the %.4g a trustworthy fit'
          % (sc_amp.max(), MIN_AMP_FOR_FIT))
    print('  needs. Raise N_AVG, or accept that Q is the only number this rig can')
    print('  give and treat the drive-dependence tests as unavailable.')
else:
    _cand = sc_Q[_noise_ok & _reg_ok]
    _plateau = float(np.median(_cand))
    _flat = np.abs(sc_Q / _plateau - 1.0) < 0.03
    _good = _noise_ok & _reg_ok & _flat
    DRIVE_WINDOW = (float(sc_D[_good].min()), float(sc_D[_good].max()))
    DRIVE_BEST = float(sc_D[_good].max())
    print('')
    print('  noise-limited below : %.5f' % (sc_D[~_noise_ok].max()
                                            if np.any(~_noise_ok) else 0.0))
    print('  compressed above    : %s'
          % ('%.5f' % sc_D[_good].max() if np.any(~_flat & _noise_ok) else 'not reached'))
    print('  USABLE WINDOW       : %.5f to %.5f' % DRIVE_WINDOW)
    print('  RECOMMENDED         : DRIVE_FRAC = %.5f  (set_amp %d)'
          % (DRIVE_BEST, int(round(DRIVE_BEST * 8192))))
    print('  quality factor there: %.1f  (plateau %.1f)'
          % (sc_Q[sc_D == DRIVE_BEST][0], _plateau))
    if DRIVE_WINDOW[0] == DRIVE_WINDOW[1]:
        print('  Only one rung qualifies - the window is marginal. More averaging')
        print('  (N_AVG) would widen it from the bottom.')

# ---- adopt it, and re-measure the resonance there ---------------------------
#  A recommendation that has to be typed back into the configuration cell is a
#  recommendation that gets forgotten, and every number in Steps 5 to 8 would
#  then belong to a different drive from the one that produced F0 and LW.
DRIVE_ADOPTED = False
print('')
if not AUTO_CALIBRATE:
    print('AUTO_CALIBRATE is off: DRIVE_FRAC stays at %.5f.' % DRIVE_FRAC)
elif not np.isfinite(DRIVE_BEST):
    print('No window to adopt - DRIVE_FRAC stays at %.5f.' % DRIVE_FRAC)
elif abs(DRIVE_BEST / max(DRIVE_FRAC, 1e-12) - 1.0) < 0.05:
    print('DRIVE_FRAC %.5f is already at the top of the window - nothing to change.'
          % DRIVE_FRAC)
else:
    _old_drive, _old_f0 = DRIVE_FRAC, F0
    _old_lw, _old_q, _old_fl = LW, Q_SWEEP, FLOOR_FRAC
    DRIVE_BEST = min(DRIVE_BEST, FB_BASE_DRIVE_MAX)
    DRIVE_FRAC = DRIVE_BEST
    DRIVE_ADOPTED = True
    print('ADOPTING conservative feedback drive %.5f (was %.5f) and re-measuring the resonance'
          % (DRIVE_FRAC, _old_drive))
    DEC_USE, f3, p3, FIT = refine_and_fit(F0, LW, DRIVE_FRAC)
    F0 = FIT['f0']; LW = FIT['lw']; Q_SWEEP = FIT['Q']
    FLOOR_FRAC = float(np.sqrt(FIT['base'] / max(FIT['peak'], 1e-300)))
    print('')
    print('  %-20s %14s %14s' % ('', 'before', 'after'))
    print('  %-20s %14.6f %14.6f' % ('centre / MHz', _old_f0 / 1e6, F0 / 1e6))
    print('  %-20s %14.2f %14.2f' % ('linewidth / kHz', _old_lw / 1e3, LW / 1e3))
    print('  %-20s %14.1f %14.1f' % ('quality factor', _old_q, Q_SWEEP))
    print('  %-20s %14.3f %14.3f' % ('floor / peak', _old_fl, FLOOR_FRAC))
    _dq = abs(Q_SWEEP / max(_old_q, 1e-30) - 1.0)
    if _dq > 0.05:
        print('  Q moved %.1f%% with the drive - which is exactly what this step'
              % (100 * _dq))
        print('  exists to catch. The earlier value was noise-limited or')
        print('  compressed; everything below uses the new one.')
    else:
        print('  Q moved %.1f%% - the earlier drive was already inside the window.'
              % (100 * _dq))

#  The last word on whether this is a resonance at all. Step 3 only warned,
#  because its fit was taken before the drive was chosen by measurement; there is
#  no such excuse now, and Steps 5 to 8 would otherwise be fitting noise.
print('')
print('final fit at the working drive: residual %.4f of peak' % FIT['resid'])
if FIT['resid'] > SEARCH_FIT_MAX_RESID:
    raise RuntimeError(
        'residual %.3f of peak at the working drive %.5f, against a %.2f limit: '
        'this curve is not a Lorentzian, so it is not the coil. Raise N_AVG '
        '(currently %d) to buy signal-to-noise, improve the coupling, or lower '
        'SEARCH_STEP (%.2f kHz was used) if the coarse pass may have landed '
        'beside the real line.'
        % (FIT['resid'], DRIVE_FRAC, SEARCH_FIT_MAX_RESID, N_AVG,
           SEARCH_STEP_USED / 1e3))


In [ ]:
# --- Step 4b pictures --------------------------------------------------------
if len(sc_D) >= 3:
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(7.6, 5.6), sharex=True)
    a1.loglog(sc_D, sc_amp, 'o-', color=CLR_BLUE, label='fitted peak')
    if np.isfinite(MIN_AMP_FOR_FIT):
        a1.axhline(MIN_AMP_FOR_FIT, ls='--', color=CLR_RED,
                   label='fit trustworthy above this')
    a1.set_ylabel('peak amplitude / counts')
    a1.set_title('Step 4b - the drive window')
    a1.legend(fontsize=8)
    a2.semilogx(sc_D, sc_Q, 'o-', color=CLR_ORANGE)
    if np.all(np.isfinite(DRIVE_WINDOW)):
        a2.axvspan(DRIVE_WINDOW[0], DRIVE_WINDOW[1], color=CLR_GREEN, alpha=0.15,
                   label='usable window')
    a2.axvline(DRIVE_FRAC, ls='-', lw=1.4, color=CLR_RED,
               label='adopted DRIVE_FRAC = %.5f' % DRIVE_FRAC)
    a2.legend(fontsize=8)
    a2.set_xlabel('DRIVE_FRAC')
    a2.set_ylabel('fitted Q')
    plt.show()
    print('Top: the fit needs the peak above the dashed line. Bottom: Q must be on')
    print('its plateau. The window is where both hold - too little drive is noise,')
    print('too much is compression, and nudging cannot find it because from inside')
    print('each looks like the other.')

# The re-measured line, at the drive everything below actually uses.
if DRIVE_ADOPTED:
    _fine = np.linspace(f3.min(), f3.max(), 600)
    fig, ax = plt.subplots(figsize=(7.4, 3.4))
    ax.plot((f3 - F0) / 1e3, p3, 'o', ms=3.5, color=CLR_BLUE, label='measured')
    ax.plot((_fine - F0) / 1e3, lorentzian(_fine, FIT), '-', lw=1.4,
            color=CLR_ORANGE, label='Lorentzian fit')
    ax.axhline(FIT['base'], ls=':', lw=1.0, color=CLR_GREY, label='fitted floor')
    ax.set_xlabel('offset from centre / kHz')
    ax.set_ylabel('power (arb.)')
    ax.set_title('re-measured at the adopted drive %.5f - f0 = %.6f MHz, '
                 'lw = %.2f kHz, Q = %.1f' % (DRIVE_FRAC, F0 / 1e6, LW / 1e3, Q_SWEEP))
    ax.legend(fontsize=8)
    plt.show()


---
# Step 4c — What the coupling costs

Two numbers, one predicted and one measured. Where they differ, the difference is telling you
about the wiring rather than the coil.

**The prediction.** The pickup capacitor and everything after it — cable, connector, board input
— form a capacitive divider. The 13 kΩ reactance of 0.3 pF at 40 MHz is the *good* half of the
trade: almost nothing is drawn from the tank, so the Q measured above is close to the coil's own.

**The measurement.** How much signal actually arrived, against the floor from Step 2 — which
decides whether an amplifier is needed.

**The conclusion, in advance:** *driven* measurements work without one, because you can drive
harder. What needs an amplifier is anything driven by the coil's **own thermal noise**, which is
how the real experiment works. The last block estimates that directly.


In [ ]:
# =============================================================================
#  STEP 4c - WHAT THE COUPLING COSTS
# =============================================================================
print('=' * 68)
print('STEP 4c - the signal budget, at the adopted drive %.5f' % DRIVE_FRAC)
print('=' * 68)

w0 = 2.0 * np.pi * F0
Xc = 1.0 / (w0 * C_COUPLE)
divider = C_COUPLE / (C_COUPLE + C_STRAY_EST)

print('PREDICTED, from the values in the configuration cell')
print('  reactance of %.2f pF at %.3f MHz : %.1f kohm'
      % (C_COUPLE * 1e12, F0 / 1e6, Xc / 1e3))
print('  capacitive divider %.2f pF / (%.2f + %.1f) pF : %.4f  (%.1f dB)'
      % (C_COUPLE * 1e12, C_COUPLE * 1e12, C_STRAY_EST * 1e12,
         divider, 20 * np.log10(divider)))
print('  -> every 10 mV across the tank arrives as %.3f mV at the board'
      % (10.0 * divider))
# A series capacitor into a resistor R looks, from the tank, like a shunt
# resistance R (1 + (Xc/R)^2). With R = 1 Mohm and Xc = 13 kohm that is
# essentially R itself - the path is capacitive, not lossy.
r_shunt = R_IN_BOARD * (1.0 + (Xc / R_IN_BOARD) ** 2)
print('  equivalent shunt resistance across the tank : %.2f Mohm' % (r_shunt / 1e6))

print('')
print('MEASURED')
peak_counts = float(np.sqrt(FIT['peak']))
base_counts = float(np.sqrt(max(FIT['base'], 1e-300)))
print('  peak response at resonance : %.1f counts (arb. units of the receiver)'
      % peak_counts)
print('  off-resonance baseline     : %.1f counts' % base_counts)
if base_counts > 0:
    print('  resonant contrast          : %.1f dB'
          % (20 * np.log10(peak_counts / base_counts)))
else:
    print('  resonant contrast          : no floor resolved by the fit')

raw_peak, TONE_PEAK, G_FWD = measure_transfer(F0, DRIVE_FRAC)
print('  raw input at resonance     : %.4f of full scale = %.2f mV peak'
      % (raw_peak, 1e3 * raw_peak * V_FULLSCALE))
print('  the tone alone             : %.4f of full scale = %.2f mV peak'
      % (TONE_PEAK, 1e3 * TONE_PEAK * V_FULLSCALE))
print('  peak / tone                : %.2fx  (the excess is DC offset and noise)'
      % (raw_peak / max(TONE_PEAK, 1e-12)))
print('  one-way transfer DAC->ADC  : %.5f  (%.1f dB) at resonance'
      % (G_FWD, 20 * np.log10(max(G_FWD, 1e-12))))
if raw_peak > 3.0 * TONE_PEAK:
    print('    -> most of the input excursion is NOT the signal. Harmless for')
    print('       clipping, but check Step 2 for what else is arriving.')

# ---- the off-resonance floor ------------------------------------------------
#  Drive well away from the line. Whatever still arrives is a floor: crosstalk,
#  stray coupling, noise. It costs precision but does not bias the width.
_far = np.array([F0 - 40.0 * LW, F0 + 40.0 * LW])
_far = _far[(_far > 1.0e6) & (_far < 0.45 * FS_CLK)]
_ff, _zf = sweep(_far, drive_frac=DRIVE_FRAC, navg=N_AVG, level_guard=0.95)
FLOOR_DIRECT = float(np.mean(np.abs(_zf)))
_on = float(np.sqrt(FIT['peak'] + FIT['base']))

print('')
print('OFF-RESONANCE FLOOR')
print('  measured +-40 linewidths out  : %.1f counts' % FLOOR_DIRECT)
print('  on-resonance response         : %.1f counts' % _on)
print('  ratio                         : %.3f' % (FLOOR_DIRECT / max(_on, 1e-300)))
print('  same ratio from the fit       : %.3f' % FLOOR_FRAC)

# Feedback can only act on the part of the response that came from the coil.
G_FWD_RES = G_FWD * float(np.sqrt(FIT['peak'] / max(FIT['peak'] + FIT['base'], 1e-300)))
print('')
print('  transfer via the coil         : %.5f  (%.1f dB)  <- Step 8 uses this'
      % (G_FWD_RES, 20 * np.log10(max(G_FWD_RES, 1e-12))))

print('')
NOISE_RATIO = float('nan')
if L_EST is None:
    print('THERMAL NOISE: skipped. Set L_EST in the configuration cell.')
else:
    kB = 1.380649e-23
    r_par = Q_SWEEP * w0 * L_EST                 # tank equivalent parallel R
    E_N_COIL = np.sqrt(4.0 * kB * T_COIL * r_par)      # V per root hertz
    E_N_DIV = E_N_COIL * divider                       # ...after the divider
    v_lsb = V_FULLSCALE / 8192.0
    # 14-bit quantisation noise density over the first Nyquist zone
    E_BOARD = (v_lsb / np.sqrt(12.0)) / np.sqrt(FS_CLK / 2.0)
    NOISE_RATIO = float(E_N_DIV / E_BOARD)
    print('THERMAL NOISE, using L = %.3f uH' % (L_EST * 1e6))
    print('  tank equivalent parallel resistance : %.1f kohm' % (r_par / 1e3))
    print('  coil Johnson noise at resonance     : %.2f nV/rtHz' % (E_N_COIL * 1e9))
    print('  after the %.4f divider              : %.2f nV/rtHz'
          % (divider, E_N_DIV * 1e9))
    print('  board quantisation floor            : %.2f nV/rtHz (ideal; the real'
          % (E_BOARD * 1e9))
    print('                                         floor is higher - Step 2)')
    print('  ratio                               : %.3f  (coil noise is %.0fx'
          % (NOISE_RATIO, 1.0 / max(NOISE_RATIO, 1e-9)))
    print('                                         below the board floor)')
    if NOISE_RATIO < 3.0:
        print('  -> the coil\'s own noise is AT OR BELOW the board floor. Driven')
        print('     measurements are fine; noise-driven ones need an amplifier.')
    else:
        print('  -> the coil\'s own noise should be visible without an amplifier.')


---
# Step 4d — The coil against its own inductance and capacitance

The strongest test in the notebook, because `L` and `C` were measured *by a different instrument
on a different day*. `f₀ = 1/(2π√(LC))` over the coil's own `C_coil`, the `C_parallel` fitted to
emulate the trap, and the two coupling capacitors each in series with their cable. Nothing is
adjustable.

**Why the 39 pF matters more than it looks.** Alone, 423.5 nH with 1.59 pF resonates near
165 MHz — above Nyquist, invisible to this board at any setting. It is what puts the resonance in
the band, and at 25× the coil's own capacitance it dominates the error budget: a 5% part puts
±2.4% on the predicted centre against `C_coil`'s ±0.07%. **If prediction and measurement
disagree, suspect the capacitor's tolerance before the coil.**

**The inversion is the deliverable.** Take the *measured* `f₀` and the *known* `L`, solve for the
capacitance actually across the tank, subtract what is accounted for, and what remains is the
stray capacitance of the real assembly — holder, leads, connectors. Nothing else here can measure
it, and it is the number worth carrying to the trap. Stray can only *add*, so a negative result
means the model is too big.

**And the losses.** With `L` known, `R_series = ω₀L/Q` and `R_parallel = Q·√(L/C)` follow, and the
latter sets both the signal budget above and the loop gain of Step 8. Adding capacitance lowers
`f₀` and the characteristic impedance together, so it costs Q and signal without changing the
coil — which is what makes this comparable with a bare-coil measurement taken before the
capacitor went on.


In [ ]:
# =============================================================================
#  STEP 4d - THE COIL AGAINST ITS OWN L AND C
# =============================================================================
#  The only independent check in this notebook. L and C came from elsewhere; f0
#  and Q came from Steps 3 and 4b. Nothing here is fitted to anything above.
# =============================================================================
print('=' * 68)
print('STEP 4d - measured resonance against the known L and C')
print('=' * 68)

# ---- the capacitance actually across the tank -------------------------------
#  Each coupling capacitor sits in series with everything after it, so what the
#  tank sees from each port is the SERIES pair - and 0.3 pF in series with 25 pF
#  is 0.297 pF, i.e. the small capacitor wins almost completely. Two ports.
C_SER_PORT = C_COUPLE * C_STRAY_EST / (C_COUPLE + C_STRAY_EST)
C_PORTS = 2.0 * C_SER_PORT
C_MODEL = C_COIL + C_PARALLEL + C_PORTS
C_MODEL_ERR = np.sqrt(C_COIL_ERR ** 2 + C_PARALLEL_ERR ** 2)


def _f_of(L, Cq):
    return 1.0 / (2.0 * np.pi * np.sqrt(L * Cq))


F0_PRED = _f_of(L_COIL, C_MODEL)
# f0 goes as (LC)^-1/2, so a fractional error in L or C becomes HALF of one in f0
F0_PRED_ERR = F0_PRED * 0.5 * np.hypot(L_COIL_ERR / L_COIL, C_MODEL_ERR / C_MODEL)

print('capacitance across the tank')
print('  coil self-capacitance      : %6.2f +/- %.2f pF'
      % (C_COIL * 1e12, C_COIL_ERR * 1e12))
print('  fitted parallel capacitor  : %6.2f +/- %.2f pF'
      % (C_PARALLEL * 1e12, C_PARALLEL_ERR * 1e12))
print('  two coupling ports         : %6.2f pF  (%.3f pF each: %.1f pF in series '
      'with %.0f pF)'
      % (C_PORTS * 1e12, C_SER_PORT * 1e12, C_COUPLE * 1e12, C_STRAY_EST * 1e12))
print('  ---------------------------------------')
print('  total, modelled            : %6.2f +/- %.2f pF'
      % (C_MODEL * 1e12, C_MODEL_ERR * 1e12))
print('')

# ---- would this even be visible without the added capacitor? ----------------
F0_BARE = _f_of(L_COIL, C_COIL + C_PORTS)
print('without the %.0f pF the coil would sit at %.1f MHz'
      % (C_PARALLEL * 1e12, F0_BARE / 1e6))
if F0_BARE > 0.5 * FS_CLK:
    print('  -> above the %.1f MHz Nyquist limit. The added capacitance is what'
          % (0.5 * FS_CLK / 1e6))
    print('     makes this measurable at all; it is not a correction.')
print('')

# ---- prediction against measurement -----------------------------------------
_dev = F0 - F0_PRED
_sig = abs(_dev) / F0_PRED_ERR if F0_PRED_ERR > 0 else float('inf')
print('centre frequency')
print('  predicted from L and C     : %.3f +/- %.3f MHz'
      % (F0_PRED / 1e6, F0_PRED_ERR / 1e6))
print('  measured                   : %.3f MHz' % (F0 / 1e6))
print('  difference                 : %+.3f MHz  (%+.2f%%, %.1f sigma)'
      % (_dev / 1e6, 100 * _dev / F0_PRED, _sig))
if _sig <= 2.0 and abs(_dev) / F0_PRED <= 0.02:
    print('  AGREE - two independent routes to the same resonance')
elif _sig <= 2.0:
    print('  WITHIN THE ERROR BAR, BUT ONLY BECAUSE THE ERROR BAR IS WIDE.')
    print('  %+.1f%% is a real offset; it passes only because the %.0f pF part is'
          % (100 * _dev / F0_PRED, C_PARALLEL * 1e12))
    print('  assumed to +-%.0f%%. Measure that capacitor and re-run - the'
          % (100 * C_PARALLEL_ERR / C_PARALLEL))
    print('  inversion below is the sharper test.')
elif _sig <= 4.0:
    print('  MARGINAL. Before blaming the coil, check the tolerance of the %.0f pF'
          % (C_PARALLEL * 1e12))
    print('  part: it is %.0fx the coil capacitance and dominates the prediction.'
          % (C_PARALLEL / C_COIL))
else:
    print('  DISAGREE. In order of likelihood: the parallel capacitor is not the')
    print('  value marked on it; there is more stray capacitance in the assembly')
    print('  than C_STRAY_EST allows; L was measured at a very different frequency')
    print('  from %.1f MHz and the coil is not a simple inductor here.' % (F0 / 1e6))
print('')

# ---- the inversion: what capacitance is REALLY across the tank ---------------
#  f0 and L are both known, so C follows, and whatever it exceeds the model by is
#  stray capacitance in the real assembly - holder, leads, connectors - which
#  nothing else here can measure. This is the number to carry to the trap.
C_IMPLIED = 1.0 / (L_COIL * (2.0 * np.pi * F0) ** 2)
C_IMPLIED_ERR = C_IMPLIED * (L_COIL_ERR / L_COIL)
C_UNACCOUNTED = C_IMPLIED - C_MODEL
print('inverting: total capacitance implied by the measured f0 and the known L')
print('  implied C_total            : %6.2f +/- %.2f pF'
      % (C_IMPLIED * 1e12, C_IMPLIED_ERR * 1e12))
print('  modelled C_total           : %6.2f +/- %.2f pF'
      % (C_MODEL * 1e12, C_MODEL_ERR * 1e12))
print('  unaccounted for            : %+6.2f pF' % (C_UNACCOUNTED * 1e12))
if C_UNACCOUNTED > 2.0 * C_MODEL_ERR:
    print('  -> real stray capacitance in the assembly. Carry it to the trap: the')
    print('     holder and leads contribute %.1f pF and the trap build will see'
          % (C_UNACCOUNTED * 1e12))
    print('     something similar.')
elif C_UNACCOUNTED < -0.5e-12:
    print('  -> NEGATIVE BY %.2f pF, WHICH IS NOT PHYSICAL. Stray capacitance can'
          % abs(C_UNACCOUNTED * 1e12))
    print('     only add, so something in the model is too big. The %.0f pF part is'
          % (C_PARALLEL * 1e12))
    print('     %.0fx the coil capacitance and is the first suspect: measure it.'
          % (C_PARALLEL / C_COIL))
    print('     A high reading of L is the other. Do not carry the implied total')
    print('     forward until this is resolved.')
else:
    print('  -> consistent with zero. The assembly adds no measurable stray.')
print('')

# ---- losses, now that L is a number -----------------------------------------
W0 = 2.0 * np.pi * F0
Z0_TANK = np.sqrt(L_COIL / C_IMPLIED)
R_SERIES = W0 * L_COIL / Q_SWEEP
R_PARALLEL = Q_SWEEP * W0 * L_COIL
print('losses, from Q = %.1f and L = %.1f nH' % (Q_SWEEP, L_COIL * 1e9))
print('  characteristic impedance   : %.1f ohm   (sqrt(L/C))' % Z0_TANK)
print('  series resistance (ESR)    : %.3f ohm' % R_SERIES)
print('  parallel resistance        : %.1f kohm  (the tank at resonance)'
      % (R_PARALLEL / 1e3))
print('')

# ---- the cost of the added capacitor ----------------------------------------
#  The capacitor does not change the series resistance, and Q = w0 L / Rs, so Q
#  simply follows f0 down. That is what makes this run comparable with any
#  measurement taken before the capacitor went on.
Q_BARE_IMPLIED = Q_SWEEP * F0_BARE / F0
print('what the %.0f pF costs' % (C_PARALLEL * 1e12))
print('  Q here                     : %.1f at %.2f MHz' % (Q_SWEEP, F0 / 1e6))
print('  implied bare-coil Q        : %.0f at %.1f MHz  (same ESR, Q scales with f0)'
      % (Q_BARE_IMPLIED, F0_BARE / 1e6))
print('  characteristic impedance   : %.0f -> %.0f ohm'
      % (np.sqrt(L_COIL / (C_COIL + C_PORTS)), Z0_TANK))
print('  -> a factor %.1f in both Q and tank impedance. Compare against an earlier'
      % (F0_BARE / F0))
print('     bare-coil measurement using the implied value, not the one here.')
print('')

# ---- does the measured transfer match the model? ----------------------------
#  A measured value well ABOVE the prediction means signal is reaching the input
#  by some route other than the pickup capacitor - the two coax runs coupling to
#  each other is the usual one - and that route has its own phase, which would
#  corrupt Step 8's axis.
_Xc = 1.0 / (W0 * C_COUPLE)
_v_tank = R_PARALLEL / np.hypot(_Xc, R_PARALLEL)
_div = C_COUPLE / (C_COUPLE + C_STRAY_EST)
G_FWD_PRED = _v_tank * _div
print('forward transfer DAC -> coil -> ADC')
print('  drive side: %.1f kohm of reactance into a %.1f kohm tank -> %.3f'
      % (_Xc / 1e3, R_PARALLEL / 1e3, _v_tank))
print('  pickup divider                                          -> %.5f' % _div)
print('  predicted G_FWD            : %.5f  (%.1f dB)'
      % (G_FWD_PRED, 20 * np.log10(max(G_FWD_PRED, 1e-12))))
print('  measured G_FWD (Step 4c)   : %.5f  (%.1f dB)'
      % (G_FWD, 20 * np.log10(max(G_FWD, 1e-12))))
print('  measured / predicted       : %.2fx' % (G_FWD / max(G_FWD_PRED, 1e-12)))
#  The prediction is PASSIVE, so the amplifier has to come out before the ratio
#  means anything. Comparing them directly is how a working amplifier gets
#  mistaken for a stray coupling path.
AMP_LIN = 10.0 ** (max(EXT_GAIN_DB, 0.0) / 20.0)
G_FWD_PASSIVE = G_FWD / AMP_LIN
if EXT_GAIN_DB > 0.0:
    print('  less the %.1f dB amplifier  : %.5f  (a factor of %.0f)'
          % (EXT_GAIN_DB, G_FWD_PASSIVE, AMP_LIN))
_resid_x = G_FWD_PASSIVE / max(G_FWD_PRED, 1e-12)
print('  passive measured / predicted: %.2fx' % _resid_x)
if _resid_x > 3.0:
    print('  -> More signal arrives than this model predicts. That is a statement')
    print('     about the MODEL first: C_STRAY_EST and the coupling capacitor')
    print('     values are estimates and the transfer depends on both. Measure the')
    print('     parts and the cable capacitance before assuming a stray path. It')
    print('     only becomes a measurement problem if the lineshape is distorted')
    print('     with it - which Step 5 tests.')
elif _resid_x < 0.33:
    print('  -> LESS signal than predicted. Suspect the coupling capacitors being')
    print('     smaller than marked, or more cable capacitance than C_STRAY_EST.')
else:
    print('  -> consistent with coupling through the capacitors alone.')

# ---- what these measurements do and do not determine ------------------------
print('')
print('WHAT IS ACTUALLY PINNED DOWN')
_sum_model = (C_COUPLE + C_STRAY_EST)
_sum_meas = C_COUPLE * (_v_tank / max(G_FWD_PASSIVE, 1e-12))
print('  C_COUPLE + stray, from the measured transfer : %.1f pF' % (_sum_meas * 1e12))
print('  C_COUPLE + stray, from the config values     : %.1f pF' % (_sum_model * 1e12))
print('  -> the SUM is determined; the split rests on the marked %.2f pF and'
      % (C_COUPLE * 1e12))
print('     scales with it.')
# The degeneracy breaks if the pickup capacitor is taken out of the path: the
# tank then sees the whole input capacitance instead of ~C_COUPLE, and the
# frequency shift gives it from L and f0 alone - no capacitor value assumed.
print('  TO SETTLE IT: bypass the pickup cap and read the new f0 - C_in then')
print('  follows from L and f0 alone, no capacitor value assumed:')
for _ci in (10e-12, 20e-12, 30e-12):
    _fn = 1.0 / (2.0 * np.pi * np.sqrt(L_COIL * (C_IMPLIED + _ci - C_COUPLE)))
    print('     C_in = %2.0f pF -> f0 %.2f MHz' % (_ci * 1e12, _fn / 1e6))


---
# Step 5 — A second, independent width

The discipline that replaces a known answer is **measuring the same quantity two ways that fail
differently**.

**Why it is no longer the phase.** `sweep()` retunes two oscillators per point with two separate
register writes, and between them `Δφ = 2π(f_new − f_old)·Δt` opens up and accumulates —
radians per point, indistinguishable from a transport delay of tens of microseconds. `Δt` jitters,
so it cannot be calibrated out. Every fit here uses `|z|²` only.

**What replaces it: receiver convergence.** Generator and receiver LO track the same frequency
word, so every swept tone lands at DC. Repeating at another decimation tests coherent SNR, FIR
status and settling rather than passband convolution.


In [ ]:
# =============================================================================
#  STEP 5 - THE SAME WIDTH, THROUGH A DIFFERENT RECEIVER
# =============================================================================
print('=' * 68)
print('STEP 5 - a second, independent width')
print('=' * 68)

DEC_ALT = int(min(12, DEC_USE + 1))
if DEC_ALT == DEC_USE:
    DEC_ALT = int(max(8, DEC_USE - 1))

for _tag, _d in (('primary  ', DEC_USE), ('alternate', DEC_ALT)):
    print('%s : tracked zero-IF R = %5d, output rate %6.1f kHz, FIR corrected'
          % (_tag, 1 << _d, FS_CLK / (1 << _d) / 1e3))
print('')

acq.start(DEC_ALT)
grid5 = F0 + np.linspace(-2.5, 2.5, 61) * LW
f5, z5 = sweep(grid5, drive_frac=DRIVE_FRAC, navg=N_AVG, level_guard=0.95)
gen.amp(0.0)
p5 = np.abs(z5) ** 2
acq.start(DEC_USE)                       # put the receiver back

try:
    FIT_ALT = fit_lorentzian(f5, p5)
    LW_ALT = FIT_ALT['lw']
    Q_ALT = FIT_ALT['Q']
    _rel = LW_ALT / LW - 1.0
    print('  width at R = %-5d : %.2f kHz, Q = %.1f' % (1 << DEC_USE, LW / 1e3, Q_SWEEP))
    print('  width at R = %-5d : %.2f kHz, Q = %.1f' % (1 << DEC_ALT, LW_ALT / 1e3, Q_ALT))
    print('  centre moved by    : %+.2f kHz  (%+.3f linewidths)'
          % ((FIT_ALT['f0'] - F0) / 1e3, (FIT_ALT['f0'] - F0) / LW))
    print('  disagreement       : %+.1f%%' % (100 * _rel))
    print('-' * 68)
    if abs(_rel) <= 0.10:
        print('  AGREE - the width is the coil\'s, not the receiver\'s')
    else:
        print('  DISAGREE by more than 10%. The receiver window is shaping the')
        print('  measurement. Raise the decimation until the two agree, and re-run')
        print('  Step 3 before trusting Q.')
except Exception as exc:
    FIT_ALT = None
    LW_ALT = Q_ALT = None
    print('  the alternate-decimation fit refused: %s' % exc)
    print('  No second width. Treat Q as unconfirmed.')


In [ ]:
# --- Step 5 picture: the two receivers on one axis ---------------------------
#  Both curves normalised to their own fitted peak, so what is being compared is
#  the SHAPE. Axis limits from the fits, so this is readable at any Q.
if FIT_ALT is not None:
    _span = 2.5 * max(LW, LW_ALT)
    _fine = np.linspace(-_span, _span, 600)
    fig, ax = plt.subplots(figsize=(7.6, 3.8))
    for _f, _p, _ft, _d, _c in ((f3, p3, FIT, DEC_USE, CLR_BLUE),
                                (f5, p5, FIT_ALT, DEC_ALT, CLR_ORANGE)):
        _n = max(_ft['peak'], 1e-300)
        ax.plot((_f - _ft['f0']) / 1e3, (_p - _ft['base']) / _n, 'o', ms=3.5,
                color=_c, alpha=0.7,
                label='R = %d,  lw %.2f kHz,  Q %.1f' % (1 << _d, _ft['lw'] / 1e3, _ft['Q']))
        ax.plot(_fine / 1e3, 1.0 / (1.0 + (2.0 * _fine / _ft['lw']) ** 2), '-',
                lw=1.3, color=_c)
    ax.set_xlim(-_span / 1e3, _span / 1e3)
    ax.set_xlabel('offset from that fit\'s own centre / kHz')
    ax.set_ylabel('power, baseline removed and peak normalised')
    ax.set_title('Step 5 - the same line through two receivers  (%+.1f%% apart)'
                 % (100 * (LW_ALT / LW - 1.0)))
    ax.legend(fontsize=8)
    plt.show()
    print('If the two shapes lie on top of each other, the width belongs to the')
    print('coil. If the wider receiver gives the wider line, it belongs to the')
    print('receiver.')


---
# Step 6 — Is the measurement linear?

The definitive test for "am I loading the coil?" is to weaken the coupling and watch Q stop
changing. With the capacitors soldered on that is unavailable, so this is the test that is.

Drive at three levels and fit each. A **linear** system returns the same centre and width at all
three; only the height changes. If either moves, something is compressing — the input stage, the
coil, or the amplifier — and every number taken at the higher level is suspect.

The scaling column (peak power against drive squared) is the least reliable of the three and
fails for reasons that are not the coil: a fitted floor comparable with the peak, or a DAC
amplitude register down at a few dozen counts, where its own quantisation costs several percent
of the fundamental. **Lowering the drive makes that worse, not better** — which is why Step 4b
finds the window by measurement instead.


In [ ]:
# =============================================================================
#  STEP 6 - LINEARITY
# =============================================================================
print('=' * 68)
print('STEP 6 - linearity against drive level')
print('=' * 68)

acq.start(DEC_USE)
grid6 = F0 + np.linspace(-3.0, 3.0, 41) * LW

lin_rows = []
lin_floor, lin_reg = [], []
for _fr in DRIVE_LIN_FRACS:
    frac = _fr * DRIVE_FRAC
    f6, z6 = sweep(grid6, drive_frac=frac, navg=N_AVG, level_guard=0.95)
    gen.amp(0.0)
    fit6 = fit_lorentzian(f6, np.abs(z6) ** 2)
    lin_rows.append((frac, fit6))
    _reg = int(round(frac * 8192))
    _floor = fit6['base'] / max(fit6['peak'], 1e-300)
    lin_floor.append(_floor); lin_reg.append(_reg)
    print('  drive %8.5f (%+5.1f dB, set_amp %4d) : f0 = %.6f MHz, lw = %6.2f kHz, '
          'Q = %6.1f, floor/peak = %.2f'
          % (frac, 20 * np.log10(_fr / DRIVE_LIN_FRACS[0]), _reg,
             fit6['f0'] / 1e6, fit6['lw'] / 1e3, fit6['Q'], _floor))

lin_f0 = np.array([r[1]['f0'] for r in lin_rows])
lin_q = np.array([r[1]['Q'] for r in lin_rows])
lin_pk = np.array([r[1]['peak'] for r in lin_rows])
lin_d = np.array([r[0] for r in lin_rows])

# NOT d_f0: Step 7 uses that name for its array of fitted centres.
d_f0_spread = (lin_f0.max() - lin_f0.min()) / LW
d_q = (lin_q.max() - lin_q.min()) / lin_q.mean()
# a linear system's peak power scales as the square of the drive
scale_err = lin_pk / (lin_pk[0] * (lin_d / lin_d[0]) ** 2) - 1.0

print('')
print('  centre spread across the ladder : %.3f linewidths' % d_f0_spread)
print('  Q spread across the ladder      : %.1f%%' % (100 * d_q))
print('  peak power vs drive-squared     : %s'
      % ', '.join('%+.1f%%' % (100 * e) for e in scale_err))
print('-' * 68)
_shape_ok = (d_f0_spread < 0.10) and (d_q < 0.05)
_scale_ok = np.max(np.abs(scale_err)) < 0.15
LINEAR_OK = _shape_ok and _scale_ok

# Is the test even answerable at this signal level? Two ways it is not: the
# fitted floor swamping the peak, and the DAC running out of counts.
_floor_bad = max(lin_floor) > 0.30 if lin_floor else False
_reg_bad = min(lin_reg) < 100 if lin_reg else False
print('  smallest set_amp used  : %d counts%s'
      % (min(lin_reg) if lin_reg else 0,
         '   <- too coarse for a scaling test' if _reg_bad else ''))
print('  worst floor / peak     : %.2f%s'
      % (max(lin_floor) if lin_floor else 0.0,
         '   <- the fit is measuring floor, not signal' if _floor_bad else ''))
print('-' * 68)
if LINEAR_OK:
    print('PASS - centre, width and scaling are all drive-independent')
elif _shape_ok and (_floor_bad or _reg_bad):
    LINEAR_OK = True
    print('INCONCLUSIVE ON SCALING, BUT SHAPE IS CLEAN.')
    print('  Centre and width are drive-independent (%.3f linewidths, %.1f%%),'
          % (d_f0_spread, 100 * d_q))
    print('  which is what non-linearity would move first. Only the power-versus-')
    print('  drive-squared column fails, and it cannot be trusted here:')
    if _floor_bad:
        print('    - the fitted floor is %.0f%% of the peak, so "peak" is mostly floor'
              % (100 * max(lin_floor)))
    if _reg_bad:
        print('    - the DAC amplitude register runs down to %d counts, where its'
              % min(lin_reg))
        print('      own quantisation costs several percent of the fundamental')
    print('  LOWERING THE DRIVE MAKES THIS WORSE. Step 4b already found the usable')
    print('  window by measurement and adopted the top of it.')
else:
    print('CHECK - the centre or width moves with drive level. That is real')
    print('        non-linearity. Set DRIVE_FRAC to the LOWER end of the Step 4b')
    print('        window, turn AUTO_CALIBRATE off, and repeat Steps 3 and 5.')


In [ ]:
# --- Step 6 picture ----------------------------------------------------------
fig, (a1, a2, a3) = plt.subplots(3, 1, figsize=(7.2, 6.4), sharex=True)

a1.semilogx(lin_d, (lin_f0 - F0) / LW, 'o-', color=CLR_BLUE)
a1.axhspan(-0.10, 0.10, color=CLR_GREEN, alpha=0.15, label='drive-independent')
a1.set_ylabel('centre / linewidths')
a1.set_title('Step 6 - linearity  (%s)' % ('PASS' if LINEAR_OK else 'CHECK'))
a1.legend(fontsize=8)

a2.semilogx(lin_d, lin_q, 'o-', color=CLR_ORANGE)
a2.axhline(Q_SWEEP, ls=':', lw=1.0, color=CLR_GREY, label='Q from Step 4b')
a2.set_ylabel('fitted Q')
a2.legend(fontsize=8)

a3.loglog(lin_d, lin_pk, 'o', color=CLR_GREEN, label='fitted peak power')
a3.loglog(lin_d, lin_pk[0] * (lin_d / lin_d[0]) ** 2, '--', lw=1.1, color=CLR_GREY,
          label='drive squared')
a3.set_xlabel('drive (fraction of full scale)')
a3.set_ylabel('peak power')
a3.legend(fontsize=8)
plt.show()

print('Top two panels are the test that matters: a linear system moves neither.')
print('The bottom panel fails for reasons that are not the coil - a fitted floor')
print('comparable with the peak, or a DAC register down at a few dozen counts.')


---
# Step 7 — Does anything move while you are measuring?

The emulated coil was exactly constant. A real one answers to room temperature, to airflow and to
anything mechanical — and if the centre moves during a feedback map, the map smears in a way that
looks like real structure.

Two quantities, easy to conflate: **drift** is a slow one-way trend that biases a long scan, and
**scatter** is point-to-point wobble that sets the floor on any single measurement. Both are
reported in **linewidths**, the unit in which they either matter or do not, and the verdict is
computed against the actual cost of the Step 8c map at its configured size.


In [ ]:
# =============================================================================
#  STEP 7 - DRIFT AND SCATTER
# =============================================================================
print('=' * 68)
print('STEP 7 - drift over %.0f minutes, sampled every %.0f s'
      % (DRIFT_MINUTES, DRIFT_PERIOD))
print('=' * 68)

acq.start(DEC_USE)
grid7 = F0 + np.linspace(-2.0, 2.0, 21) * LW      # short: this runs many times

t_start = time.time()
d_t, d_f0, d_q = [], [], []

while time.time() - t_start < DRIFT_MINUTES * 60.0:
    t_point = time.time()
    try:
        f7, z7 = sweep(grid7, drive_frac=DRIVE_FRAC, navg=32, level_guard=0.95)
        gen.amp(0.0)
        fit7 = fit_lorentzian(f7, np.abs(z7) ** 2)
        d_t.append(t_point - t_start)
        d_f0.append(fit7['f0'])
        d_q.append(fit7['Q'])
        print('  %6.1f s : f0 %+9.1f Hz from start, Q = %6.1f'
              % (d_t[-1], d_f0[-1] - d_f0[0], d_q[-1]))
    except Exception as exc:
        # One bad fit must not end a twenty-minute run. Record and continue.
        print('  %6.1f s : fit failed (%s)' % (time.time() - t_start, exc))
    time.sleep(max(0.0, DRIFT_PERIOD - (time.time() - t_point)))

gen.off()
d_t = np.array(d_t); d_f0 = np.array(d_f0); d_q = np.array(d_q)

DRIFT_VALID = False
DRIFT_Q_SCATTER_FRAC = float('nan')
DRIFT_OBSERVED_MINUTES = float(d_t[-1] / 60.0) if len(d_t) else 0.0
if len(d_t) < 4:
    # Do not leave the summary printing nan. If the fits failed this often the
    # cause is upstream and saying so is more use than a blank.
    DRIFT_HZ_PER_MIN = SCATTER_HZ = float('nan')
    print('ONLY %d USABLE POINTS OUT OF %d ATTEMPTS - drift not characterised.'
          % (len(d_t), max(1, int(DRIFT_MINUTES * 60.0 / max(DRIFT_PERIOD, 1e-9)))))
    print('The fits themselves are failing, not the coil. Check the Step 3')
    print('residual and the Step 4b drive window before re-running.')
else:
    slope, icept = np.polyfit(d_t, d_f0, 1)
    resid = d_f0 - (slope * d_t + icept)
    DRIFT_HZ_PER_MIN = slope * 60.0
    SCATTER_HZ = float(np.std(resid))
    total = slope * d_t[-1]
    print('')
    print('  drift    : %+.1f Hz/min  =  %+.4f linewidths/min'
          % (DRIFT_HZ_PER_MIN, DRIFT_HZ_PER_MIN / LW))
    print('  over the whole run: %+.0f Hz = %+.3f linewidths' % (total, total / LW))
    print('  scatter  : %.1f Hz  =  %.4f linewidths' % (SCATTER_HZ, SCATTER_HZ / LW))
    DRIFT_Q_SCATTER_FRAC = float(np.std(d_q, ddof=1) / np.mean(d_q))
    DRIFT_VALID = bool(DRIFT_MINUTES >= 5.0 and DRIFT_OBSERVED_MINUTES >= 4.5
                       and np.isfinite(DRIFT_HZ_PER_MIN)
                       and np.isfinite(DRIFT_Q_SCATTER_FRAC))
    print('  Q scatter: %.1f%%' % (100 * DRIFT_Q_SCATTER_FRAC))
    print('  boundary-diagnostic drift gate: %s'
          % ('PASS' if DRIFT_VALID else 'STOP - need a complete >=5 minute record'))
    print('-' * 68)
    # A feedback map takes minutes; if the centre moves an appreciable fraction
    # of a linewidth in that time the map is of several different systems. The
    # cost is computed from the map's CONFIGURED size, not from a remembered
    # figure, so changing FB_MAP_* changes this verdict with it.
    map_minutes = (FB_MAP_PHASES * FB_MAP_GAINS
                   * (FB_MAP_NPTS * (FB_MAP_SETTLE
                                     + (2 * FB_MAP_NAVG + GUARD_NSAMP) * 1.0e-5)
                      + 0.05)) / 60.0
    move = abs(DRIFT_HZ_PER_MIN) * map_minutes / LW
    print('  the Step 8c map as configured takes about %.1f minutes, over which the'
          % map_minutes)
    print('  centre would move %.3f linewidths' % move)
    if move > 0.10:
        print('  -> enough to smear the map. Either shorten the scan (fewer phases')
        print('     or gains), run a servo during it, or re-centre between rows.')
    else:
        print('  -> negligible. The scan can run open-loop.')

    fig, (b1, b2) = plt.subplots(2, 1, figsize=(7.5, 4.6), sharex=True)
    b1.plot(d_t / 60.0, (d_f0 - d_f0[0]) / 1e3, 'o-', ms=3.5, lw=1.0)
    b1.plot(d_t / 60.0, (slope * d_t + icept - d_f0[0]) / 1e3, '-', lw=1.2,
            color=CLR_ORANGE, label='linear drift')
    b1.set_ylabel('centre shift / kHz')
    b1.legend(fontsize=8)
    b1.set_title('Step 7 - drift %+.1f Hz/min, scatter %.1f Hz'
                 % (DRIFT_HZ_PER_MIN, SCATTER_HZ))
    b2.plot(d_t / 60.0, d_q, 'o-', ms=3.5, lw=1.0, color=CLR_GREEN)
    b2.set_xlabel('time / minutes')
    b2.set_ylabel('Q')
    plt.show()


---
# Step 8 — Feedback against a real coil

Feedback takes the coil's own signal, shifts its phase, scales it and adds it back, changing
**Q** — feedback aligned with the coil's motion cancels part of its loss — and the **centre**,
where feedback aligned with its displacement acts like retuning.

Five things differ from every previous stage, and all are handled by measuring, not assuming.

**1. The phase axis has no absolute meaning.** Cables, connectors and the coil add delay the
latency correction cannot know about, so the two special phases are *found* and everything is
reported relative to them. A fixed offset is expected.

**2. There is no overload flag.** The emulator's sticky clip bit has been the guard for ten
stages and it is gone. Every point checks the input level and abandons rather than fitting
nonsense.

**3. The loop may not have enough gain, and that is a result.** The feedback crosses both
coupling capacitors, carrying the divider *twice*. If `K = A × G_fwd` cannot reach a percent or
so, no phase looks different from any other — and the notebook says so up front rather than
letting an `argmax` invent a narrowing phase out of fit noise.

**4. Everything is measured differentially.** Feedback is antisymmetric in phase — what narrows
the line at φ broadens it at φ+180 — while drift, baseline bias and a sagging drive are not.
Subtracting the opposite phase cancels those, doubles the signal, and the result is checked
against the measurement's own scatter before any phase is named.

**5. Self-oscillation is approached, not entered.** Past a certain gain the coil sustains itself.
On the emulator that was safe; here it saturates the input, and a saturated input still looks
vaguely like a resonance. The boundary is **extrapolated** from the linear region.


In [ ]:
# =============================================================================
#  STEP 8a - THE PHASE AXIS: find the two special phases
# =============================================================================
print('=' * 68)
print('STEP 8 - feedback on the real coil')
print('=' * 68)
CRYOGENIC_BASELINE_READY = bool(ENVIRONMENT_PROFILE != '4K' or
    (globals().get('Q_ALT') is not None and globals().get('LINEAR_OK',False) and
     globals().get('DRIFT_VALID',False) and globals().get('HIGH_Q_FILTER_VALID',False)))
if ENVIRONMENT_PROFILE == '4K' and not CRYOGENIC_BASELINE_READY:
    print('SOFT WARNING - 4 K baseline has not converged across drive, decimation and drift;')
    print('measurements continue for diagnosis but cannot qualify a later feedback map.')

SA_LEVELS_CONFIRMED = bool(np.isfinite(MEASURED_GAIN_INCREASE_DB) and
                           np.isfinite(MEASURED_SECOND_INPUT_DBM) and
                           np.isfinite(MEASURED_FINAL_OUTPUT_DBM) and
                           7.0 <= MEASURED_GAIN_INCREASE_DB <= 11.0 and
                           MEASURED_SECOND_INPUT_DBM <= SECOND_IN_MAX_DBM and
                           MEASURED_FINAL_OUTPUT_DBM <= AMP_OUT_MAX_DBM)
SA_HARMONICS_CONFIRMED = bool(np.isfinite(MEASURED_WORST_HARMONIC_DBC) and
                              np.isfinite(MEASURED_BROADBAND_RISE_DB) and
                              MEASURED_WORST_HARMONIC_DBC <= -40.0 and
                              MEASURED_BROADBAND_RISE_DB <= 3.0)
HIGH_GAIN_CHAIN_CONFIRMED = bool(SCOPE_NO_CLIPPING_CONFIRMED and
                                  OSC_PROTECTION_CONFIRMED and
                                  SA_LEVELS_CONFIRMED and SA_HARMONICS_CONFIRMED)
if not HIGH_GAIN_CHAIN_CONFIRMED:
    print('CAUTION - external 6 dB-chain validation is incomplete. Step 8 will run')
    print('within software headroom guards, but the result is not externally safety-certified.')
    print('The probe-off SEO onset search remains separately locked.')

acq.start(DEC_USE)
whole, frac = ps.config_delay(F0)
print('quarter-period delay line: %d whole clocks + %.3f' % (whole, frac))
print('')

# =============================================================================
#  LOOP-GAIN BUDGET
# =============================================================================
#  The feedback path crosses BOTH coupling capacitors, so the round trip carries
#  the divider twice. Step 4c measured the one-way half as G_FWD, so K = A x G_FWD
#  and K = 1 is self-oscillation.
# =============================================================================
_HEADROOM = 0.95 - DRIVE_FRAC
if _HEADROOM <= 0.0:
    raise RuntimeError('DRIVE_FRAC = %.2f leaves no converter headroom for any '
                       'feedback at all. Lower it before running Step 8.'
                       % DRIVE_FRAC)
#  CLOSED-LOOP. Feedback multiplies the converter signal by 1/(1-K), so dividing
#  the headroom by the OPEN-loop level under-estimates it badly once K is large.
#     DAC:  A <= (H-D) / (D*r + G*(H-D))       ADC:  A <= (1 - D*r/guard) / G
_r = raw_peak / max(DRIVE_FRAC, 1e-9)          # peak-based transfer
_H = 0.95
_a_dac = (_H - DRIVE_FRAC) / max(DRIVE_FRAC * _r + G_FWD_RES * (_H - DRIVE_FRAC), 1e-12)
_a_adc = (1.0 - DRIVE_FRAC * _r / FB_LEVEL_GUARD) / max(G_FWD_RES, 1e-12)
# The driven transfer is not the feedback loop gain. Use only a conservative
# calibration ceiling here; Step 8b measures K/A and sets the campaign ceiling.
FB_CAL_GAIN_MAX = 1.0
FB_GAIN_CEIL = float(min(FB_GAIN_MAX, FB_CAL_GAIN_MAX))
#  Keep the probe clear of the boundary. FB_GAIN_CEIL can sit at K = 1 exactly
#  (the closed-loop headroom formula stops there), and probing ON the boundary
#  makes half the phases oscillate, leaves the cosine fit with too few points,
#  and hands Step 8b a NaN phase.
FB_PROBE_USE = float(min(FB_GAIN_PROBE, FB_GAIN_CEIL, 0.65))
FB_GAINS_USE = [round(fr * FB_GAIN_CEIL, 4) for fr in FB_GAIN_FRACS]

print('  engineering headroom estimates: DAC %.2f, ADC %.2f' % (_a_dac, _a_adc))
print('  calibration gain ceiling: %.2f; measured paired K sets later gains'
      % FB_GAIN_CEIL)
if FB_GAIN_CEIL < 0.5 * FB_GAIN_MAX:
    _d_ok = 0.9 * (_H - 0.0) / (1.0 + FB_GAIN_MAX * _r / max(1.0 - FB_GAIN_MAX * G_FWD_RES, 0.05))
    print('  *** THE DRIVE IS EATING THE FEEDBACK RANGE: at DRIVE_FRAC = %.2f'
          % DRIVE_FRAC)
    print('  only A = %.2f fits (K = %.3f). Lower it to about %.2f and re-run.'
          % (FB_GAIN_CEIL, FB_GAIN_CEIL * G_FWD_RES, max(0.05, min(0.3, _d_ok))))

#  G_FWD_RES, not G_FWD: feedback acts only on the part of the returning signal
#  that came out of the resonator. The off-resonance floor is fed back too, but
#  it never touches the coil, so counting it over-predicts the loop gain.
K_MAX_POSSIBLE = FB_GAIN_CEIL * G_FWD_RES
K_AT_PROBE = FB_PROBE_USE * G_FWD_RES
print('loop-gain budget')
print('  one-way transfer, all of it    : %.5f  (%.1f dB)'
      % (G_FWD, 20 * np.log10(max(G_FWD, 1e-12))))
print('  ...of which went via the coil  : %.5f  (%.1f dB)   <- what feedback acts on'
      % (G_FWD_RES, 20 * np.log10(max(G_FWD_RES, 1e-12))))
print('  ...the rest is the floor       : %.0f%% of the returning amplitude'
      % (100 * (1.0 - G_FWD_RES / max(G_FWD, 1e-12))))
print('  converter headroom for feedback: %.3f of full scale' % _HEADROOM)
print('  largest gain that fits in it   : %.2f' % FB_GAIN_CEIL)
print('  gain ceiling actually used     : %.2f%s'
      % (FB_GAIN_CEIL,
         '  (capped by headroom, not by FB_GAIN_MAX = %.2f)' % FB_GAIN_MAX
         if FB_GAIN_CEIL < FB_GAIN_MAX else '  (the register ceiling)'))
print('  driven-transfer K estimate only: %.4f (not a measured loop K)'
      % K_MAX_POSSIBLE)
print('  estimated Q change only        : %+.1f%%'
      % (100 * (1.0 / (1.0 - min(K_MAX_POSSIBLE, 0.99)) - 1.0)))
print('  K at the probe gain %.2f       : %.4f  (Q %+.1f%%)'
      % (FB_PROBE_USE, K_AT_PROBE,
         100 * (1.0 / (1.0 - min(K_AT_PROBE, 0.99)) - 1.0)))
print('  drive + feedback at the DAC    : %.3f of full scale'
      % (DRIVE_FRAC + FB_GAIN_CEIL * raw_peak))

if K_MAX_POSSIBLE < 0.01:
    print('')
    print('  *** THIS SWEEP CANNOT SHOW A FEEDBACK EFFECT. ***')
    print('  Even at the register ceiling the loop gain is %.4f, which moves Q by'
          % K_MAX_POSSIBLE)
    print('  %.1f%% - below what a Lorentzian fit resolves. The sweep will run and'
          % (100 * K_MAX_POSSIBLE))
    print('  return flat Q against phase. That IS the measurement: with both ports')
    print('  coupled through %.1f pF there is not enough gain in the loop to move a'
          % (C_COUPLE * 1e12))
    print('  real coil, and an amplifier in the pickup path is what changes it.')
elif K_MAX_POSSIBLE < 0.05:
    print('')
    print('  Marginal but workable: %.1f%% of Q at the ceiling. The differential'
          % (100 * K_MAX_POSSIBLE))
    print('  analysis below is what makes an effect this small readable.')
else:
    print('')
    print('  Comfortable: %.1f%% of Q at the ceiling.' % (100 * K_MAX_POSSIBLE))
print('')
print('probing %d phases at gain %.2f, %d averages per point'
      % (len(FB_PHASES), FB_PROBE_USE, FB_NAVG))
print('')

# +-4 linewidths, and wider than it looks it needs to be. Feedback both narrows
# the line and shifts it, and a grid that the shifted line has walked out of
# fails to fit at exactly the phases that matter most.
grid8 = F0 + np.linspace(-4.0, 4.0, 33) * LW

fb_phi, fb_q, fb_df = [], [], []
try:
    for phi in FB_PHASES:
        ps.set(FB_PROBE_USE, phi)
        time.sleep(SETTLE)
        gen.freq(F0); gen.amp(DRIVE_FRAC); gen.run(); time.sleep(0.02)
        lvl = sc.peak_frac(GUARD_NSAMP)
        gen.amp(0.0)
        if lvl > FB_LEVEL_GUARD:
            print('  %4d deg : ABANDONED, input at %.2f of full scale' % (phi, lvl))
            fb_phi.append(phi); fb_q.append(np.nan); fb_df.append(np.nan)
            continue
        try:
            f8, z8 = sweep(grid8, drive_frac=DRIVE_FRAC, navg=FB_NAVG,
                           level_guard=FB_LEVEL_GUARD)
            gen.amp(0.0)
            fit8 = fit_lorentzian(f8, np.abs(z8) ** 2)
            # A saturated or oscillating loop still returns a "fit", and it can
            # be absurd. Anything this far from the unfed line is not a
            # measurement of the coil.
            if not (0.02 * LW < fit8['lw'] < 50.0 * LW):
                raise ValueError('width %.1f kHz is nowhere near the %.1f kHz '
                                 'line - loop saturated?'
                                 % (fit8['lw'] / 1e3, LW / 1e3))
            fb_phi.append(phi); fb_q.append(fit8['Q']); fb_df.append(fit8['f0'] - F0)
            print('  %4d deg : Q = %7.1f  (x%.2f),  centre %+8.1f Hz (%+.3f lw)'
                  % (phi, fit8['Q'], fit8['Q'] / Q_SWEEP, fb_df[-1], fb_df[-1] / LW))
        except Exception as exc:
            fb_phi.append(phi); fb_q.append(np.nan); fb_df.append(np.nan)
            print('  %4d deg : fit failed (%s)' % (phi, exc))
finally:
    # The feedback loop is closed while this runs. Any exception, or
    # a Ctrl-C between two phases, must not leave it that way.
    ps.off()
    gen.off()

fb_phi = np.array(fb_phi, dtype=float)
fb_q = np.array(fb_q, dtype=float)
fb_df = np.array(fb_df, dtype=float)

if np.all(~np.isfinite(fb_q)):
    raise RuntimeError('every feedback point failed - check levels and coupling')

# =============================================================================
#  DIFFERENTIAL ANALYSIS - phi against phi + 180
# =============================================================================
#  Feedback is antisymmetric in phase; drift, baseline bias and a sagging drive
#  are not. Subtracting the opposite phase cancels those and doubles the signal.
# =============================================================================
half = len(fb_phi) // 2
_opp = np.array([int(np.argmin(np.abs(((fb_phi - (p + 180.0)) + 180.0) % 360.0 - 180.0)))
                 for p in fb_phi[:half]])
dq = fb_q[:half] - fb_q[_opp]            # antisymmetric: the feedback signature
ddf = fb_df[:half] - fb_df[_opp]

# ---- fit a cosine, do not take an argmax ------------------------------------
#  Theory says the differential IS a cosine. An argmax returns a grid point,
#  is biased to the ends of the half-turn, and discards eight of nine points.
def _fit_cos(phi_deg, y):
    """Least-squares fit of y = D cos(phi - phi0). Returns (D, phi0_deg)."""
    good = np.isfinite(y)
    if int(np.count_nonzero(good)) < 3:
        return float('nan'), float('nan')
    p = np.radians(np.asarray(phi_deg, dtype=float)[good])
    basis = np.column_stack([np.cos(p), np.sin(p)])
    coef, _, _, _ = np.linalg.lstsq(basis, np.asarray(y, dtype=float)[good], rcond=-1)
    return float(np.hypot(coef[0], coef[1])), float(np.degrees(np.arctan2(coef[1], coef[0])))

_n_osc = int(np.sum(~np.isfinite(fb_q)))
if _n_osc:
    print('  %d of %d phases gave no fit (oscillating or level-guarded)'
          % (_n_osc, len(fb_q)))
DQ_AMP, PHI_NARROW = _fit_cos(fb_phi[:half], dq)
DDF_AMP, PHI_SHIFT = _fit_cos(fb_phi[:half], ddf)
PHI_NARROW = float(PHI_NARROW % 360.0)
PHI_SHIFT = float(PHI_SHIFT % 360.0)
PHI_BROAD = (PHI_NARROW + 180.0) % 360.0

# Noise from successive differences along a curve that should be smooth: the
# measurement declaring its own error bar rather than us assuming one.
_fin = fb_q[np.isfinite(fb_q)]
Q_NOISE = float(np.std(np.diff(_fin)) / np.sqrt(2.0)) if len(_fin) > 2 else float('nan')
FB_DETECTED = bool(np.isfinite(Q_NOISE) and DQ_AMP > 3.0 * Q_NOISE)

# Q+ = Q0/(1-K) and Q- = Q0/(1+K), so the differential amplitude is
#     m = DQ_AMP / (2 Q0) = K / (1 - K^2)
# which is K only for small K - it over-reads by 25% at K = 0.5 and diverges at
# K = 1. Invert it exactly instead: K = (-1 + sqrt(1 + 4 m^2)) / (2 m).
#  (Q+ - Q-)/(Q+ + Q-) = K exactly, for every pair, with Q0 cancelled. The
#  cosine amplitude is only used to locate the phase: at large K the curve is
#  Q0/(1 - K cos phi), which is peaked rather than sinusoidal, so its amplitude
#  is not a good measure of K even though its symmetry still fixes the phase.
_sq = fb_q[:half] + fb_q[_opp]
with np.errstate(invalid='ignore', divide='ignore'):
    _kp = dq / _sq
K_MEASURED = float(np.nanmax(_kp)) if np.any(np.isfinite(_kp)) else 0.0

print('')
print('  differential amplitude       : %.2f%% of Q0' % (100 * DQ_AMP / Q_SWEEP))
print('  MEASURED K                   : %.4f' % K_MEASURED)
print('  predicted K, resonant only   : %.4f' % K_AT_PROBE)
print('  (counting the floor as signal would predict %.4f)'
      % (FB_PROBE_USE * G_FWD))
print('  point-to-point Q noise       : %.2f%% of Q0' % (100 * Q_NOISE / Q_SWEEP))
print('  signal-to-noise              : %.1f' % (DQ_AMP / max(Q_NOISE, 1e-12)))
print('')

if not FB_DETECTED:
    print('  NO FEEDBACK EFFECT DETECTED - and the phases below are meaningless.')
    print('  The modulation is under three times the noise, so the "narrowing')
    print('  phase" is just wherever the noise peaked. Do NOT read it as physics,')
    print('  and do not read Step 8b\'s K either.')
    print('')
    if K_MAX_POSSIBLE < 0.01:
        print('  This agrees with the budget above: the loop simply has no gain.')
        print('  The fix is not in this notebook. Either put a low-noise amplifier')
        print('  in the pickup path before IN1, or couple more strongly than')
        print('  %.1f pF - both of which change the divider, and with it every' % (C_COUPLE * 1e12))
        print('  number Step 4c predicted.')
    else:
        print('  The budget said an effect SHOULD be visible, so this disagreement')
        print('  is worth chasing: check the delay line configured above, check')
        print('  that the phase shifter is not bypassed, and re-run Step 4c to')
        print('  confirm the coupling has not changed.')
else:
    print('  narrowing phase (cosine fit)   : %.1f deg' % PHI_NARROW)
    print('  broadening phase (opposite)    : %.1f deg' % PHI_BROAD)
    print('  largest centre shift at        : %.1f deg' % PHI_SHIFT)
    _sep = (PHI_SHIFT - PHI_NARROW) % 360.0
    print('  separation between them        : %.1f deg  (theory: 90 or 270)' % _sep)
    _off = min(abs(_sep - 90.0), abs(_sep - 270.0))
    if _off > 35.0:
        print('  WARNING: the two effects should be a quarter turn apart and are')
        print('  %.0f degrees off it. At a signal-to-noise of %.1f that is unlikely'
              % (_off, DQ_AMP / max(Q_NOISE, 1e-12)))
        print('  to be scatter - suspect the loop compressing, or a second path')
        print('  with its own phase (Step 4d compares the measured transfer against the model).')
    else:
        print('  -> a quarter turn, as theory demands. The phase axis is sound.')

    _ratio = K_MEASURED / max(FB_PROBE_USE * G_FWD_RES, 1e-12)
    print('')
    print('  measured K / predicted K : %.2fx' % _ratio)
    print('  round-trip transfer from the loop : %.5f  (%.1f dB)'
          % (K_MEASURED / FB_PROBE_USE,
             20 * np.log10(max(K_MEASURED / FB_PROBE_USE, 1e-12))))
    print('  The measured value is the measurement; the budget rests on estimated')
    print('  capacitances, so a factor of two either way means little.')
    if not (0.33 < _ratio < 3.0):
        print('  -> over 3x apart: compare the transfer above with Step 4b.')


In [ ]:
# --- Step 8a picture ---------------------------------------------------------
#  The differential curves are the measurement; the raw ones are shown faint
#  behind them so a common-mode drift is visible for what it is.
fig, (c1, c2) = plt.subplots(2, 1, figsize=(7.5, 5.4), sharex=True)

_rel = (fb_phi[:half] - PHI_NARROW + 180.0) % 360.0 - 180.0
_o = np.argsort(_rel)
_relall = (fb_phi - PHI_NARROW + 180.0) % 360.0 - 180.0
_oa = np.argsort(_relall)

c1.plot(_relall[_oa], (fb_q[_oa] - np.nanmean(fb_q)) / Q_SWEEP * 100.0, '.',
        ms=4, color='0.75', label='raw, mean removed')
c1.plot(_rel[_o], dq[_o] / Q_SWEEP * 100.0, 'o-', ms=4, lw=1.1,
        color=CLR_BLUE, label='differential, phi - (phi+180)')
c1.axhline(0.0, ls=':', lw=1.0, color=CLR_GREY)
if np.isfinite(Q_NOISE):
    c1.axhspan(-300.0 * Q_NOISE / Q_SWEEP, 300.0 * Q_NOISE / Q_SWEEP,
               color=CLR_RED, alpha=0.12, label='3x noise')
c1.set_ylabel('Q change / % of Q0')
c1.set_title('Step 8 - feedback at gain %.2f   [%s]'
             % (FB_PROBE_USE,
                'effect detected' if FB_DETECTED else 'NO effect above the noise'))
c1.legend(fontsize=8)

c2.plot(_relall[_oa], (fb_df[_oa] - np.nanmean(fb_df)) / LW, '.', ms=4, color='0.75')
c2.plot(_rel[_o], ddf[_o] / LW, 'o-', ms=4, lw=1.1, color=CLR_ORANGE)
c2.axhline(0.0, ls=':', lw=1.0, color=CLR_GREY)
c2.set_xlabel('feedback phase relative to the narrowing phase / degrees')
c2.set_ylabel('centre shift / linewidths')
plt.show()

if FB_DETECTED:
    print('The two curves should be a quarter turn apart: where the Q difference')
    print('peaks the centre-shift difference crosses zero, and vice versa. The SIGN')
    print('of the lower curve depends on how the feedback reaches the coil and is')
    print('not expected to match the paper - their loop couples through a')
    print('capacitor, this one sums into the drive.')
else:
    print('Both curves are consistent with zero. The shaded band is three times the')
    print('measurement\'s own point-to-point noise; a real feedback effect would')
    print('leave it. Nothing here is a statement about the coil - it is a statement')
    print('about how much gain survives two 0.3 pF capacitors.')


In [ ]:
# =============================================================================
#  STEP 8b - HOW THE EFFECT SCALES WITH GAIN, and where the boundary is
# =============================================================================
#  Define K from the measured quality factor by Q_eff = Q0 / (1 - K). K is then
#  the fraction of the coil's loss the feedback is cancelling, it should be
#  proportional to the gain, and K = 1 is the self-oscillation boundary.
#
#  The boundary is EXTRAPOLATED from the linear region, never approached. See
#  the note above this step.
# =============================================================================
print('')
print('STEP 8b - K against gain at the narrowing phase')
print('-' * 68)

#  Measure each gain at BOTH phi_narrow and phi_narrow + 180. With
#  Q+ = Q0/(1-K) and Q- = Q0/(1+K),
#      (Q+ - Q-) / (Q+ + Q-) = K
#  exactly, with Q0 cancelled - no reference measurement, no drift term.
def _q_at(phi, A):
    """Quality factor with the loop closed at gain A and phase phi."""
    ps.set(A, phi)
    time.sleep(SETTLE)
    gen.freq(F0); gen.amp(DRIVE_FRAC); gen.run(); time.sleep(0.02)
    lvl = sc.peak_frac(GUARD_NSAMP)
    gen.amp(0.0)
    if lvl > FB_LEVEL_GUARD:
        raise RuntimeError('input at %.2f of full scale' % lvl)
    fq, zq = sweep(grid8, drive_frac=DRIVE_FRAC, navg=FB_NAVG,
                   level_guard=FB_LEVEL_GUARD)
    _f = fit_lorentzian(fq, np.abs(zq) ** 2)
    if not (0.02 * LW < _f['lw'] < 50.0 * LW):
        raise ValueError('width %.1f kHz is nowhere near the %.1f kHz line'
                         % (_f['lw'] / 1e3, LW / 1e3))
    return _f


FB_8B_OK = bool(np.isfinite(PHI_NARROW) and FB_DETECTED)
if not FB_8B_OK:
    # Skip rather than halt: Step 8c is still worth running, and a Lorentzian
    # fitted at a phase that means nothing is worse than no number at all.
    k_slope = A_ONSET = k_free = k_int = float('nan')
    gains_ok, ks = [], []
    print('  SKIPPED - Step 8a found no usable narrowing phase (effect detected'
          ' = %s).' % FB_DETECTED)
    print('  If the probe gain sat on the oscillation boundary, lower')
    print('  FB_GAIN_PROBE; if the loop simply has no gain, this is the answer.')

gains_ok, ks = [], []
if FB_8B_OK:
  try:
      for A in FB_GAINS_USE:
          try:
              fplus = _q_at(PHI_NARROW, A)
              fminus = _q_at(PHI_BROAD, A)
              qp, qm = fplus['Q'], fminus['Q']
              K = (qp - qm) / (qp + qm)
              gains_ok.append(A); ks.append(K)
              print('  gain %.2f : Q+ = %7.1f, Q- = %7.1f  ->  K = %.4f  '
                    '(centre split %+.0f Hz)'
                    % (A, qp, qm, K, fplus['f0'] - fminus['f0']))
          except Exception as exc:
              print('  gain %.2f : abandoned (%s)' % (A, exc))
  finally:
      # Highest gains in the notebook are applied here. Open the loop
      # and kill the drive on every exit path, including Ctrl-C.
      ps.off()
      gen.off()

if len(gains_ok) >= 2:
    gains_ok = np.array(gains_ok); ks = np.array(ks)
    # K(0)=0 is physical, but a line through all points is meaningful only
    # while the measured loop remains in its small-signal linear regime.
    k_slope = float(np.sum(gains_ok * ks) / np.sum(gains_ok ** 2))
    # ALSO fit with a free intercept: anything gain-independent otherwise gets
    # absorbed into the origin-forced slope and biases the loop gain.
    k_free = k_int = float('nan')
    if len(gains_ok) >= 3:
        k_free, k_int = np.polyfit(gains_ok, ks, 1)
        k_free = float(k_free); k_int = float(k_int)
    A_ONSET = 1.0 / k_slope if k_slope > 0 else float('nan')
    FB_GAIN_CEIL = float(min(FB_GAIN_MAX, 0.98 / max(k_slope, 1e-9)))
    pred = k_slope * gains_ok
    lin_err = float(np.max(np.abs(ks - pred))) if len(ks) else float('nan')
    print('')
    print('  K = %.4f x gain          (origin-constrained descriptive fit)' % k_slope)
    if np.isfinite(k_free):
        print('  K = %.4f + %.4f x gain  (free intercept)' % (k_int, k_free))
        if abs(k_int) > 0.25 * float(np.max(np.abs(ks))):
            print('  *** THE INTERCEPT IS NOT NEGLIGIBLE: %.0f%% of the largest K'
                  % (100 * abs(k_int) / float(np.max(np.abs(ks)))))
            print('  measured. K must be zero at zero gain: a non-zero fitted')
            print('  intercept can indicate curvature/compression as well as drift')
            print('  or imperfect phase pairing. Do not use either straight-line')
            print('  slope for onset prediction until Step 8g establishes the')
            print('  drive-independent small-signal K0(A) curve.')
    print('  worst departure from proportional : %.3f' % lin_err)
    _ka = ks / np.maximum(gains_ok, 1e-12)
    print('  K/A across measured gains         : %s'
          % ', '.join('%.3f' % x for x in _ka))
    if np.ptp(_ka) > 0.03 * max(np.mean(_ka), 1e-12):
        print('  WARNING - K/A is not constant. The origin-constrained fit is')
        print('  descriptive only and its K=1 intercept is not justified.')
    print('  driven-transfer estimate (not K)  : %.4f x gain' % G_FWD_RES)
    print('  measured-K campaign ceiling       : A = %.3f' % FB_GAIN_CEIL)
    print('  extrapolated self-oscillation onset at gain %.2f' % A_ONSET)
    if not FB_DETECTED:
        print('')
        print('  DO NOT READ THE NUMBERS ABOVE AS PHYSICS. Step 8a found no effect')
        print('  above the noise, so this fit is a straight line through scatter.')
        print('  K, its slope and the onset gain are all meaningless here.')
    elif A_ONSET >= 1.99:
        print('')
        print('  The onset is beyond the largest coefficient the register can hold')
        print('  (1.99), so self-oscillation is UNREACHABLE with this coupling. That')
        print('  is the honest result for a passively coupled coil rather than a')
        print('  shortcoming of the measurement: the round trip crosses %.1f pF'
              % (C_COUPLE * 1e12))
        print('  twice and loses about %.0f dB, and no setting recovers it.'
              % abs(20 * np.log10(max(G_FWD_RES, 1e-12))))
        print('  An amplifier in the pickup path is the change that would.')
    elif A_ONSET < max(FB_GAINS_USE) * 1.3:
        print('  NOTE: that is close to the gains already used. Do not raise')
        print('  FB_GAIN_FRACS further without watching the input level.')
    print('')
    print('  NOT MEASURED ON PURPOSE: the boundary itself. On the emulator,')
    print('  bisecting into oscillation was safe and informative. Here it')
    print('  saturates the input, and a saturated input still looks vaguely like')
    print('  a resonance. Make that a separate, deliberate experiment.')
else:
    A_ONSET = float('nan')
    k_slope = float('nan')
    print('  too few usable gains to fit K')


---
# Step 8c — The gain/phase map

Herkenhoff *et al.*'s resonator modification scan: Q and detuning over feedback **gain** against
feedback **phase**, unstable region left blank, 1D cuts beneath. Theirs is 128×128; each pixel
here costs a sweep and a fit, so the default is 24×12 — the same object in minutes rather than
days.

**The grid tracks the line**, because feedback narrows the resonance by 1/(1−K) and a span chosen
for the unfed line ends up with barely one point across it.

**Self-oscillation is detected, not approached.** Before each pixel the drive is switched off and
the input read: an unstable loop produces a large signal with no drive at all. Those pixels are
drawn black, as in the paper, and the feedback is opened immediately. Whether any black appears
depends on whether the loop can reach K = 1 — and, with an external amplifier fitted, on whether
doing so is safe.


In [ ]:
# --- Step 8c picture: the resonator modification scan ------------------------
#  Laid out as Herkenhoff's thesis does it: phase on -180..+180, gain in dB,
#  the quality factor on a LOGARITHMIC colour scale, the detuning on a symmetric
#  one, and the unstable region left black.
_rel = ((map_phi - PHI_NARROW + 180.0) % 360.0) - 180.0
_o = np.argsort(_rel)
_Q = np.ma.masked_invalid(MAP_Q[:, _o])
_D = np.ma.masked_invalid(MAP_DF[:, _o] / 1e3)
_cq, _cd = CMAP_SEQ, CMAP_DIV
_ext = [_rel[_o][0], _rel[_o][-1], map_gain_db[0], map_gain_db[-1]]
_dlim = float(np.nanmax(np.abs(MAP_DF / 1e3))) or 1.0

# cuts: one across phase at high gain, one across gain at the narrowing phase
_cg = int(np.argmin(np.abs(map_gain - 0.8 * map_gain[-1])))
_cp = int(np.argmin(np.abs(_rel[_o])))

fig = plt.figure(figsize=(9.6, 7.4))
for k, (dat, cmap, ttl, lab, norm) in enumerate([
        (_Q, _cq, '(a)  quality factor', 'Q',
         matplotlib.colors.LogNorm(vmin=max(np.nanmin(MAP_Q), 1.0),
                                   vmax=np.nanmax(MAP_Q))),
        (_D, _cd, '(b)  detuning of the resonance', r'$\Delta\nu_\mathrm{res}$ / kHz',
         matplotlib.colors.Normalize(vmin=-_dlim, vmax=_dlim))]):
    ax = fig.add_subplot(2, 2, k + 1)
    ax.grid(False)
    im = ax.imshow(dat, origin='lower', aspect='auto', extent=_ext,
                   cmap=cmap, norm=norm, interpolation='nearest')
    ax.axhline(map_gain_db[_cg], ls='--', lw=1.0, color='w')
    ax.axvline(_rel[_o][_cp], ls='--', lw=1.0, color='w')
    ax.set_xticks([-180, -90, 0, 90, 180])
    ax.set_xlabel(r'$\arg\{A\}$ rel. to narrowing / $^\circ$')
    ax.set_ylabel(r'$|A|$ / dB')
    ax.set_title(ttl)
    fig.colorbar(im, ax=ax).set_label(lab)

    ax2 = fig.add_subplot(2, 2, k + 3)
    ax2.plot(_rel[_o], dat[_cg, :], 'o-', color=CLR_BLUE,
             label='across phase, %+.0f dB' % map_gain_db[_cg])
    ax2.set_xticks([-180, -90, 0, 90, 180])
    ax2.set_xlabel(r'$\arg\{A\}$ rel. to narrowing / $^\circ$')
    ax2.set_ylabel(lab)
    if k == 0:
        ax2.set_yscale('log')
        ax2.axhline(Q_SWEEP, ls=':', color=CLR_GREY, label='no feedback')
    else:
        ax2.axhline(0.0, ls=':', color=CLR_GREY)
    ax3 = ax2.twiny()
    ax3.plot(map_gain_db, dat[:, _cp], 's-', color=CLR_ORANGE,
             label='across gain, at the narrowing phase')
    ax3.set_xlabel(r'$|A|$ / dB')
    ax3.grid(False)
    _h1, _l1 = ax2.get_legend_handles_labels()
    _h2, _l2 = ax3.get_legend_handles_labels()
    ax2.legend(_h1 + _h2, _l1 + _l2, loc='best')
plt.show()

print('THE TWO PANELS BENEATH ARE CUTS THROUGH THE MAPS, along the dashed lines.')
print('  blue  - across PHASE at fixed gain: Q should peak at 0 (the narrowing')
print('          phase) and dip a half turn away; the detuning should cross zero')
print('          where Q peaks and be extreme a QUARTER turn from it.')
print('  amber - across GAIN at the narrowing phase: Q should rise as Q0/(1-K)')
print('          with K proportional to gain, so it runs away towards the')
print('          boundary; the detuning stays near zero, because at the')
print('          narrowing phase feedback changes damping and not frequency.')
print('Measured: Q %.0f -> %.0f (x%.1f), detuning %+.0f to %+.0f kHz.'
      % (Q_SWEEP, np.nanmax(MAP_Q), np.nanmax(MAP_Q) / Q_SWEEP,
         np.nanmin(MAP_DF) / 1e3, np.nanmax(MAP_DF) / 1e3))


---
# Step 8d — The lineshape family

Herkenhoff's Fig. 6.3 shows the resonator's **noise** spectrum at a series of feedback gains: the
line sharpens and rises with positive feedback, flattens with negative.

**The noise version is not obtainable here** — Step 4c measures the coil's Johnson noise arriving
far below the converter floor, where Herkenhoff's resonator is cryogenic with a low-noise
amplifier at the tank. The **driven** equivalent carries the same information.

**Both signs are available.** The coefficient register cannot go negative, but negative feedback
is simply the opposite phase, `A` at `φ_narrow + 180°` — exactly what the thesis's negative `G̃`
curves are.

**Each setting gets its own sweep span.** Q here spans an order of magnitude, so a fixed span puts
three points across the narrowest line and the Lorentzian stops looking like one. The span is
predicted from the Step 8b slope, then refined by a second pass around what the first found, and
the figure's limits follow: panel (a) from the widest fitted line, panel (b) zoomed onto the
narrowest. Nothing is set by hand, so the same cell works at any Q.


In [ ]:
# =============================================================================
#  STEP 8d - LINESHAPE AT A FAMILY OF FEEDBACK GAINS  (thesis Fig. 6.3)
# =============================================================================
#  Each setting is swept on ITS OWN grid. Feedback narrows the line by 1/(1-K)
#  and shifts it, so a span fixed at the unfed linewidth puts three points across
#  the narrowest curve - and three points do not look like a Lorentzian. Two
#  passes per setting: the first span is predicted from the Step 8b slope, the
#  second is set by what the first actually found. Nothing is entered by hand, so
#  the same cell works on a coil of any Q, warm or cold.
# =============================================================================
print('=' * 68)
print('STEP 8d - lineshape versus feedback gain, both signs')
print('=' * 68)
FB_8D_OK = bool(FB_8B_OK)
if not FB_8D_OK:
    print('SKIPPED - Step 8a found no usable narrowing phase, so there is no')
    print('meaningful axis to sweep along. Not a fault in this step.')

spec_f, spec_p, spec_lbl, spec_fit, spec_A = [], [], [], [], []
if FB_8D_OK:
    acq.start(DEC_USE)
    ps.config_delay(F0)
    #  K per unit gain, for predicting the first span. The Step 8b slope is the
    #  measured one; the budget's G_FWD_RES is the fallback if 8b could not fit.
    K_PER_GAIN = k_slope if np.isfinite(k_slope) and k_slope > 0 else G_FWD_RES

    def _lineshape(f_hint, lw_hint, dec_log2=None, return_complex=False):
        """Locate on a span wide enough to catch a shifted line, then resweep
        tight around the fit. Returns (f, p, fit)."""
        out = None
        for _pass in (0, 1, 2):
            use_dec = receiver_policy(lw_hint, high_q=(lw_hint < 0.6*LW),
                                      requested=dec_log2)
            acq.start(use_dec)
            span = FB_SPEC_HALFSPAN * max(lw_hint, 0.02 * LW)
            if _pass == 0:
                span = max(span, 1.5 * LW)     # must not miss a shifted line
            grid = adaptive_frequency_grid(f_hint, max(lw_hint, span/HIGH_Q_WING_LW),
                                           points_per_lw=HIGH_Q_POINTS_PER_LW,
                                           wing_lw=max(HIGH_Q_WING_LW, span/max(lw_hint,1.0)))
            use_settle = adaptive_settle(lw_hint, use_dec, 2.0 if _pass == 2 else 1.0)
            fs_, zs_ = sweep(grid, drive_frac=DRIVE_FRAC, navg=FB_NAVG,
                             settle=use_settle, level_guard=FB_LEVEL_GUARD)
            ft = fit_lorentzian(fs_, np.abs(zs_) ** 2)
            if not (0.005 * LW < ft['lw'] < 50.0 * LW):
                raise ValueError('width %.1f kHz is nowhere near the %.1f kHz line'
                                 % (ft['lw'] / 1e3, LW / 1e3))
            hp = half_power_width(fs_, np.abs(zs_)**2, base=ft.get('base'))
            edge = (ft['f0']-ft['lw'] <= fs_.min() or ft['f0']+ft['lw'] >= fs_.max())
            if (_pass < 2 and (edge or hp is None)):
                f_hint, lw_hint = ft['f0'], max(ft['lw'], lw_hint); continue
            f_hint, lw_hint = ft['f0'], ft['lw']
            out = ((fs_, np.abs(zs_) ** 2, ft, zs_) if return_complex else
                   (fs_, np.abs(zs_) ** 2, ft))
        return out

    print('%8s %7s %9s %10s %11s %11s %9s'
          % ('gain/', 'A', 'predicted', 'Q', 'f_res', 'linewidth', 'K'))
    print('%8s %7s %9s %10s %11s %11s %9s'
          % ('ceiling', '', 'lw / kHz', '', 'MHz', 'kHz', ''))
    try:
        for fr in FB_SPEC_FRACS:
            A = abs(fr) * FB_GAIN_CEIL
            phi = PHI_NARROW if fr >= 0 else PHI_BROAD
            #  K = K_PER_GAIN x A, and lw = lw0 (1 - K), so the first span is
            #  already close instead of an order of magnitude too wide.
            _kp = float(np.clip(K_PER_GAIN * A * (1.0 if fr >= 0 else -1.0),
                                -8.0, 0.95))
            _lw_hint = max(LW * (1.0 - _kp), 0.02 * LW)
            ps.set(A, phi)
            time.sleep(SETTLE)
            gen.amp(0.0); time.sleep(0.005)
            if sc.peak_frac(GUARD_NSAMP) > FB_OSC_LEVEL:
                ps.off()
                print('%+8.2f %7.2f %9.1f    oscillating, skipped' % (fr, A, _lw_hint / 1e3))
                continue
            try:
                fs_, pp_, ft = _lineshape(F0, _lw_hint)
                spec_f.append(fs_); spec_p.append(pp_)
                spec_lbl.append(fr); spec_fit.append(ft); spec_A.append(A)
                print('%+8.2f %7.2f %9.1f %10.1f %11.6f %11.2f %+9.3f'
                      % (fr, A, _lw_hint / 1e3, ft['Q'], ft['f0'] / 1e6,
                         ft['lw'] / 1e3, 1.0 - Q_SWEEP / ft['Q']))
            except Exception as exc:
                print('%+8.2f %7.2f %9.1f    %s' % (fr, A, _lw_hint / 1e3, exc))
    finally:
        ps.off(); gen.off()

    if len(spec_p) < 2:
        raise RuntimeError('too few usable spectra')
    _lws = np.array([f['lw'] for f in spec_fit])
    _qs = np.array([f['Q'] for f in spec_fit])
    print('')
    print('  Q spans %.0f to %.0f  (no feedback %.0f)' % (_qs.min(), _qs.max(), Q_SWEEP))
    print('  linewidth spans %.1f to %.1f kHz - a factor of %.1f, which is why each'
          % (_lws.min() / 1e3, _lws.max() / 1e3, _lws.max() / max(_lws.min(), 1e-30)))
    print('  curve needs its own span and the figure needs two x-scales.')


In [ ]:
# --- Step 8d picture: thesis Fig. 6.3 layout, auto-zoomed -------------------
#  Orange for positive feedback, teal for negative, black for none. Panel (a) is
#  scaled by the WIDEST fitted line so nothing is cropped; panel (b) is centred
#  and scaled on the UNFED line - the one the whole family is measured against,
#  and the only curve in the set that is the coil on its own. Both limits come
#  from this run's fits, so the zoom follows the resonator into a trap or a
#  cryostat without anyone editing a number.
if not spec_p or len(spec_p) < 2:
    print('Step 8d produced no data - nothing to plot.')
else:
    _ref = None
    for _lb, _p in zip(spec_lbl, spec_p):
        if _lb == 0.0:
            _ref = float(np.median(_p))
    _ref = _ref or float(np.median(spec_p[0]))

    _lws = np.array([f['lw'] for f in spec_fit])
    _f0s = np.array([f['f0'] for f in spec_fit])
    #  The unfed curve: the fraction nearest zero, which is the setting with the
    #  feedback coefficient at zero. It is the reference the rest of the family
    #  is quoted against, so it is what the zoom is built around.
    _unfed = int(np.argmin(np.abs(np.asarray(spec_lbl, dtype=float))))
    _xa = FB_SPEC_HALFSPAN * _lws.max()
    _xb_c = _f0s[_unfed] - F0
    _xb_w = FB_SPEC_ZOOM_LW * _lws[_unfed]
    _order = sorted(range(len(spec_lbl)), key=lambda i: spec_lbl[i])

    fig, axes = plt.subplots(1, 2, figsize=(11.2, 4.9))
    _panels = ((axes[0], -_xa, _xa, '(a)  the whole family'),
               (axes[1], _xb_c - _xb_w, _xb_c + _xb_w,
                '(b)  zoom on the unfed line, Q = %.0f, lw = %.1f kHz'
                % (spec_fit[_unfed]['Q'], _lws[_unfed] / 1e3)))
    for ax, _lo, _hi, _ttl in _panels:
        for i in _order:
            _lb = spec_lbl[i]; _ft = spec_fit[i]
            if _lb > 0:     col, lwd = '#E8820C', 1.4
            elif _lb < 0:   col, lwd = '#12A5A5', 1.4
            else:           col, lwd = 'k', 1.8
            ax.plot((spec_f[i] - F0) / 1e3,
                    10 * np.log10(np.maximum(spec_p[i], 1e-30) / _ref),
                    '-', lw=lwd, color=col,
                    label='%+5.2f    %6.0f    %10.5f    %7.1f'
                          % (_lb, _ft['Q'], _ft['f0'] / 1e6, _ft['lw'] / 1e3))
        ax.set_xlim(_lo / 1e3, _hi / 1e3)
        ax.set_xlabel(r'$\nu - \nu_\mathrm{res,0}$ / kHz')
        ax.set_ylabel('magnitude / dB')
        ax.set_title(_ttl)
    axes[0].legend(fontsize=7, loc='best', labelspacing=0.25,
                   title=r'$\tilde{G}$        Q       $\nu_\mathrm{res}$/MHz   lw/kHz')
    plt.show()

    # The same numbers as a table, because a legend is for identifying curves and
    # a table is for reading values off.
    print('')
    print('  RESONATOR MODIFICATION - each curve in the figure above')
    print('  %8s %7s %9s %8s %14s %12s %9s'
          % ('G/ceil', 'A', 'phase', 'Q', 'f_res / MHz', 'lw / kHz', 'K'))
    print('  ' + '-' * 72)
    for i in _order:
        _ft = spec_fit[i]
        print('  %+8.2f %7.2f %9s %8.1f %14.6f %12.2f %+9.3f'
              % (spec_lbl[i], spec_A[i],
                 'narrow' if spec_lbl[i] >= 0 else 'broaden',
                 _ft['Q'], _ft['f0'] / 1e6, _ft['lw'] / 1e3,
                 1.0 - Q_SWEEP / _ft['Q']))
    print('  ' + '-' * 72)
    print('  %8s %7s %9s %8.1f %14.6f %12.2f' % ('unfed', 0.0, '-', Q_SWEEP,
                                                 F0 / 1e6, LW / 1e3))
    print('')
    print('Orange is positive feedback (narrowing, taller), teal is negative')
    print('(broadening, flatter), black is none. Panel (b) is +-%.1f linewidths of'
          % FB_SPEC_ZOOM_LW)
    print('the UNFED line (%.2f kHz), which is the curve everything else is'
          % (_lws[_unfed] / 1e3))
    print('measured against; the narrowed curves stand up inside it.')
    if np.isfinite(NOISE_RATIO):
        print('This is the DRIVEN equivalent of thesis Fig. 6.3 - the noise version')
        print('needs the coil noise above the converter floor, and here it is %.0fx'
              % (1.0 / max(NOISE_RATIO, 1e-9)))
        print('below it (Step 4c).')


---
# Step 8e — How far, and how accurately, the resonance can be moved

Thesis Fig. 6.2 shows the resonance pushed aside by feedback, decoupling the ion from the
detection system.

**How far?** Feedback in quadrature with the coil's motion shifts the centre rather than the
damping, so the sweep runs at `φ_shift` — a quarter turn from the narrowing phase — and at the
opposite phase for the other direction. Reported in **linewidths**, the only fair comparison
against a resonator of different Q.

**How accurately?** The shift-versus-gain curve is fitted, inverted to request a set of target
shifts, and each one applied and re-measured. The residual is the accuracy with which a wanted
detuning can actually be set.


In [ ]:
# =============================================================================
#  STEP 8e - FREQUENCY SHIFT: RANGE AND ACCURACY
# =============================================================================
print('=' * 68)
print('STEP 8e - how far and how accurately the resonance moves')
print('=' * 68)
FB_8E_OK = bool(FB_8B_OK)
if not FB_8E_OK:
    print('SKIPPED - no usable calibrated feedback phase is available.')

if FB_8E_OK:
    acq.start(DEC_USE)
    ps.config_delay(F0)
    PHI_SHIFT_OPP = (PHI_SHIFT + 180.0) % 360.0
    sh_gain, sh_df = [], []

    def _shift_at(A, phi):
        """Centre shift with the loop closed at (A, phi). Returns Hz, or None."""
        ps.set(A, phi)
        time.sleep(SETTLE)
        gen.amp(0.0); time.sleep(0.005)
        if sc.peak_frac(GUARD_NSAMP) > FB_OSC_LEVEL:
            ps.off(); return None
        fq, pq, ft, zq = _lineshape(F0, LW, return_complex=True)
        if not (0.02 * LW < ft['lw'] < 50.0 * LW):
            return None
        return ft['f0'] - F0

    try:
        for phi, sgn in ((PHI_SHIFT, +1), (PHI_SHIFT_OPP, -1)):
            for fr in FB_SHIFT_FRACS:
                A = fr * FB_GAIN_CEIL
                try:
                    d = _shift_at(A, phi)
                except Exception:
                    d = None
                if d is None:
                    continue
                sh_gain.append(sgn * A); sh_df.append(d)
                print('  A = %5.2f at %5.1f deg : shift %+9.1f Hz (%+.3f linewidths)'
                      % (A, phi, d, d / LW))
    finally:
        ps.off(); gen.off()

    if len(sh_df) < 4:
        raise RuntimeError('too few usable shift points')
    sh_gain = np.array(sh_gain); sh_df = np.array(sh_df)
    SHIFT_MAX = float(np.max(np.abs(sh_df)))
    _ord = np.argsort(sh_gain)
    _slope = float(np.polyfit(sh_gain, sh_df, 1)[0])
    print('')
    print('  RANGE   : %+.1f to %+.1f Hz  =  %+.2f to %+.2f linewidths'
          % (sh_df.min(), sh_df.max(), sh_df.min() / LW, sh_df.max() / LW))
    print('            %.3f%% of the centre frequency' % (100 * SHIFT_MAX / F0))
    print('  slope   : %.0f Hz per unit gain' % _slope)

    # ---- accuracy: request a shift, apply it, measure what you got --------------
    print('')
    print('  ACCURACY - requesting a shift and measuring the result')
    _targets = np.linspace(0.35, 0.85, FB_SHIFT_TESTS) * SHIFT_MAX
    _err = []
    try:
        for tgt in _targets:
            A_req = float(np.clip(abs(tgt / _slope), 0.0, FB_GAIN_CEIL))
            got = _shift_at(A_req, PHI_SHIFT if _slope > 0 else PHI_SHIFT_OPP)
            if got is None:
                print('    target %+8.1f Hz : abandoned' % tgt); continue
            _err.append(got - tgt)
            print('    target %+8.1f Hz -> A = %.2f -> got %+8.1f Hz  (error %+7.1f Hz'
                  ', %+.3f lw)' % (tgt, A_req, got, got - tgt, (got - tgt) / LW))
    finally:
        ps.off(); gen.off()

    if _err:
        SHIFT_RMS_ERR = float(np.sqrt(np.mean(np.square(_err))))
        print('')
        print('  rms error : %.1f Hz = %.4f linewidths = %.1f ppm of f0'
              % (SHIFT_RMS_ERR, SHIFT_RMS_ERR / LW, 1e6 * SHIFT_RMS_ERR / F0))
        _drift = globals().get('DRIFT_HZ_PER_MIN', float('nan'))
        if np.isfinite(_drift):
            print('  Step 7 measured the coil drifting %.1f Hz/min unaided, so a shift'
                  % abs(_drift))
            print('  can be set %s accurately as the coil holds still.'
                  % ('more' if SHIFT_RMS_ERR < abs(_drift) else 'less'))
    else:
        SHIFT_RMS_ERR = float('nan')


---
# Step 8f — Verify the two useful operating points

This is the recommendation step. It does **not** trust the largest colour-map pixel or an
extrapolation. It repeats a small neighbourhood around the best measured Q setting and around
the maximum-shift setting, using only the Red Pitaya.

Each feedback measurement is enclosed by two feedback-off references. The centre frequency,
linewidth and Q of the unmodified coil are interpolated to the time of the feedback measurement,
which removes slow linear drift. A point is rejected if the coefficient readback fails, the loop
oscillates with the drive off, the input clips, or too few frequency samples describe the fitted
linewidth.

**Run this cell once.** It can take several minutes. A STOP result is deliberately not replaced by
a mapped or extrapolated value. Drive and feedback are disabled on every exit path.


In [ ]:
# =============================================================================
#  STEP 8f - BRACKETED, HIGH-DENSITY VERIFICATION AND RECOMMENDATIONS
# =============================================================================
print('=' * 68)
print('STEP 8f - verifying maximum Q and maximum shift')
print('=' * 68)

def _g(name, default=None):
    """Fetch a result that an earlier step may have skipped."""
    value = globals().get(name, default)
    return default if value is None else value

VERIFY_Q_PHASE_OFFSETS = (-2.0, 0.0, 2.0)       # small safe neighbourhood [deg]
VERIFY_SHIFT_PHASE_OFFSETS = (-5.0, 0.0, 5.0)   # shift is less phase-sensitive
VERIFY_NPTS = 161
VERIFY_MIN_IN_LINEWIDTH = HIGH_Q_POINTS_PER_LW
VERIFY_REFERENCE_MAX_CHANGE = 0.10              # fraction of baseline linewidth
FIT_MAX_RESIDUAL_RMS = 0.020
FIT_MAX_STRUCTURED_RESIDUAL = 0.010
FIT_MAX_FLOOR_PEAK = 0.05
FIT_MAX_WIDTH_DISAGREEMENT = 0.05
FIT_MAX_REFERENCE_K_MOVE = 0.01

class HardSafetyStop(RuntimeError):
    pass

HARD_STOP_REQUESTED = False
HARD_STOP_RECORD = None
HARDWARE_COMM_FAILURES = 0
ACQUISITION_SEQUENCE = 0
if not TERMINATED_OUTPUT:
    HARD_STOP_REQUESTED = True
    HARD_STOP_RECORD = {'failure_class':'hard_safety',
                        'stop_reason':'termination protection is not confirmed'}
if np.isfinite(MEASURED_SECOND_INPUT_DBM) and MEASURED_SECOND_INPUT_DBM > SECOND_IN_MAX_DBM:
    HARD_STOP_REQUESTED = True
    HARD_STOP_RECORD = {'failure_class':'hard_safety',
                        'stop_reason':'measured second-amplifier input exceeds hard limit'}
if np.isfinite(MEASURED_FINAL_OUTPUT_DBM) and MEASURED_FINAL_OUTPUT_DBM >= FINAL_P1DB_DBM:
    HARD_STOP_REQUESTED = True
    HARD_STOP_RECORD = {'failure_class':'hard_safety',
                        'stop_reason':'measured final output reaches P1dB'}
if np.isfinite(MEASURED_BROADBAND_RISE_DB) and MEASURED_BROADBAND_RISE_DB > 3.0:
    HARD_STOP_REQUESTED = True
    HARD_STOP_RECORD = {'failure_class':'hard_safety',
                        'stop_reason':'unexpected broadband instability exceeds 3 dB'}

def _failure_class(exc):
    text = str(exc).lower()
    hard_tokens = ('clipped', 'clipping', 'drive-off oscillation', 'coefficient readback',
                   'register overflow', 'absolute input limit', 'p1db',
                   'termination protection', 'broadband instability')
    return 'hard_safety' if isinstance(exc, HardSafetyStop) or any(x in text for x in hard_tokens) else 'soft_measurement'
COHERENT_PEAK_TARGET = float(np.sqrt(max(FIT['peak'], 0.0)))
COHERENT_PEAK_HIGH = 1.15 * COHERENT_PEAK_TARGET


def _fit_quality(diag, drive_frac, label):
    """Fail closed before a trace can contribute to K."""
    reasons = []
    drive_lo = (min(float(DRIVE_WINDOW[0]), 0.0017) if QUALIFICATION_ONLY
                and np.all(np.isfinite(DRIVE_WINDOW)) else
                (float(DRIVE_WINDOW[0]) if np.all(np.isfinite(DRIVE_WINDOW)) else 0.0))
    if drive_frac < drive_lo:
        reasons.append('drive %.6f below validated fit window %.6f'
                       % (drive_frac, drive_lo))
    if diag['residual_rms'] > FIT_MAX_RESIDUAL_RMS:
        reasons.append('RMS residual %.4f > %.4f'
                       % (diag['residual_rms'], FIT_MAX_RESIDUAL_RMS))
    if diag['structured_residual'] > FIT_MAX_STRUCTURED_RESIDUAL:
        reasons.append('smoothed residual %.4f > %.4f'
                       % (diag['structured_residual'], FIT_MAX_STRUCTURED_RESIDUAL))
    if diag['floor_peak_amp'] > FIT_MAX_FLOOR_PEAK:
        reasons.append('floor/peak %.3f > %.3f'
                       % (diag['floor_peak_amp'], FIT_MAX_FLOOR_PEAK))
    if diag['half_power_disagreement'] > FIT_MAX_WIDTH_DISAGREEMENT:
        reasons.append('half-power width differs by %.1f%%'
                       % (100*diag['half_power_disagreement']))
    # The complex model remains diagnostic-only until it reproduces saved
    # measured references. A systematic model mismatch must not erase a valid
    # power/half-power/repeat result.
    if reasons:
        raise RuntimeError('%s fit-quality gate: %s' % (label, '; '.join(reasons)))
    return True


def _reference_trace(dec_log2=None, drive_frac=None):
    """Feedback-off reference using the established two-pass adaptive sweep."""
    global DRIVE_FRAC
    old_drive = DRIVE_FRAC; use_drive = old_drive if drive_frac is None else float(drive_frac)
    try:
        DRIVE_FRAC = use_drive; ps.off()
        f, p, fit, z = _lineshape(F0, LW, dec_log2=dec_log2, return_complex=True)
        diag = trace_fit_diagnostics(f, p, fit, z=z)
        quality_error = ''
        try:
            _fit_quality(diag, use_drive, 'reference')
        except Exception as exc:
            quality_error = str(exc)
        _rdec = int(acq.dec); _rsettle = adaptive_settle(fit['lw'], _rdec)
        rxmeta = receiver_metadata(_rdec, fit['lw'], _rsettle, f=f, fit=fit)
        if rxmeta['fir_bypassed'] or rxmeta['fir_overrun']:
            quality_error = '; '.join(x for x in (quality_error, 'reference receiver status invalid') if x)
        return {'time': time.time(), 'drive_frac': use_drive, 'f': f, 'p': p, 'z': z,
                'fit': fit, 'fit_diagnostics': diag, 'receiver': rxmeta,
                'quality_error': quality_error}
    finally:
        DRIVE_FRAC = old_drive


def _dense_feedback_trace(A, phi, f_hint, lw_hint, dec_log2=None, retry_mode=False):
    """One guarded high-density feedback trace; caller owns final shutdown."""
    ps.config_delay(float(f_hint)) # follow measured centre shifts and cryogenic retuning
    if OUT2_MONITOR_INPUT:
        ps.monitor('input', shift=0)  # also clears the monitor's sticky clip flag
    ci, cq = ps.set(A, phi)
    readback = ps.readback()
    if readback['ci'] != ci or readback['cq'] != cq:
        raise HardSafetyStop('feedback coefficient readback changed')
    time.sleep(SETTLE)
    gen.amp(0.0)
    time.sleep(0.005)
    if sc.peak_frac(GUARD_NSAMP) > FB_OSC_LEVEL:
        raise HardSafetyStop('drive-off oscillation detected outside armed onset test')

    # First locate the line, then put a dense grid around the measured result.
    _, _, located = _lineshape(f_hint, lw_hint, dec_log2=dec_log2)
    use_dec = receiver_policy(located['lw'], weak=(DRIVE_FRAC < 0.002),
                              high_q=(located['lw'] < 0.6*LW), requested=dec_log2)
    acq.start(use_dec)
    grid = adaptive_frequency_grid(located['f0'], located['lw'],
                                   points_per_lw=(2*HIGH_Q_POINTS_PER_LW if retry_mode
                                                      else HIGH_Q_POINTS_PER_LW),
                                   wing_lw=(6.0 if retry_mode else HIGH_Q_WING_LW))
    use_settle = adaptive_settle(located['lw'], use_dec, 2.0 if retry_mode else 1.0)
    f, z = sweep(grid, drive_frac=DRIVE_FRAC, navg=FB_NAVG,
                 settle=use_settle, level_guard=FB_LEVEL_GUARD)
    p = np.abs(z) ** 2
    fit = fit_lorentzian(f, p)
    in_line = int(np.sum(np.abs(f - fit['f0']) <= 0.5 * fit['lw']))
    geometry_errors = []
    if in_line < VERIFY_MIN_IN_LINEWIDTH:
        geometry_errors.append('under-resolved fit: only %d points in the linewidth' % in_line)
    if fit['f0'] - fit['lw'] <= f.min() or fit['f0'] + fit['lw'] >= f.max():
        geometry_errors.append('fit is not bracketed by the measured frequency span')
    gen.freq(fit['f0']); gen.amp(DRIVE_FRAC); gen.run(); time.sleep(0.02)
    adc_peak = sc.peak_frac(GUARD_NSAMP)
    gen.amp(0.0)
    if adc_peak > FB_LEVEL_GUARD:
        raise HardSafetyStop('ADC clipping risk: input at %.2f of full scale' % adc_peak)
    if OUT2_MONITOR_INPUT and ps.monitor_clipped():
        raise HardSafetyStop('OUT2 input monitor clipped')
    diag = trace_fit_diagnostics(f, p, fit, z=z)
    if fit['lw'] < 0.60*LW and diag['complex_lw_disagreement'] > HIGH_Q_COMPLEX_WIDTH_TOL:
        geometry_errors.append('high-Q complex/power width disagreement %.1f%%' %
                               (100*diag['complex_lw_disagreement']))
    rxmeta = receiver_metadata(use_dec, fit['lw'], use_settle, f=f, fit=fit)
    if rxmeta['fir_bypassed'] or rxmeta['fir_overrun'] or rxmeta['dma_overflow'] or rxmeta['write_error']:
        geometry_errors.append('receiver status invalid: 0x%X' % rxmeta['status2'])
    quality_error = '; '.join(geometry_errors)
    try:
        _fit_quality(diag, DRIVE_FRAC, 'feedback')
    except Exception as exc:
        quality_error = '; '.join([x for x in (quality_error, str(exc)) if x])
    return {'time': time.time(), 'f': f, 'p': p, 'z': z, 'fit': fit,
            'fit_diagnostics': diag,
            'in_linewidth': in_line, 'adc_peak_frac': adc_peak,
            'coefficients': (ci, cq), 'receiver': rxmeta,
            'settling_validation': ('double' if retry_mode else 'nominal'),
            'quality_error': quality_error}


def _verify_point_once(kind, A, phi, f_hint, lw_hint, dec_log2=None,
                       retry_mode=False, sweep_direction='up'):
    """Reference -> feedback -> reference, with drift interpolation and shutdown."""
    global HARD_STOP_REQUESTED, HARD_STOP_RECORD, HARDWARE_COMM_FAILURES, ACQUISITION_SEQUENCE
    ACQUISITION_SEQUENCE += 1
    record = {'kind': kind, 'gain': float(A), 'phase_deg': float(phi % 360.0),
              'drive_frac': float(DRIVE_FRAC), 'navg': int(FB_NAVG),
              'dec_log2': int(acq.dec), 'valid': False, 'stop_reason': '',
              'attempted_at': time.time(), 'sequence': int(ACQUISITION_SEQUENCE),
              'sweep_direction': sweep_direction, 'retry_mode': bool(retry_mode),
              'failure_class': '', 'rejection_reasons': []}
    try:
        before = _reference_trace(dec_log2=dec_log2, drive_frac=REFERENCE_DRIVE_FRAC)
        record['before'] = before
        if before.get('quality_error'):
            raise RuntimeError(before['quality_error'])
        measured = _dense_feedback_trace(A, phi, f_hint, lw_hint, dec_log2=dec_log2,
                                         retry_mode=retry_mode)
        record['measured'] = measured
        if measured.get('quality_error'):
            raise RuntimeError(measured['quality_error'])
        ps.off()
        after = _reference_trace(dec_log2=dec_log2, drive_frac=REFERENCE_DRIVE_FRAC)
        record['after'] = after
        if after.get('quality_error'):
            raise RuntimeError(after['quality_error'])

        dt = after['time'] - before['time']
        alpha = 0.5 if dt <= 0 else np.clip(
            (measured['time'] - before['time']) / dt, 0.0, 1.0)
        base = {}
        for key in ('f0', 'lw', 'Q'):
            base[key] = ((1.0 - alpha) * before['fit'][key]
                         + alpha * after['fit'][key])
        base_complex_lw = ((1.0-alpha)*before['fit_diagnostics']['complex_fit']['lw'] +
                           alpha*after['fit_diagnostics']['complex_fit']['lw'])
        reference_move = abs(after['fit']['f0'] - before['fit']['f0'])
        reference_lw_move_k = abs(after['fit']['lw']-before['fit']['lw']) / base['lw']
        if reference_move > VERIFY_REFERENCE_MAX_CHANGE * base['lw']:
            raise RuntimeError('feedback-off references moved by %.3f linewidths'
                               % (reference_move / base['lw']))
        if reference_lw_move_k > FIT_MAX_REFERENCE_K_MOVE:
            raise RuntimeError('feedback-off linewidth moved by %.4f K-equivalent'
                               % reference_lw_move_k)

        record.update({
            'valid': True, 'before': before, 'measured': measured, 'after': after,
            'baseline': base, 'q_ratio': measured['fit']['Q'] / base['Q'],
            'shift_hz': measured['fit']['f0'] - base['f0'],
            'shift_over_b0': (measured['fit']['f0'] - base['f0']) / base['lw'],
            'reference_move_hz': reference_move,
            'reference_lw_move_k': reference_lw_move_k,
            'baseline_complex_lw': float(base_complex_lw),
            'k_complex_damping': float(
                1.0-measured['fit_diagnostics']['complex_fit']['lw']/base_complex_lw),
        })
    except BaseException as exc:
        failure_class = _failure_class(exc)
        _txt = str(exc).lower()
        if any(x in _txt for x in ('communication', 'timeout', 'read failed', 'write failed')):
            HARDWARE_COMM_FAILURES += 1
            if HARDWARE_COMM_FAILURES >= HARD_COMM_FAILURE_LIMIT:
                failure_class = 'hard_safety'
        record['failure_class'] = failure_class
        record['stop_reason'] = '%s: %s' % (type(exc).__name__, exc)
        record['rejection_reasons'].append(record['stop_reason'])
        if failure_class == 'hard_safety':
            HARD_STOP_REQUESTED = True
            HARD_STOP_RECORD = dict(record)
        if isinstance(exc, (KeyboardInterrupt, SystemExit)):
            raise
    finally:
        gen.off()
        ps.off()
    return record


def _verify_point(kind, A, phi, f_hint, lw_hint, dec_log2=None, sweep_direction='up'):
    """Retry one soft failure with a wider, denser sweep; never hide either attempt."""
    attempts = []
    for retry_mode in (False, True):
        rec = _verify_point_once(kind, A, phi, f_hint, lw_hint, dec_log2,
                                 retry_mode=retry_mode, sweep_direction=sweep_direction)
        attempts.append(rec)
        if rec.get('failure_class') == 'hard_safety':
            break
        if rec.get('valid'):
            # High-Q points must also survive a doubled-settling acquisition.
            if retry_mode or rec['measured']['fit']['lw'] >= 0.40*LW:
                break
    result = dict(attempts[-1])
    if len(attempts) == 2 and all(x.get('valid') for x in attempts):
        a, b = attempts
        dlw = abs(b['measured']['fit']['lw']/a['measured']['fit']['lw']-1.0)
        df0 = abs(b['measured']['fit']['f0']-a['measured']['fit']['f0'])/max(a['measured']['fit']['lw'],1.0)
        result['settling_relative_width_change'] = float(dlw)
        result['settling_centre_change_linewidths'] = float(df0)
        result['settling_converged'] = bool(max(dlw,df0) <= SETTLING_CONVERGENCE_TOL)
        if not result['settling_converged']:
            result['valid'] = False; result['accepted'] = False
            result['failure_class'] = 'soft_measurement'
            result['stop_reason'] = 'settling-time convergence failed'
            result.setdefault('rejection_reasons',[]).append(result['stop_reason'])
    else:
        result['settling_converged'] = bool(result.get('valid'))
    result['attempts'] = attempts
    result['retry_count'] = len(attempts)-1
    result['accepted'] = bool(result.get('valid'))
    return result


VERIFY_RECORDS = []
Q_RECOMMENDATION = SHIFT_RECOMMENDATION = None
FIRST_ORDER_STOP_REASON = ''

if QUALIFICATION_ONLY:
    FIRST_ORDER_STOP_REASON = 'Legacy maximum-Q/shift verification skipped by qualification mode.'
    print('SKIPPED - extremes are tested efficiently in Step 8g.')
elif not FB_8D_OK or not globals().get('spec_fit'):
    FIRST_ORDER_STOP_REASON = 'No valid narrowing lineshape is available from Step 8d.'
    print('STOP - ' + FIRST_ORDER_STOP_REASON)
elif not globals().get('sh_df') is not None or len(sh_df) < 2:
    FIRST_ORDER_STOP_REASON = 'No valid shift sweep is available from Step 8e.'
    print('STOP - ' + FIRST_ORDER_STOP_REASON)
else:
    # Use the best actually measured positive-feedback lineshape, never MAP_Q.
    q_candidates = [(ft['Q'], A, ft) for fr, A, ft in zip(spec_lbl, spec_A, spec_fit)
                    if fr > 0 and np.isfinite(ft['Q'])]
    if not q_candidates:
        FIRST_ORDER_STOP_REASON = 'Step 8d produced no valid positive-feedback candidate.'
        print('STOP - ' + FIRST_ORDER_STOP_REASON)
    else:
        _, q_gain, q_seed = max(q_candidates, key=lambda row: row[0])
        shift_i = int(np.nanargmax(np.abs(sh_df)))
        shift_gain_signed = float(sh_gain[shift_i])
        shift_gain = abs(shift_gain_signed)
        shift_phase = PHI_SHIFT if shift_gain_signed >= 0 else PHI_SHIFT_OPP
        shift_hint = F0 + float(sh_df[shift_i])

        acq.start(DEC_USE)
        ps.config_delay(F0)
        try:
            print('  Q neighbourhood: gain %.3f around phase %.1f deg'
                  % (q_gain, PHI_NARROW))
            for offset in VERIFY_Q_PHASE_OFFSETS:
                r = _verify_point('q', q_gain, PHI_NARROW + offset,
                                  q_seed['f0'], q_seed['lw'])
                VERIFY_RECORDS.append(r)
                print('    phase %6.1f : %s' % (r['phase_deg'],
                      ('Q %.1f (x%.3f)' % (r['measured']['fit']['Q'], r['q_ratio']))
                       if r['valid'] else 'REJECTED - ' + r['stop_reason']))

            print('  shift neighbourhood: gain %.3f around phase %.1f deg'
                  % (shift_gain, shift_phase))
            for offset in VERIFY_SHIFT_PHASE_OFFSETS:
                r = _verify_point('shift', shift_gain, shift_phase + offset,
                                  shift_hint, LW)
                VERIFY_RECORDS.append(r)
                print('    phase %6.1f : %s' % (r['phase_deg'],
                      ('shift %+.1f Hz (%+.3f linewidths)'
                       % (r['shift_hz'], r['shift_over_b0']))
                       if r['valid'] else 'REJECTED - ' + r['stop_reason']))
        finally:
            gen.off()
            ps.off()

        q_ok = [r for r in VERIFY_RECORDS if r['valid'] and r['kind'] == 'q']
        s_ok = [r for r in VERIFY_RECORDS if r['valid'] and r['kind'] == 'shift']
        if q_ok:
            Q_RECOMMENDATION = max(q_ok, key=lambda r: r['q_ratio'])
            _q_values = np.array([r['measured']['fit']['Q'] for r in q_ok])
            Q_RECOMMENDATION['uncertainty'] = float(
                max(np.std(_q_values, ddof=1) if len(_q_values) > 1 else 0.0,
                    abs(Q_RECOMMENDATION['before']['fit']['Q']
                        - Q_RECOMMENDATION['after']['fit']['Q'])))
            Q_RECOMMENDATION['safe_phase_neighbourhood_deg'] = [
                float(min(r['phase_deg'] for r in q_ok)),
                float(max(r['phase_deg'] for r in q_ok))]
            Q_RECOMMENDATION['limiting_factor'] = (
                'awaiting the Step 8g small-signal drive plateau and receiver convergence')
        if s_ok:
            SHIFT_RECOMMENDATION = max(s_ok, key=lambda r: abs(r['shift_hz']))
            _s_values = np.array([r['shift_hz'] for r in s_ok])
            SHIFT_RECOMMENDATION['uncertainty'] = float(
                max(np.std(_s_values, ddof=1) if len(_s_values) > 1 else 0.0,
                    _g('SHIFT_RMS_ERR', 0.0)))
            SHIFT_RECOMMENDATION['safe_phase_neighbourhood_deg'] = [
                float(min(r['phase_deg'] for r in s_ok)),
                float(max(r['phase_deg'] for r in s_ok))]
            _sdamp = 1.0 - (SHIFT_RECOMMENDATION['measured']['fit']['lw'] /
                            SHIFT_RECOMMENDATION['baseline']['lw'])
            _squad = 2.0 * SHIFT_RECOMMENDATION['shift_over_b0']
            SHIFT_RECOMMENDATION['k_damping'] = float(_sdamp)
            SHIFT_RECOMMENDATION['k_quadrature'] = float(_squad)
            SHIFT_RECOMMENDATION['k_vector'] = float(np.hypot(_sdamp, _squad))
            SHIFT_RECOMMENDATION['limiting_factor'] = (
                'awaiting drive/receiver convergence of the full vector K')

        if Q_RECOMMENDATION is None or SHIFT_RECOMMENDATION is None:
            FIRST_ORDER_STOP_REASON = 'One or both verification neighbourhoods had no valid point.'
            print('STOP - ' + FIRST_ORDER_STOP_REASON)
        else:
            print('')
            print('PASS - both recommendations were measured, bracketed and verified.')


def _recommendation_text():
    if Q_RECOMMENDATION is None or SHIFT_RECOMMENDATION is None:
        return ('FIRST-ORDER OPTIMISATION: STOP\n\nReason: %s\n\n'
                'No mapped or extrapolated value is a safe setting.'
                % (FIRST_ORDER_STOP_REASON or 'verification incomplete'))
    qr, sr = Q_RECOMMENDATION, SHIFT_RECOMMENDATION
    map_q = float(np.nanmax(MAP_Q)) if 'MAP_Q' in globals() else float('nan')
    onset = _g('A_ONSET', float('nan'))
    ceiling = 0.5 * sr['baseline']['lw']
    q_final = bool(globals().get('Q_CONVERGED_RECOMMENDATION_OK', False))
    s_final = bool(globals().get('SHIFT_CONVERGENCE_OK', False))
    final_ok = bool(q_final and s_final)
    heading = ('FIRST-ORDER OPTIMISATION: VERIFIED CONVERGED RECOMMENDATIONS'
               if final_ok else
               'FIRST-ORDER OPTIMISATION: PRELIMINARY REPEATED RESULTS')
    return """{heading}

MAXIMUM Q ({qclass})
  Q                    : {q:.1f} +/- {qu:.1f}
  Q / Q0               : {qr:.3f}
  linewidth            : {lw:.2f} kHz
  feedback gain, phase : {qg:.4f}, {qp:.1f} deg
  measured neighbourhood: {qn0:.1f} to {qn1:.1f} deg
  first limiting factor: {qlim}

MAXIMUM RESONANCE SHIFT ({sclass})
  shift                : {sf:+.1f} +/- {su:.1f} Hz
  shift / B0           : {sn:+.3f}
  quadrature loop term  : {kquad:+.3f} (= 2 shift/B0)
  full vector K         : {kvec:.3f}
  B0/2 reference        : {ceil:.1f} Hz (only a ceiling if vector K < 1)
  feedback gain, phase : {sg:.4f}, {sp:.1f} deg
  measured neighbourhood: {sn0:.1f} to {sn1:.1f} deg
  first limiting factor: {slim}

CONTEXT - NOT SAFE RECOMMENDATIONS
  largest map pixel    : Q = {mq:.1f} (measured once / mapped)
  fitted K slope       : {ks:.4f} per unit gain (fitted)
  K=1 onset gain       : {on:.3f} (extrapolated; not approached)

Maximum Q and maximum shift are different operating points. Phase values are
specific to this cable, coil and amplifier arrangement.""".format(
        heading=heading,
        qclass=('verified across drive and receiver settings' if q_final else
                'repeated at one measurement configuration'),
        sclass=('verified across drive and receiver settings' if s_final else
                'repeated at one measurement configuration'),
        q=qr['measured']['fit']['Q'], qu=qr['uncertainty'], qr=qr['q_ratio'],
        lw=qr['measured']['fit']['lw']/1e3, qg=qr['gain'], qp=qr['phase_deg'],
        qn0=qr['safe_phase_neighbourhood_deg'][0], qn1=qr['safe_phase_neighbourhood_deg'][1],
        qlim=qr['limiting_factor'], sf=sr['shift_hz'], su=sr['uncertainty'],
        sn=sr['shift_over_b0'], kquad=sr.get('k_quadrature', float('nan')),
        kvec=sr.get('k_vector', float('nan')), ceil=ceiling,
        sg=sr['gain'], sp=sr['phase_deg'],
        sn0=sr['safe_phase_neighbourhood_deg'][0], sn1=sr['safe_phase_neighbourhood_deg'][1],
        slim=sr['limiting_factor'], mq=map_q, ks=_g('k_slope', float('nan')), on=onset)


REPORT_TEXT = _recommendation_text()
print('')
print(REPORT_TEXT)


---
# Step 8g — Diagnose the remaining loss and safely bracket K = 1

`K = 1` is the self-oscillation boundary, not an operating point. This step first checks why the
high-gain Q result may disagree with the low-gain extrapolation. It measures several gains, finds
the local narrowing phase at each one, and compares three independent estimates of K.

**Six dB validation.** The earlier 15 dB result is the control; this notebook measures the new
chain independently. External checks are recommended during ordinary Step 8 mapping and required
before the probe-off SEO search. Use the oscilloscope at OUT1, the interstage node, final output,
and copied IN1-to-OUT2 monitor; use the spectrum analyser to record carrier, noise floor and
harmonics. With feedback open, confirm the 6 dB chain adds about 9 dB without compression,
keeps the second-stage input below its rating, and keeps the final output at least 3 dB below
P1dB. Enter the measured worst cases and confirmations once in CONFIGURATION; every feedback
cell uses those same values and no later cell resets them.

SEO onset means a persistent narrow line with the injected probe disabled, observed by both
the Red Pitaya and external spectrum analyser. A large fitted Q alone is not onset. Stop for
clipping, new harmonics, broadband excess, excessive reference motion, or estimator failure.

The final onset search is **locked by default**. It cannot run until the amplifier path has been
made safe at full Red Pitaya output, the protection is explicitly confirmed below, all K estimates
agree, and a stable measurement has already reached K >= 0.98. When unlocked, every short
drive-off capture ends by opening the loop; the first oscillating point is a boundary observation,
never a recommended setting.


In [ ]:
# =============================================================================
#  STEP 8g - HIGH-GAIN DIAGNOSIS AND SAFETY-GATED K=1 BRACKET
# =============================================================================
print('=' * 68)
print('STEP 8g - diagnosing the path to K = 1')
print('=' * 68)

# Safety values are defined once in CONFIGURATION and used by every feedback
# cell. They are intentionally not reset here.
SA_LEVELS_CONFIRMED = bool(np.isfinite(MEASURED_SECOND_INPUT_DBM) and
                           np.isfinite(MEASURED_FINAL_OUTPUT_DBM) and
                           MEASURED_SECOND_INPUT_DBM <= SECOND_IN_MAX_DBM and
                           MEASURED_FINAL_OUTPUT_DBM <= AMP_OUT_MAX_DBM and
                           7.0 <= MEASURED_GAIN_INCREASE_DB <= 11.0)
SA_HARMONICS_CONFIRMED = bool(np.isfinite(MEASURED_WORST_HARMONIC_DBC) and
                              np.isfinite(MEASURED_BROADBAND_RISE_DB) and
                              MEASURED_WORST_HARMONIC_DBC <= -40.0 and
                              MEASURED_BROADBAND_RISE_DB <= 3.0)
HIGH_GAIN_CHAIN_CONFIRMED = bool(SCOPE_NO_CLIPPING_CONFIRMED and
                                  SA_LEVELS_CONFIRMED and
                                  SA_HARMONICS_CONFIRMED)
ONSET_SEARCH_ARMED = False
OSC_SAFETY_MARGIN_DB = 3.0
SEO_MIN_DB_ABOVE_NOISE = 10.0
SEO_CONFIRM_CAPTURES = 3
SEO_EXTERNAL_CARRIER_CONFIRMED = False  # set only while analyser sees probe-off carrier
SEO_HARMONIC_LIMIT_DBC = -40.0
SEO_BROADBAND_RISE_LIMIT_DB = 3.0
OSC_DESIGN_MAX_A = 1.90
DIAG_TARGET_K = (0.10, 0.30, 0.50, 0.70, 0.75, 0.80, 0.85)
FULL_CIRCLE_TARGET_K = (0.20, 0.40, 0.60, 0.70)
HIGH_K_SELECTED_TARGET_K = (0.75, 0.80, 0.85)
RUN_WIDER_MAP_AFTER_QUALIFICATION = False # enable only after this short qualification passes
FULL_CIRCLE_PHASE_STEP_DEG = 15.0
CALIBRATION_TARGET_K = DIAG_TARGET_K
FULL_PHASE_TARGET_K = (0.20, 0.40, 0.60, 0.75)
_diag_slope = max(float(_g('k_slope', 0.0)), 1e-9)
DIAG_GAIN_TARGETS = tuple(sorted(set(
    round(min(FB_GAIN_MAX, k/_diag_slope), 4)
    for k in sorted(set(CALIBRATION_TARGET_K + DIAG_TARGET_K))
    if min(FB_GAIN_MAX, k/_diag_slope) > 0.0)))
DIAG_BASE_DRIVE = float(min(0.0031, DRIVE_LINEAR_MAX))
DIAG_BASE_NAVG = int(FB_NAVG)
DIAG_PHASE_OFFSETS_DEG = (-8.0, -4.0, 0.0, 4.0, 8.0)
_diag_drive_lo = min(float(DRIVE_WINDOW[0]), 0.0017)
DIAG_DRIVE_LEVELS = tuple(x for x in (0.0017, 0.0024, 0.0031)
                          if x <= DRIVE_LINEAR_MAX*(1+1e-9))
                          # deliberately below the marginal 0.0055 compression rung
DIAG_PLATEAU_POINTS = 3
DIAG_PLATEAU_ABS_K = 0.015
                          # at low gain, 3% of K is smaller than the observed
                          # fit scatter; require 3% OR this absolute K band
DIAG_PHASE_REFINE_MIN_K = 0.40
                          # below this, Q is too phase-insensitive to locate a
                          # maximum reliably; retain the full-circle fitted phase
DIAG_MIN_EXCITATION_COUNTS = 4
                          # below this, DAC amplitude quantisation—not the coil—
                          # limits a driven lineshape measurement
DIAG_GATE_MIN_K = 0.40
                          # lower-gain points characterise K0(A) but do not stop
                          # the campaign: their small linewidth change makes K
                          # disproportionately sensitive to fit bias
DIAG_FIXED_NAVG = 768
                          # all drives at one gain use the same averaging, so
                          # drive is not confused with sweep duration
DIAG_K_AGREEMENT_TOL = 0.05  # reported; no longer the sole low-K gate
DIAG_K_AGREEMENT_ABS = 0.015
DIAG_CONFIDENCE_Z = 1.96
DIAG_CONVERGENCE_TOL = 0.03
DIAG_REFERENCE_MOVE_MAX = 0.05
VERIFY_REFERENCE_MAX_CHANGE = DIAG_REFERENCE_MOVE_MAX
VERIFY_MIN_IN_LINEWIDTH = HIGH_Q_POINTS_PER_LW
DIAG_REPEAT_COUNT = 2     # routine points; high-K points escalate to three
DIAG_ESCALATE_K = 0.65
REFERENCE_DRIVE_FRAC = 0.0031 # stable feedback-off references, independent of probe
HARD_COMM_FAILURE_LIMIT = 3


def _amplitude_controlled_drive(trace, current_drive):
    """Reduce drive only after a measured coherent peak exceeds its target."""
    coherent = float(np.sqrt(max(trace['measured']['fit']['peak'], 0.0)))
    if coherent <= COHERENT_PEAK_HIGH:
        return float(current_drive)
    requested = current_drive*COHERENT_PEAK_TARGET/max(coherent, 1e-30)
    return float(np.clip(requested, _diag_drive_lo, DIAG_BASE_DRIVE))


def _broadening_probe_drive(narrow_drive):
    """Broadening is weaker, so use the largest independently linear probe."""
    ceiling = min(float(DRIVE_LINEAR_MAX), float(REFERENCE_DRIVE_FRAC))
    eligible = [x for x in DIAG_DRIVE_LEVELS if x <= ceiling*(1+1e-9)]
    return float(max([float(narrow_drive)] + eligible))


def _metrics(record, opposite=None):
    """Three first-order K estimates and phase from one bracketed record."""
    if not record.get('valid'):
        return None
    b0 = record['baseline']['lw']
    b = record['measured']['fit']['lw']
    df = record['shift_hz']
    k_q = 1.0 - 1.0 / record['q_ratio']
    damping = 1.0 - b / b0
    quadrature = 2.0 * df / b0
    out = {'k_q': float(k_q),
           'k_vector': float(np.hypot(damping, quadrature)),
           'k_complex': float(record.get('k_complex_damping', float('nan'))),
           'theta_deg': float(np.degrees(np.arctan2(quadrature, damping)))}
    if opposite is not None and opposite.get('valid'):
        rp, rm = record['q_ratio'], opposite['q_ratio']
        out['k_paired'] = float((rp - rm) / (rp + rm))
    else:
        out['k_paired'] = float('nan')
    finite = np.array([out['k_q'], out['k_vector'], out['k_paired']], dtype=float)
    finite = finite[np.isfinite(finite)]
    out['agreement_frac'] = (float(np.ptp(finite) / max(np.mean(np.abs(finite)), 1e-12))
                             if len(finite) > 1 else float('inf'))
    return out


def _joint_paired_metrics(plus, minus):
    """Opposite-phase solution with a pooled, drift-interpolated baseline."""
    if not plus.get('valid') or not minus.get('valid'):
        return None
    b0 = 0.5 * (plus['baseline']['lw'] + minus['baseline']['lw'])
    bp = plus['measured']['fit']['lw']; bm = minus['measured']['fit']['lw']
    bpc = plus['measured']['fit_diagnostics']['complex_fit']['lw']
    bmc = minus['measured']['fit_diagnostics']['complex_fit']['lw']
    damping = (bm - bp) / max(bm + bp, 1e-12)
    quadrature = (plus['shift_hz'] - minus['shift_hz']) / max(b0, 1e-12)
    return {'k_joint': float(np.hypot(damping, quadrature)),
            'k_complex_paired': float((bmc-bpc)/max(bmc+bpc, 1e-12)),
            'k_joint_damping': float(damping),
            'k_joint_quadrature': float(quadrature),
            'shared_baseline_lw': float(b0)}


def _variant_point(A, phi, f_hint, lw_hint, drive_frac, navg, dec_log2,
                   sweep_direction='up'):
    """Run the established bracketed measurement at a different drive/receiver."""
    global DRIVE_FRAC, FB_NAVG
    old_drive, old_navg = DRIVE_FRAC, FB_NAVG
    try:
        DRIVE_FRAC, FB_NAVG = float(drive_frac), int(navg)
        acq.start(int(dec_log2))
        return _verify_point('convergence', A, phi, f_hint, lw_hint,
                             dec_log2=int(dec_log2), sweep_direction=sweep_direction)
    finally:
        DRIVE_FRAC, FB_NAVG = old_drive, old_navg
        acq.start(DEC_USE)
        gen.off(); ps.off()


def _adaptive_phase(A, centre, f_hint, lw_hint):
    """Recenter a coarse phase grid, then refine it; never accept an edge."""
    all_records = []
    best = None
    for offsets in (DIAG_PHASE_OFFSETS_DEG, (-2.0, 0.0, 2.0),
                    (-0.5, 0.0, 0.5)):
        for attempt in range(3):
            records = []
            for off in offsets:
                records.append(_verify_point('diagnostic_phase', A, centre + off,
                                             f_hint, lw_hint))
                if HARD_STOP_REQUESTED:
                    return None, all_records + records, 'hard safety stop requested'
            all_records.extend(records)
            valid = [r for r in records if r.get('valid')]
            if not valid:
                if attempt < 2:
                    continue
                return None, all_records, 'no valid phase-locator points'
            best = max(valid, key=lambda r: r['q_ratio'])
            best_index = records.index(best)
            if len(valid) < 3:
                # Locator traces rank the phase; only the selected dense trace
                # must pass every high-Q fit gate.
                centre = best['phase_deg']; f_hint = best['measured']['fit']['f0']
                lw_hint = best['measured']['fit']['lw']; break
            if best_index not in (0, len(records) - 1):
                centre = best['phase_deg']; f_hint = best['measured']['fit']['f0']
                lw_hint = best['measured']['fit']['lw']; break
            centre = best['phase_deg']
        else:
            if best is not None:
                centre = best['phase_deg']; continue
            return None, all_records, 'phase locator produced no usable point'
    return best, all_records, ''


def _characterise_failed_gain(A, phase, direction='up'):
    """Collect broadening and quadrature evidence after a soft narrowing failure."""
    records = []
    for label, phi in (('broadening_after_failure', phase+180.0),
                       ('quadrature_after_failure', phase+90.0),
                       ('quadrature_opposite_after_failure', phase-90.0)):
        rec = _verify_point(label, A, phi, F0, LW, sweep_direction=direction)
        records.append(rec)
        if HARD_STOP_REQUESTED:
            break
    return records


DIAG_RECORDS = []
DIAG_SUMMARY = []
DIAG_AGREEMENT_OK = False
DIAG_CONVERGENCE_OK = False
Q_CONVERGED_RECOMMENDATION_OK = False
FIRST_LIMITING_FACTOR = ''
ONSET_RECORDS = []
ONSET_BRACKET = None
BOUNDARY_STOP_REASON = ''

_diag_candidates = [(ft['Q'], ft) for fr, ft in zip(spec_lbl, spec_fit)
                    if fr > 0 and np.isfinite(ft['Q'])]
_diag_seed = (Q_RECOMMENDATION['measured']['fit'] if Q_RECOMMENDATION is not None
              else (max(_diag_candidates, key=lambda row: row[0])[1]
                    if _diag_candidates else FIT))
if not globals().get('DRIFT_VALID', False):
    print('SOFT WARNING - five-minute drift record is incomplete; chronological references remain active')
if HARD_STOP_REQUESTED:
    BOUNDARY_STOP_REASON = HARD_STOP_RECORD.get('stop_reason','hard safety stop')
    print('HARD SAFETY STOP - ' + BOUNDARY_STOP_REASON)
elif _diag_seed is None:
    BOUNDARY_STOP_REASON = 'No valid positive-feedback lineshape is available.'
else:
    acq.start(DEC_USE)
    ps.config_delay(F0)
    try:
        gain_queue = [x for x in DIAG_GAIN_TARGETS if x <= FB_GAIN_MAX]
        _consecutive_gain_failures = 0
        while gain_queue:
            A = gain_queue.pop(0)
            predicted_k = np.clip(_g('k_slope', 0.0) * A, 0.0, 0.98)
            DRIVE_FRAC = DIAG_BASE_DRIVE
            FB_NAVG = max(DIAG_BASE_NAVG, DIAG_FIXED_NAVG)
            print('  gain %.4f (target K %.3f): local phase scan, drive %.5f, navg %d'
                  % (A, predicted_k, DRIVE_FRAC, FB_NAVG))
            if predicted_k < DIAG_PHASE_REFINE_MIN_K:
                phase_records = []
                phase_best = _verify_point('diagnostic_phase', A, PHI_NARROW,
                                           _diag_seed['f0'], _diag_seed['lw'])
                phase_records.append(phase_best)
                phase_stop = '' if phase_best.get('valid') else phase_best.get('stop_reason',
                                                                             'phase check failed')
                print('    using the full-circle narrowing phase %.1f deg: the effect is too small for a local Q maximum'
                      % PHI_NARROW)
            else:
                phase_best, phase_records, phase_stop = _adaptive_phase(
                    A, PHI_NARROW, _diag_seed['f0'], _diag_seed['lw'])
            DIAG_RECORDS.extend(phase_records)
            if HARD_STOP_REQUESTED:
                print('    HARD SAFETY STOP - active outputs disabled; continuing to reporting')
                break
            if phase_best is None:
                print('    REJECTED - ' + phase_stop)
                _consecutive_gain_failures += 1
                DIAG_RECORDS.extend(_characterise_failed_gain(A, PHI_NARROW, 'up'))
                DIAG_SUMMARY.append({'gain':float(A),'phase_deg':float(PHI_NARROW),
                    'attempted':True,'converged':False,'agreement_ok':False,
                    'stop_reason':phase_stop,'failure_class':'soft_measurement',
                    'drive_results':[],'k_vector':float('nan'),'k_paired':float('nan'),
                    'q':float('nan'),'q_ratio':float('nan')})
                if HARD_STOP_REQUESTED:
                    break
                continue
            _consecutive_gain_failures = 0
            _controlled_drive = _amplitude_controlled_drive(phase_best, DRIVE_FRAC)
            if _controlled_drive < DRIVE_FRAC:
                print('    coherent peak above target: reducing drive %.6f -> %.6f'
                      % (DRIVE_FRAC, _controlled_drive))
                DRIVE_FRAC = _controlled_drive
                phase_best = _verify_point('diagnostic_phase_controlled', A,
                                           phase_best['phase_deg'],
                                           phase_best['measured']['fit']['f0'],
                                           phase_best['measured']['fit']['lw'])
                DIAG_RECORDS.append(phase_best)
                if not phase_best.get('valid'):
                    print('    SOFT FAILURE - amplitude-controlled trace failed; collecting other phases')
                    DIAG_RECORDS.extend(_characterise_failed_gain(A, phase_best['phase_deg'], 'up'))
                    if HARD_STOP_REQUESTED:
                        break
                    continue
            # Remeasure the selected phase with a receiver chosen for its
            # measured closed-loop linewidth before calculating K.
            d_primary = receiver_policy(phase_best['measured']['fit']['lw'],
                                        high_q=(predicted_k >= HIGH_K_PAIR_OPTIONAL))
            d_adjacent = max(8, d_primary - 1)
            plus = _verify_point('diagnostic', A, phase_best['phase_deg'],
                                 phase_best['measured']['fit']['f0'],
                                 phase_best['measured']['fit']['lw'], d_primary)
            DIAG_RECORDS.append(plus)
            if not plus.get('valid'):
                print('    REJECTED - adaptive receiver measurement failed')
                if plus.get('failure_class') == 'hard_safety':
                    break
                DIAG_RECORDS.extend(_characterise_failed_gain(A, phase_best['phase_deg'], 'up'))
                continue
            broad_drive = _broadening_probe_drive(DRIVE_FRAC)
            minus = _variant_point(A, plus['phase_deg'] + 180.0, F0, LW,
                                   broad_drive, FB_NAVG, d_primary)
            DIAG_RECORDS.append(minus)
            quadrature = _variant_point(A, plus['phase_deg'] + 90.0, F0, LW,
                                        max(DRIVE_FRAC, _diag_drive_lo), FB_NAVG, d_primary)
            DIAG_RECORDS.append(quadrature)
            m = _metrics(plus, minus)
            qmetric = _metrics(quadrature)
            high_k_vector = (float(np.hypot(m['k_q'],
                2.0*quadrature['shift_hz']/quadrature['baseline']['lw']))
                if qmetric is not None else float('nan'))
            row = {'gain': float(A), 'phase_deg': plus['phase_deg'],
                   'q': plus['measured']['fit']['Q'], 'q_ratio': plus['q_ratio'],
                   'k_q': m['k_q'], 'k_paired': m['k_paired'],
                   'k_vector': (high_k_vector if predicted_k >= HIGH_K_PAIR_OPTIONAL
                                and np.isfinite(high_k_vector) else m['k_vector']),
                   'k_vector_narrow_quadrature': high_k_vector,
                   'quadrature_available': bool(qmetric is not None),
                   'broadening_available': bool(minus.get('valid')),
                   'broadening_drive_frac': float(broad_drive),
                   'theta_deg': m['theta_deg'],
                   'agreement_frac': m['agreement_frac'],
                   'record': plus,
                   'phase_q_values': [r['measured']['fit']['Q'] for r in phase_records
                                      if r.get('valid')],
                   'primary_dec_log2': int(d_primary),
                   'adjacent_dec_log2': int(d_adjacent),
                   'converged': None, 'stop_reason': ''}

            if True:  # every high-gain target must pass drive/receiver convergence
                variants = []
                variant_records = {}
                variant_failures = []
                _repeat_count = 3 if predicted_k >= DIAG_ESCALATE_K else DIAG_REPEAT_COUNT
                _dec_list = ((d_primary, d_adjacent) if predicted_k >= DIAG_ESCALATE_K
                             else (d_primary,))
                _diag_rng = np.random.RandomState(int(round(A*1e6)) & 0x7fffffff)
                for repeat in range(_repeat_count):
                    # Deterministic randomisation makes drive distinguishable from
                    # thermal/acquisition order while keeping a run reproducible.
                    _levels = [x for x in DIAG_DRIVE_LEVELS if x <= DRIVE_FRAC*(1+1e-9)]
                    if not _levels or abs(_levels[-1]-DRIVE_FRAC) > 1e-9:
                        _levels.append(DRIVE_FRAC)
                    _drive_order = tuple(_diag_rng.permutation(_levels))
                    for drive_variant in _drive_order:
                        drive_scale = drive_variant / max(DRIVE_FRAC, 1e-30)
                        drive_counts = int(round(drive_variant * 8192))
                        if drive_counts < DIAG_MIN_EXCITATION_COUNTS:
                            if repeat == 0:
                                print('      drive %.0f%% skipped: %d DAC counts is below the %d-count floor'
                                      % (100*drive_scale, drive_counts,
                                         DIAG_MIN_EXCITATION_COUNTS))
                            continue
                        for dec in _dec_list:
                            navg = int(max(FB_NAVG, DIAG_FIXED_NAVG))
                            vr = _variant_point(A, plus['phase_deg'],
                                                plus['measured']['fit']['f0'],
                                                plus['measured']['fit']['lw'],
                                                drive_variant, navg, dec)
                            DIAG_RECORDS.append(vr)
                            variant_records[(repeat, float(drive_scale), int(dec))] = vr
                            if vr.get('failure_class') == 'hard_safety':
                                break
                            if dec == d_primary and vr.get('valid'):
                                _vminus = _variant_point(
                                    A, vr['phase_deg'] + 180.0, F0, LW,
                                    _broadening_probe_drive(drive_variant),
                                    navg, d_primary)
                                DIAG_RECORDS.append(_vminus)
                                variant_records[(repeat, float(drive_scale), 'minus')] = _vminus
                            vm = _metrics(vr)
                            if vm is not None:
                                window_frac = 0.0 # tracked zero-IF; passband is not a linewidth convolution
                                _mf = vr['measured']['fit']
                                _amp = float(np.sqrt(max(_mf['peak'], 0.0)))
                                _floor = float(np.sqrt(max(_mf['base'], 0.0)) /
                                               max(_amp, 1e-30))
                                variants.append((drive_scale, dec, vr['q_ratio'],
                                                 vm['k_vector'], window_frac, repeat,
                                                 drive_counts, _mf['resid'], _floor,
                                                 vr['measured']['adc_peak_frac'],
                                                 vr['reference_move_hz']/vr['baseline']['lw'],
                                                 vm['k_complex'],
                                                 _amp/max(drive_variant, 1e-30),
                                                 vr['measured']['fit_diagnostics']['half_power_disagreement'],
                                                 vr['measured']['fit_diagnostics']['structured_residual'],
                                                 vr['measured']['fit_diagnostics']['residual_rms']))
                                print('      repeat %d, drive %.0f%%, R=%d: Q %.1f, '
                                      'Kq %.3f, Kvector %.3f, linewidth %.2f kHz, '
                                      '%d pts/lw, zero-IF R=%d, navg %d, '
                                      'resid %.4f, floor/peak %.3f, ADC %.3f, ref move %.4f lw'
                                      % (repeat + 1, 100*drive_scale, 1 << dec,
                                         vr['measured']['fit']['Q'], vm['k_q'],
                                         vm['k_vector'],
                                         vr['measured']['fit']['lw']/1e3,
                                         vr['measured']['in_linewidth'],
                                         1 << dec, navg, _mf['resid'], _floor,
                                         vr['measured']['adc_peak_frac'],
                                         vr['reference_move_hz']/vr['baseline']['lw']))
                            else:
                                variant_failures.append(vr.get('stop_reason', 'invalid trace'))
                                print('      repeat %d, drive %.6f, R=%d: REJECTED - %s'
                                      % (repeat+1, drive_variant, 1 << dec,
                                         vr.get('stop_reason', 'invalid trace')))
                        if HARD_STOP_REQUESTED:
                            break
                    if HARD_STOP_REQUESTED:
                        break
                if HARD_STOP_REQUESTED:
                    print('    HARD SAFETY STOP - remaining active measurements skipped')
                values = np.asarray(variants, dtype=float) if variants else np.empty((0, 16))
                drive_rows = []
                for ds in sorted(np.unique(values[:, 0]), reverse=True) if len(values) else []:
                    dv = values[values[:, 0] == ds]
                    _expected = _repeat_count * len(_dec_list)
                    receiver_ok = bool(len(dv) == _expected)
                    qspread = np.ptp(dv[:, 2]) / max(np.mean(dv[:, 2]), 1e-12)
                    kspread = np.ptp(dv[:, 3]) / max(np.mean(dv[:, 3]), 1e-12)
                    _reject = []
                    if len(dv) != _expected:
                        _reject.append('missing repeat/receiver measurements')
                    if qspread > DIAG_CONVERGENCE_TOL:
                        _reject.append('Q repeat spread %.1f%% > %.1f%%'
                                       % (100*qspread, 100*DIAG_CONVERGENCE_TOL))
                    if kspread > DIAG_CONVERGENCE_TOL:
                        _reject.append('K repeat spread %.1f%% > %.1f%%'
                                       % (100*kspread, 100*DIAG_CONVERGENCE_TOL))
                    if np.max(dv[:, 10]) > DIAG_REFERENCE_MOVE_MAX:
                        _reject.append('reference motion exceeds limit')
                    drive_rows.append({'scale': float(ds),
                                       'drive': float(DRIVE_FRAC * ds),
                                       'counts': int(round(np.mean(dv[:, 6]))),
                                       'q': float(np.mean(dv[:, 2])),
                                       'k': float(np.mean(dv[:, 3])),
                                       'k_complex': float(np.mean(dv[:, 11])),
                                       'k_standard_error': float(np.std(dv[:, 3], ddof=1) /
                                                                  np.sqrt(len(dv))),
                                       'variant_spread': float(max(qspread, kspread)),
                                       'fit_residual_max': float(np.max(dv[:, 7])),
                                       'floor_peak_max': float(np.max(dv[:, 8])),
                                       'adc_peak_max': float(np.max(dv[:, 9])),
                                       'reference_move_max_lw': float(np.max(dv[:, 10])),
                                       'coherent_per_drive': float(np.mean(dv[:, 12])),
                                       'half_power_disagreement_max': float(np.max(dv[:, 13])),
                                       'structured_residual_max': float(np.max(dv[:, 14])),
                                       'residual_rms_max': float(np.max(dv[:, 15])),
                                       'rejection_reasons': _reject,
                                       'receiver_repeat_ok': bool(receiver_ok and not _reject)})
                # One opposite-phase measurement per drive separates a real
                # loop-gain change from bias in the narrowed linewidth alone.
                for drow in drive_rows:
                    _ds = drow['scale']
                    _pairs = []
                    _width_pairs = []
                    for _repeat in range(_repeat_count):
                        _pr = variant_records.get((_repeat, _ds, int(d_primary)))
                        _mr = variant_records.get((_repeat, _ds, 'minus'))
                        _pm = (_metrics(_pr, _mr) if _pr is not None and _mr is not None
                               else None)
                        _jm = (_joint_paired_metrics(_pr, _mr)
                               if _pr is not None and _mr is not None else None)
                        if _pm is not None and _jm is not None:
                            _pairs.append((_pm['k_paired'], _jm['k_joint'],
                                           _jm['k_complex_paired']))
                            _b0 = 0.5*(_pr['baseline']['lw']+_mr['baseline']['lw'])
                            _width_pairs.append((_pr['measured']['fit']['lw']/_b0,
                                                 _mr['measured']['fit']['lw']/_b0))
                    if not _pairs:
                        drow['k_paired'] = float('nan'); continue
                    _pairs = np.asarray(_pairs, dtype=float)
                    drow['k_paired'] = float(np.mean(_pairs[:, 0]))
                    drow['k_paired_standard_error'] = float(
                        np.std(_pairs[:, 0], ddof=1)/np.sqrt(len(_pairs))
                        if len(_pairs) > 1 else float('nan'))
                    drow['k_joint'] = float(np.mean(_pairs[:, 1]))
                    drow['k_complex_paired'] = float(np.mean(_pairs[:, 2]))
                    drow['narrow_lw_over_b0'] = float(np.mean(np.asarray(_width_pairs)[:, 0]))
                    drow['broad_lw_over_b0'] = float(np.mean(np.asarray(_width_pairs)[:, 1]))
                    print('      drive %.0f%% diagnostic: Kvector %.4f, Kpaired %.4f, '
                          'Kjoint %.4f, Kcomplex-pair %.4f, difference %+.4f%s'
                          % (100*_ds, drow['k'], drow['k_paired'], drow['k_joint'],
                             drow['k_complex_paired'],
                             drow['k'] - drow['k_paired'],
                             ('; REJECTED: ' + '; '.join(drow['rejection_reasons'])
                              if drow['rejection_reasons'] else '')))
                usable = [d for d in drive_rows if d['receiver_repeat_ok']]
                plateau_required = min(DIAG_PLATEAU_POINTS, max(1, len(drive_rows)))
                low = sorted(usable, key=lambda d: d['drive'])[:plateau_required]
                if len(low) >= plateau_required:
                    _low_k = np.asarray([d['k'] for d in low], dtype=float)
                    _plateau_centre = float(np.median(_low_k))
                    _plateau_abs = float(np.max(np.abs(_low_k - _plateau_centre)))
                    plateau_spread = _plateau_abs / max(abs(_plateau_centre), 1e-12)
                    plateau_limit = max(DIAG_PLATEAU_ABS_K,
                                        DIAG_CONVERGENCE_TOL*abs(_plateau_centre))
                else:
                    _plateau_abs = plateau_spread = float('nan')
                    plateau_limit = DIAG_PLATEAU_ABS_K
                row['drive_results'] = drive_rows
                _paired = np.asarray([d.get('k_paired', np.nan) for d in drive_rows],
                                     dtype=float)
                _vector = np.asarray([d['k'] for d in drive_rows], dtype=float)
                _both = np.isfinite(_paired) & np.isfinite(_vector)
                _joint = np.asarray([d.get('k_joint', np.nan) for d in drive_rows], dtype=float)
                # Mandatory agreement uses only validated physical estimators.
                # Complex K is reported above but remains diagnostic-only.
                _all_est = np.column_stack([_vector, _paired, _joint])
                _valid_est = np.all(np.isfinite(_all_est), axis=1)
                _diff = np.max(_all_est[_valid_est], axis=1)-np.min(_all_est[_valid_est], axis=1)
                _diff_se = (float(np.std(_diff, ddof=1) / np.sqrt(len(_diff)))
                            if len(_diff) >= 2 else float('nan'))
                row['agreement_abs_k'] = (float(abs(np.mean(_diff)))
                                          if len(_diff) else float('nan'))
                row['agreement_ci95_k'] = (DIAG_CONFIDENCE_Z * _diff_se
                                           if np.isfinite(_diff_se) else float('nan'))
                row['agreement_limit_k'] = float(max(
                    DIAG_K_AGREEMENT_ABS,
                    row['agreement_ci95_k'] if np.isfinite(row['agreement_ci95_k']) else 0.0))
                row['pair_requirement'] = ('optional above K %.2f' % HIGH_K_PAIR_OPTIONAL
                                           if predicted_k >= HIGH_K_PAIR_OPTIONAL else 'required')
                if predicted_k >= HIGH_K_PAIR_OPTIONAL:
                    _vfinite = _vector[np.isfinite(_vector)]
                    _vspread = float(np.ptp(_vfinite)) if len(_vfinite)>=2 else float('inf')
                    row['agreement_abs_k'] = _vspread
                    row['agreement_ok'] = bool(row['quadrature_available'] and
                        len(_vfinite)>=2 and _vspread <= row['agreement_limit_k'])
                    if not row['broadening_available']:
                        row['broadening_status'] = 'broadening SNR insufficient; optional at high K'
                    else:
                        row['broadening_status'] = 'measured as cross-check'
                else:
                    row['agreement_ok'] = bool(np.isfinite(row['agreement_abs_k']) and
                                               row['agreement_abs_k'] <= row['agreement_limit_k'])
                    row['broadening_status'] = ('measured' if row['broadening_available']
                                                  else 'required broadening measurement missing')
                row['paired_drive_spread_k'] = (float(np.ptp(_paired[_both]))
                                                if np.sum(_both) >= 2 else float('nan'))
                row['vector_drive_spread_k'] = (float(np.ptp(_vector[_both]))
                                                if np.sum(_both) >= 2 else float('nan'))
                if np.sum(_both) >= 3:
                    if row['paired_drive_spread_k'] > DIAG_PLATEAU_ABS_K:
                        row['drive_dependence_diagnosis'] = (
                            'physical loop response changes with excitation (analogue/digital nonlinearity)')
                    elif row['vector_drive_spread_k'] > DIAG_PLATEAU_ABS_K:
                        row['drive_dependence_diagnosis'] = (
                            'narrowed-line/reference estimator bias; paired K is drive-independent')
                    else:
                        row['drive_dependence_diagnosis'] = 'no resolved drive dependence'
                else:
                    row['drive_dependence_diagnosis'] = 'insufficient paired-drive diagnostics'
                row['trace_rejections'] = list(variant_failures)
                row['receiver_repeat_ok'] = bool(len(usable) >= 2)
                row['small_signal_plateau_ok'] = bool(len(low) >= plateau_required and
                                                       _plateau_abs <= plateau_limit)
                row['plateau_spread_frac'] = float(plateau_spread)
                row['plateau_abs_k'] = float(_plateau_abs)
                row['plateau_limit_k'] = float(plateau_limit)
                row['settling_converged'] = bool(all(
                    r.get('settling_converged',False) for r in variant_records.values()
                    if r.get('valid')))
                row['decimation_converged'] = bool(row['receiver_repeat_ok'])
                row['converged'] = bool(row['receiver_repeat_ok'] and
                                        row['small_signal_plateau_ok'] and
                                        row['agreement_ok'] and row['settling_converged'])
                row['convergence_spread_frac'] = float(plateau_spread)
                row['receiver_window_ok'] = bool(len(usable) >= 2)
                if low:
                    row['k_small_signal'] = float(np.mean([d['k'] for d in low]))
                    row['q_small_signal'] = float(np.mean([d['q'] for d in low]))
                    row['small_signal_drive_range'] = [float(min(d['drive'] for d in low)),
                                                       float(max(d['drive'] for d in low))]
                    _selected_scale = float(min(low, key=lambda d: d['drive'])['scale'])
                    _selected = variant_records.get((0, _selected_scale, int(d_primary)))
                    if _selected is not None and _selected.get('valid'):
                        _selected_minus = _variant_point(
                            A, _selected['phase_deg'] + 180.0, F0, LW,
                            _broadening_probe_drive(_selected['drive_frac']),
                            int(np.ceil(FB_NAVG/_selected_scale**2)), d_primary)
                        DIAG_RECORDS.append(_selected_minus)
                        _selected_metrics = _metrics(_selected, _selected_minus)
                        if _selected_metrics is not None:
                            _selected_joint = _joint_paired_metrics(_selected, _selected_minus)
                            row.update({'record': _selected,
                                        'k_q': _selected_metrics['k_q'],
                                        'k_paired': _selected_metrics['k_paired'],
                                        'k_vector': (row['k_vector']
                                            if predicted_k >= HIGH_K_PAIR_OPTIONAL
                                            else _selected_metrics['k_vector']),
                                        'k_complex': _selected_metrics['k_complex'],
                                        'theta_deg': _selected_metrics['theta_deg'],
                                        'agreement_frac': _selected_metrics['agreement_frac'],
                                        'q': _selected['measured']['fit']['Q'],
                                        'q_ratio': _selected['q_ratio'],
                                        'k_joint': (_selected_joint['k_joint']
                                                    if _selected_joint is not None else float('nan'))})
                else:
                    row['k_small_signal'] = row['q_small_signal'] = float('nan')
                if len(usable) >= 4:
                    _cd = np.asarray([d['drive'] for d in usable], dtype=float)
                    _ck = np.asarray([d['k'] for d in usable], dtype=float)
                    _compression_corr = float(np.corrcoef(_cd, _ck)[0, 1])
                    _compression_recovery = float(_ck[-1] - _ck[0])
                else:
                    _compression_corr = _compression_recovery = float('nan')
                row['compression_correlation'] = _compression_corr
                row['compression_recovery_k'] = _compression_recovery
                row['compression_signature'] = bool(
                    len(usable) >= 4 and _compression_corr <= -0.70 and
                    _compression_recovery > max(DIAG_PLATEAU_ABS_K,
                                                0.03*abs(usable[0]['k'])))
                if not row['receiver_repeat_ok']:
                    row['stop_reason'] = 'fewer than two drives passed repeat/receiver checks'
                elif not row['small_signal_plateau_ok']:
                    row['stop_reason'] = ('small-signal plateau not reached before the DAC excitation floor'
                                          if len(low) < plateau_required else
                                          'lowest drives still change K by more than 3%')
            else:
                row['converged'] = True
                row['convergence_spread_frac'] = 0.0

            DIAG_SUMMARY.append(row)
            print('    phase %.1f deg, Q/Q0 %.3f, Kq %.3f, Kpair %.3f, '
                  'Kvector %.3f, theta %+.2f deg, agreement %.1f%%'
                  % (row['phase_deg'], row['q_ratio'], row['k_q'], row['k_paired'],
                     row['k_vector'], row['theta_deg'], 100*row['agreement_frac']))
            if np.isfinite(row.get('k_small_signal', float('nan'))):
                print('    small-signal K %.4f, Q/Q0 %.3f over drive %.6f to %.6f; '
                      'max |dK| %.4f (limit %.4f): %s'
                      % (row['k_small_signal'], row['q_small_signal'],
                         row['small_signal_drive_range'][0],
                         row['small_signal_drive_range'][1],
                         row['plateau_abs_k'], row['plateau_limit_k'],
                         'PASS' if row['small_signal_plateau_ok'] else 'STOP'))
            else:
                print('    small-signal plateau: insufficient eligible points (%d/%d)'
                      % (len(low), plateau_required))
            print('    uncertainty-aware agreement: |mean dK| %.4f <= %.4f: %s'
                  % (row['agreement_abs_k'], row['agreement_limit_k'],
                     'PASS' if row['agreement_ok'] else 'STOP'))
            print('    diagnosis: %s' % row['drive_dependence_diagnosis'])
            if row.get('compression_signature'):
                print('    WARNING - K increased when drive was reduced: driven compression')
                print('              the converged low-drive plateau, not the compressed point, is authoritative')
            if not row['converged']:
                if predicted_k < DIAG_GATE_MIN_K and row['receiver_repeat_ok']:
                    print('    WARNING - %s; retained as low-gain diagnostic and continuing'
                          % row['stop_reason'])
                else:
                    print('    SOFT FAILURE - %s; continuing to characterise higher gains' % row['stop_reason'])
            if not QUALIFICATION_ONLY and row['k_small_signal'] >= 0.80:
                dk = (0.005 if row['k_small_signal'] > 0.97 else
                      0.01 if row['k_small_signal'] > 0.92 else
                      0.02 if row['k_small_signal'] > 0.85 else 0.05)
                local_slope = row['k_small_signal'] / max(A, 1e-9)
                next_a = A + dk / max(local_slope, 1e-9)
                gain_queue = ([next_a] if next_a <= FB_GAIN_MAX else [])
    finally:
        DRIVE_FRAC, FB_NAVG = DIAG_BASE_DRIVE, DIAG_BASE_NAVG
        gen.off(); ps.off()
if DIAG_SUMMARY:
    # The feedback-off brackets span the entire diagnostic, extending Step 7's
    # drift record over the exact period in which K is estimated.
    _diag_refs = [(r[key]['time'], r[key]['fit']['f0'], r[key]['fit']['lw'])
                  for r in DIAG_RECORDS if r.get('valid')
                  for key in ('before', 'after')]
    if len(_diag_refs) >= 4:
        _diag_refs = np.asarray(_diag_refs, dtype=float)
        _diag_order = np.argsort(_diag_refs[:, 0]); _diag_refs = _diag_refs[_diag_order]
        DIAG_DRIFT_COVERAGE_MIN = float((_diag_refs[-1, 0]-_diag_refs[0, 0])/60.0)
        _diag_lw_mean = float(np.mean(_diag_refs[:, 2]))
        _diag_lw_span = float(np.ptp(_diag_refs[:, 2]) / _diag_lw_mean)
        _diag_elapsed = _diag_refs[:, 0]-_diag_refs[0, 0]
        _diag_lw_slope, _diag_lw_intercept = np.polyfit(
            _diag_elapsed, _diag_refs[:, 2], 1)
        _diag_lw_residual = (_diag_refs[:, 2] -
                             (_diag_lw_slope*_diag_elapsed+_diag_lw_intercept))
        _diag_lw_unc95 = float(1.96*np.std(_diag_lw_residual, ddof=1)/_diag_lw_mean)
        _diag_lw_trend_k = float(_diag_lw_slope*_diag_elapsed[-1]/_diag_lw_mean)
        _diag_f_slope = float(np.polyfit(_diag_refs[:, 0]-_diag_refs[0, 0],
                                         _diag_refs[:, 1], 1)[0])
        DIAG_TEMPORAL_OK = bool(_diag_lw_unc95 <= FIT_MAX_REFERENCE_K_MOVE)
        print('  full-campaign references: centre %+.1f Hz/min; linewidth trend %+.4f K, '
              'span %.4f K, residual 95%% %.4f K; %s'
              % (60*_diag_f_slope, _diag_lw_trend_k, _diag_lw_span, _diag_lw_unc95,
                 'PASS' if DIAG_TEMPORAL_OK else 'STOP'))
    else:
        DIAG_DRIFT_COVERAGE_MIN = 0.0
        _diag_lw_span = _diag_lw_unc95 = _diag_lw_trend_k = _diag_f_slope = float('nan')
        DIAG_TEMPORAL_OK = False
    _gated_rows = [r for r in DIAG_SUMMARY
                   if r['gain'] * max(_g('k_slope', 0.0), 0.0) >= DIAG_GATE_MIN_K]
    DIAG_AGREEMENT_OK = bool(_gated_rows) and all(
        bool(r.get('agreement_ok', False)) for r in _gated_rows)
    DIAG_REPEAT_OK = bool(_gated_rows) and all(bool(r['converged'])
                                                   for r in _gated_rows)
    DIAG_CONVERGENCE_OK = bool(DIAG_REPEAT_OK and DIAG_TEMPORAL_OK)
    highest = max(DIAG_SUMMARY, key=lambda r: r['gain'])
    _k0_rows = [r for r in DIAG_SUMMARY if r.get('converged') and
                np.isfinite(r.get('k_small_signal', float('nan')))]
    K0_FIT = {'valid': False}
    if len(_k0_rows) >= 3:
        _k0a = np.asarray([r['gain'] for r in _k0_rows], dtype=float)
        _k0 = np.asarray([r['k_small_signal'] for r in _k0_rows], dtype=float)
        _k0s = float(np.sum(_k0a*_k0) / np.sum(_k0a**2))
        _k0free, _k0int = np.polyfit(_k0a, _k0, 1)
        _k0resid = _k0 - _k0s*_k0a
        K0_FIT = {'valid': True, 'slope_zero': _k0s,
                  'slope_free': float(_k0free), 'intercept_free': float(_k0int),
                  'rms_zero': float(np.sqrt(np.mean(_k0resid**2))),
                  'onset_gain_linear': float(1.0/_k0s) if _k0s > 0 else float('nan')}
        print('  small-signal K0(A): %.4f A; free fit %.4f A %+.4f'
              % (_k0s, _k0free, _k0int))
        print('  linear onset %.3f is extrapolated and is not the SEO measurement'
              % K0_FIT['onset_gain_linear'])
    K_CALIBRATION = {'valid': False}
    _cal_rows = [r for r in DIAG_SUMMARY if r.get('converged') and
                 r.get('agreement_ok', False) and
                 r.get('k_small_signal', 9.0) <= 0.55]
    if len(_cal_rows) >= 3:
        _ca = np.asarray([r['gain'] for r in _cal_rows], dtype=float)
        _ck = np.asarray([r['k_paired'] for r in _cal_rows], dtype=float)
        _cs = np.asarray([max(r.get('agreement_ci95_k', 0.0)/1.96, 0.002)
                          for r in _cal_rows], dtype=float)
        _cw = 1.0/_cs**2
        _slope = float(np.sum(_cw*_ca*_ck)/np.sum(_cw*_ca**2))
        _free_slope, _free_intercept = np.polyfit(_ca, _ck, 1)
        _ratio = _ck/_ca
        _curvature = float(np.polyfit(_ca, _ratio, 1)[0])
        K_CALIBRATION = {'valid': True, 'slope': _slope,
                         'free_slope': float(_free_slope),
                         'free_intercept': float(_free_intercept),
                         'k_per_a_trend': _curvature,
                         'gain': _ca, 'k_paired': _ck, 'sigma_k': _cs,
                         'compression_warning': bool(_curvature < -0.03*_slope)}
        print('  measured K(A) calibration: K = %.4f A; K/A trend %+.4f per A%s'
              % (_slope, _curvature,
                 ' - STOP: possible compression' if K_CALIBRATION['compression_warning'] else ''))
    expected = float(K0_FIT.get('slope_zero', _g('k_slope', float('nan'))) *
                     highest['gain'])
    missing = expected - highest.get('k_small_signal', highest['k_vector'])
    if not DIAG_AGREEMENT_OK:
        FIRST_LIMITING_FACTOR = ('independent K estimates disagree beyond the absolute/95% '
                                 'confidence limit')
    elif not globals().get('DIAG_REPEAT_OK', False):
        _reasons = [r.get('stop_reason', '') for r in DIAG_SUMMARY if not r.get('converged')]
        FIRST_LIMITING_FACTOR = next((x for x in _reasons if x),
                                     'small-signal convergence was not established')
    elif not globals().get('DIAG_TEMPORAL_OK', False):
        FIRST_LIMITING_FACTOR = ('detrended feedback-off linewidth uncertainty exceeds '
                                 '0.01 K')
    elif missing > 0.05:
        FIRST_LIMITING_FACTOR = 'measured loop effect saturates below the low-gain extrapolation'
    else:
        FIRST_LIMITING_FACTOR = 'electrical protection prevents a safe approach to oscillation'
    _stable_rows = [r for r in DIAG_SUMMARY if r.get('converged') and
                    r.get('agreement_ok', False)]
    if _stable_rows:
        Q_CONVERGED_RECOMMENDATION_OK = True
        _best_row = max(_stable_rows, key=lambda r: r['q_ratio'])
        Q_RECOMMENDATION = _best_row['record']
        _pq = np.asarray(_best_row['phase_q_values'], dtype=float)
        Q_RECOMMENDATION['uncertainty'] = float(np.std(_pq, ddof=1)
                                                    if len(_pq) > 1 else 0.0)
        Q_RECOMMENDATION['safe_phase_neighbourhood_deg'] = [
            float(_best_row['phase_deg'] - 0.5),
            float(_best_row['phase_deg'] + 0.5)]
        Q_RECOMMENDATION['limiting_factor'] = FIRST_LIMITING_FACTOR
        print('  final Q recommendation updated from converged diagnostics: Q %.1f at A %.3f'
              % (Q_RECOMMENDATION['measured']['fit']['Q'],
                 Q_RECOMMENDATION['gain']))
else:
    highest = None
    FIRST_LIMITING_FACTOR = 'no valid high-gain diagnostic point'

# Verify the shift result with the same drive/receiver discipline. A shift
# beyond B0/2 is permitted only when its measured vector and damping component
# are reported; B0/2 is a ceiling only under the additional |K| < 1 condition.
SHIFT_CONVERGENCE_OK = False
SHIFT_VARIANTS = []
if SHIFT_RECOMMENDATION is not None:
    _sd = choose_dec(SHIFT_RECOMMENDATION['measured']['fit']['lw'])
    _sda = max(8, _sd - 1)
    for repeat in range(DIAG_REPEAT_COUNT):
        for drive_scale in (1.0, 0.70):
            for dec in (_sd, _sda):
                vr = _variant_point(SHIFT_RECOMMENDATION['gain'],
                                    SHIFT_RECOMMENDATION['phase_deg'],
                                    SHIFT_RECOMMENDATION['measured']['fit']['f0'],
                                    SHIFT_RECOMMENDATION['measured']['fit']['lw'],
                                    DIAG_BASE_DRIVE * drive_scale,
                                    int(np.ceil(DIAG_BASE_NAVG/drive_scale**2)), dec)
                DIAG_RECORDS.append(vr)
                vm = _metrics(vr)
                if vm is not None:
                    wf = 0.0 # exact tracked zero-IF; retained column for export compatibility
                    SHIFT_VARIANTS.append((vr['shift_hz'], vm['k_vector'], wf))
    if len(SHIFT_VARIANTS) == 4 * DIAG_REPEAT_COUNT:
        _sv = np.asarray(SHIFT_VARIANTS, dtype=float)
        _ss = np.ptp(_sv[:, 0]) / max(np.mean(np.abs(_sv[:, 0])), 1e-12)
        _sk = np.ptp(_sv[:, 1]) / max(np.mean(np.abs(_sv[:, 1])), 1e-12)
        SHIFT_CONVERGENCE_OK = bool(max(_ss, _sk) <= DIAG_CONVERGENCE_TOL)
        SHIFT_RECOMMENDATION['limiting_factor'] = (
            'damping stability K*cos(theta) < 1; vector magnitude may exceed one'
            if SHIFT_CONVERGENCE_OK else
            'shift or vector K changes with drive or receiver settings')

# Full-scale OUT1 is +10 dBm. The declared pad/rating must retain 3 dB margin.
required_pad_db = max(0.0, 10.0 - AMP_IN_MAX_DBM + OSC_SAFETY_MARGIN_DB)
protection_margin_db = EXT_PAD_DB - required_pad_db
required_interstage_db = max(0.0, FIRST_P1DB_DBM - SECOND_IN_MAX_DBM
                                 + OSC_SAFETY_MARGIN_DB)
interstage_margin_db = INTERSTAGE_PAD_DB - required_interstage_db
measured_k_per_a = (highest.get('k_small_signal', highest['k_vector']) / highest['gain']
                    if highest and highest['gain'] > 0 else 0.0)
transfer_ok = measured_k_per_a >= 1.0 / OSC_DESIGN_MAX_A
SAFETY_GATE_OK = bool(OSC_PROTECTION_CONFIRMED and HIGH_GAIN_CHAIN_CONFIRMED and
                      globals().get('DRIFT_VALID', False) and TERMINATED_OUTPUT and
                      protection_margin_db >= 0.0 and
                      interstage_margin_db >= 0.0 and transfer_ok)

print('')
print('  K-estimator agreement : %s' % ('PASS' if DIAG_AGREEMENT_OK else 'STOP'))
print('  drive/receiver repeat : %s' % ('PASS' if globals().get('DIAG_REPEAT_OK', False) else 'STOP'))
print('  campaign temporal     : %s' % ('PASS' if globals().get('DIAG_TEMPORAL_OK', False) else 'STOP'))
print('  drift/scatter gate    : %s' % ('PASS' if globals().get('DRIFT_VALID', False) else 'STOP'))
print('  external chain check  : %s' % ('PASS' if HIGH_GAIN_CHAIN_CONFIRMED else 'STOP'))
print('  measured K per A      : %.4f (need >= %.4f for K=1 by A=%.2f)'
      % (measured_k_per_a, 1.0/OSC_DESIGN_MAX_A, OSC_DESIGN_MAX_A))
print('  OUT1 protection       : %.1f dB declared, %.1f dB required'
      % (EXT_PAD_DB, required_pad_db))
print('  interstage protection : %.1f dB declared, %.1f dB required'
      % (INTERSTAGE_PAD_DB, required_interstage_db))
print('  final output ceiling  : %+.1f dBm (3 dB below P1dB)'
      % AMP_OUT_MAX_DBM)


def _drive_off_growth(A, phi):
    """Three probe-off captures; onset also requires external confirmation."""
    record = {'gain': float(A), 'phase_deg': float(phi % 360.0),
              'oscillating': False, 'peak_frac': float('nan'),
              'growth_ratio': float('nan')}
    try:
        gen.off(); ps.off(); time.sleep(SETTLE)
        noise_x, _ = sc.acquire(dec=64, n=SC_NMAX)
        noise_rms = float(np.sqrt(np.mean((noise_x-np.mean(noise_x))**2)))
        ps.set(A, phi)
        time.sleep(SETTLE)
        captures = []
        for _ in range(SEO_CONFIRM_CAPTURES):
            x, fs = sc.acquire(dec=64, n=SC_NMAX); x = x-np.mean(x)
            half = len(x)//2
            r0 = float(np.sqrt(np.mean(x[:half]**2)))
            r1 = float(np.sqrt(np.mean(x[half:]**2)))
            rms = float(np.sqrt(np.mean(x**2)))
            captures.append({'peak_frac': float(np.max(np.abs(x)))/8191.0,
                             'growth_ratio': r1/max(r0, 1e-12),
                             'db_above_noise': 20*np.log10(rms/max(noise_rms, 1e-12))})
        rp_seen = all(c['db_above_noise'] >= SEO_MIN_DB_ABOVE_NOISE
                      for c in captures)
        peak = max(c['peak_frac'] for c in captures)
        growth = max(c['growth_ratio'] for c in captures)
        record.update({'sample_rate_hz': float(fs), 'noise_rms_counts': noise_rms,
                       'captures': captures, 'peak_frac': peak,
                       'growth_ratio': growth, 'red_pitaya_carrier_confirmed': rp_seen,
                       'external_carrier_confirmed': bool(SEO_EXTERNAL_CARRIER_CONFIRMED),
                       'oscillating': bool(rp_seen and SEO_EXTERNAL_CARRIER_CONFIRMED)})
        return record
    finally:
        gen.off(); ps.off()


if not ONSET_SEARCH_ARMED:
    BOUNDARY_STOP_REASON = 'onset search is not armed (normal safe state)'
elif not SAFETY_GATE_OK:
    BOUNDARY_STOP_REASON = 'amplifier protection/loop-transfer safety gate failed'
elif not DIAG_AGREEMENT_OK or not DIAG_CONVERGENCE_OK:
    BOUNDARY_STOP_REASON = 'diagnostic K agreement or convergence failed'
elif highest is None or highest.get('k_small_signal', highest['k_vector']) < 0.98:
    BOUNDARY_STOP_REASON = 'highest repeatedly measured stable K is below 0.98'
else:
    # Find the first oscillating point, then bisect to a K interval below 0.01.
    low_a, low_k = highest['gain'], highest.get('k_small_signal', highest['k_vector'])
    phase = highest['phase_deg']
    high_a = None
    step_a = max(0.005/measured_k_per_a, 0.002)
    try:
        A = low_a + step_a
        while A <= min(OSC_DESIGN_MAX_A, FB_GAIN_MAX):
            probe = _drive_off_growth(A, phase); ONSET_RECORDS.append(probe)
            if probe['oscillating']:
                high_a = A; break
            low_a = A
            A += step_a
        if high_a is None:
            BOUNDARY_STOP_REASON = 'no oscillation found inside the audited gain range'
        else:
            while measured_k_per_a * (high_a - low_a) >= 0.01:
                mid = 0.5*(low_a + high_a)
                probe = _drive_off_growth(mid, phase); ONSET_RECORDS.append(probe)
                if probe['oscillating']:
                    high_a = mid
                else:
                    low_a = mid
            ONSET_BRACKET = {'last_stable_gain': float(low_a),
                             'first_oscillating_gain': float(high_a),
                             'k_interval': float(measured_k_per_a*(high_a-low_a)),
                             'onset_gain': float(0.5*(low_a+high_a)),
                             'uncertainty_gain': float(0.5*(high_a-low_a)),
                             'phase_deg': float(phase)}
    finally:
        gen.off(); ps.off()

if BOUNDARY_STOP_REASON:
    print('  STOP - ' + BOUNDARY_STOP_REASON)
if ONSET_BRACKET:
    print('  PASS - K=1 onset bracketed between gain %.5f and %.5f (dK < %.3f)'
          % (ONSET_BRACKET['last_stable_gain'],
             ONSET_BRACKET['first_oscillating_gain'], ONSET_BRACKET['k_interval']))

# Quick descending-gain pass: one middle drive and the primary receiver only.
# This adds little runtime but separates hysteresis/warming from gain curvature.
HYSTERESIS_ROWS = []
_hyst_drive = min(0.0017, DRIVE_LINEAR_MAX)
for _src in sorted(DIAG_SUMMARY, key=lambda r: r['gain'], reverse=True):
    if HARD_STOP_REQUESTED:
        break
    _dec = int(_src.get('primary_dec_log2', choose_dec(LW)))
    _hp = _variant_point(_src['gain'], _src['phase_deg'], F0, LW,
                         _hyst_drive, DIAG_FIXED_NAVG, _dec, sweep_direction='down')
    _hm = _variant_point(_src['gain'], _src['phase_deg']+180.0, F0, LW,
                         _broadening_probe_drive(_hyst_drive), DIAG_FIXED_NAVG,
                         _dec, sweep_direction='down')
    DIAG_RECORDS.extend([_hp, _hm])
    _hmet = _metrics(_hp, _hm)
    HYSTERESIS_ROWS.append({'gain': float(_src['gain']),
                            'k_paired': (_hmet['k_paired'] if _hmet else float('nan')),
                            'centre_hz': (_hp['measured']['fit']['f0'] if _hp.get('valid') else float('nan')),
                            'valid': bool(_hmet is not None),
                            'stop_reason': ('' if _hmet else '%s; %s' %
                                            (_hp.get('stop_reason',''), _hm.get('stop_reason','')))})

def phase_schedule_for_k(k_value, narrow_phase):
    if k_value <= 0.70:
        return list(np.arange(0.0,360.0,FULL_CIRCLE_PHASE_STEP_DEG))
    return [float((narrow_phase+x)%360.0) for x in
            (-3.0,-1.0,0.0,1.0,3.0,90.0,180.0,270.0)]

WIDER_MAP_RECORDS = []
if RUN_WIDER_MAP_AFTER_QUALIFICATION:
    for _src in [r for r in DIAG_SUMMARY if r.get('converged')]:
        _kmap=float(_src.get('k_small_signal',_src.get('k_vector',np.nan)))
        for _phi in phase_schedule_for_k(_kmap,_src['phase_deg']):
            _maprec=_verify_point('wider_map',_src['gain'],_phi,F0,LW,
                                  dec_log2=_src.get('primary_dec_log2'))
            WIDER_MAP_RECORDS.append(_maprec); DIAG_RECORDS.append(_maprec)
            if HARD_STOP_REQUESTED: break
        if HARD_STOP_REQUESTED: break

# One compact qualification figure: every panel addresses a distinct failure mode.
_flat = [(r, d) for r in DIAG_SUMMARY for d in r.get('drive_results', [])]
fig, axs = plt.subplots(4, 2, figsize=(12.0, 14.0), sharex=False)
if _flat:
    _drives = sorted(set(d['drive'] for r, d in _flat))
    _colors = {d: plt.cm.viridis(i/max(len(_drives)-1, 1)) for i, d in enumerate(_drives)}
    for _drive in _drives:
        _rows = [(r,d) for r,d in _flat if abs(d['drive']-_drive)<1e-12]
        _a = np.asarray([r['gain'] for r,d in _rows]); _kp = np.asarray([d.get('k_paired',np.nan) for r,d in _rows])
        _kv = np.asarray([d.get('k',np.nan) for r,d in _rows]); _kj = np.asarray([d.get('k_joint',np.nan) for r,d in _rows])
        _ok = np.asarray([d.get('receiver_repeat_ok',False) for r,d in _rows])
        for _y, _marker, _label in ((_kp,'s','pair'),(_kv,'^','vector'),(_kj,'v','joint')):
            axs[0,0].plot(_a[_ok], _y[_ok], _marker+'-', color=_colors[_drive],
                          label='%.4f %s' % (_drive,_label))
            axs[0,0].plot(_a[~_ok], _y[~_ok], _marker, mfc='none', mec=_colors[_drive])
        axs[0,1].plot(_a, _kp/np.maximum(_a,1e-12), 'o-', color=_colors[_drive], label='drive %.4f'%_drive)
        axs[1,0].plot(_a, [d.get('coherent_per_drive',np.nan) for r,d in _rows], 'o-', color=_colors[_drive])
        axs[1,1].plot(_a, [d.get('narrow_lw_over_b0',np.nan) for r,d in _rows], 'o-', color=_colors[_drive])
        axs[1,1].plot(_a, [d.get('broad_lw_over_b0',np.nan) for r,d in _rows], 's--', color=_colors[_drive])
        axs[2,1].plot(_a, 100*np.asarray([d.get('half_power_disagreement_max',np.nan) for r,d in _rows]), 'o-', color=_colors[_drive])
        axs[2,1].plot(_a, 100*np.asarray([d.get('structured_residual_max',np.nan) for r,d in _rows]), 'x--', color=_colors[_drive])
    _valid_low = [(r,d) for r,d in _flat if d.get('receiver_repeat_ok') and np.isfinite(d.get('k_paired',np.nan))]
    if _valid_low:
        _la=np.asarray([r['gain'] for r,d in _valid_low]); _lk=np.asarray([d['k_paired'] for r,d in _valid_low])
        _sel=_la<=np.percentile(_la,40); QUALIFICATION_K_PER_A=float(np.sum(_la[_sel]*_lk[_sel])/np.sum(_la[_sel]**2))
        _xx=np.linspace(0,max(_la)*1.03,100); axs[0,0].plot(_xx,QUALIFICATION_K_PER_A*_xx,'k--',label='low-A fit')
        axs[0,1].axhline(QUALIFICATION_K_PER_A,color='k',ls='--')
    else:
        QUALIFICATION_K_PER_A=float('nan')
_dg=np.asarray([r['gain'] for r in DIAG_SUMMARY],dtype=float)
axs[2,0].plot(_dg,[r['phase_deg'] for r in DIAG_SUMMARY],'o-',label='narrowing phase')
axs[2,0].axhline(PHI_SHIFT,color=CLR_ORANGE,ls='--',label='quadrature phase')
_up={}
for _r in DIAG_SUMMARY:
    _cand=[d for d in _r.get('drive_results',[]) if np.isfinite(d.get('k_paired',np.nan))]
    _up[round(_r['gain'],6)]=(min(_cand,key=lambda d:abs(d['drive']-_hyst_drive))['k_paired']
                               if _cand else float('nan'))
_ha=np.asarray([r['gain'] for r in HYSTERESIS_ROWS]); _hk=np.asarray([r['k_paired'] for r in HYSTERESIS_ROWS])
axs[3,0].plot(_dg,[_up.get(round(a,6),np.nan) for a in _dg],'o-',label='up')
axs[3,0].plot(_ha,_hk,'s--',label='down')
axs[3,1].axhline(-40,color='r',ls=':',label='harmonic limit')
if np.isfinite(MEASURED_WORST_HARMONIC_DBC): axs[3,1].plot(_dg,np.full_like(_dg,MEASURED_WORST_HARMONIC_DBC),'o-',label='worst harmonic')
if np.isfinite(MEASURED_BROADBAND_RISE_DB): axs[3,1].text(.02,.08,'broadband rise %.1f dB'%MEASURED_BROADBAND_RISE_DB,transform=axs[3,1].transAxes)
_titles=('K against A','K/A: compression view','coherent response / drive','opposite-phase widths / B0',
         'phase extrema','fit validity (%)','up/down hysteresis','external spectral evidence')
_ylabs=('K','K/A','counts / drive','B/B0','phase / deg','percent','K paired','dBc')
for _ax,_title,_yl in zip(axs.ravel(),_titles,_ylabs): _ax.set_title(_title); _ax.set_xlabel('gain A'); _ax.set_ylabel(_yl); _ax.grid(alpha=.25)
axs[0,0].legend(fontsize=6,ncol=2); axs[0,1].legend(fontsize=7); axs[2,0].legend(fontsize=7); axs[3,0].legend(fontsize=7); axs[3,1].legend(fontsize=7)
plt.tight_layout(); plt.show()

# Evidence-based cause classification for non-proportional K(A).
_valid_rows = [(r,d) for r,d in _flat if d.get('receiver_repeat_ok') and
               np.isfinite(d.get('k_paired',np.nan))]
_ratio_drop = False; _drive_signature = False; _response_drop = False
_fit_signature = False; _receiver_signature = False
if _valid_rows and np.isfinite(QUALIFICATION_K_PER_A):
    _ratio_drop = bool(min(d['k_paired']/r['gain'] for r,d in _valid_rows
                           if r['gain']>0) < 0.95*QUALIFICATION_K_PER_A)
    for _r in DIAG_SUMMARY:
        _ds=sorted([d for d in _r.get('drive_results',[]) if d.get('receiver_repeat_ok')],key=lambda d:d['drive'])
        if len(_ds)>=2 and (_ds[0].get('k_paired',np.nan)-_ds[-1].get('k_paired',np.nan))>0.015:
            _drive_signature=True
    for _drive in sorted(set(d['drive'] for r,d in _valid_rows)):
        _rs=sorted([(r['gain'],d['coherent_per_drive']) for r,d in _valid_rows if d['drive']==_drive])
        if len(_rs)>=2 and _rs[-1][1] < 0.95*_rs[0][1]: _response_drop=True
_fit_signature = any(d.get('half_power_disagreement_max',0)>0.05 or
                     d.get('structured_residual_max',0)>FIT_MAX_STRUCTURED_RESIDUAL
                     for r,d in _flat)
_receiver_signature = any(any('receiver' in x for x in d.get('rejection_reasons',[]))
                          for r,d in _flat)
_phase_rotation = (np.ptp([r['phase_deg'] for r in DIAG_SUMMARY])>3.0
                   if len(DIAG_SUMMARY)>=2 else False)
_hyst_diffs=[abs(_up.get(round(r['gain'],6),np.nan)-r['k_paired'])
             for r in HYSTERESIS_ROWS if r.get('valid') and np.isfinite(_up.get(round(r['gain'],6),np.nan))]
QUALIFICATION_HYSTERESIS_K=max(_hyst_diffs) if _hyst_diffs else float('inf')
_external_compression = bool((np.isfinite(MEASURED_WORST_HARMONIC_DBC) and
                              MEASURED_WORST_HARMONIC_DBC>-40.0) or
                             (np.isfinite(MEASURED_BROADBAND_RISE_DB) and
                              MEASURED_BROADBAND_RISE_DB>3.0))
if len(_valid_rows)<4:
    QUALIFICATION_DIAGNOSIS='insufficient evidence'
elif _ratio_drop and (_drive_signature or _response_drop or _external_compression):
    QUALIFICATION_DIAGNOSIS='analogue compression'
elif _phase_rotation and _ratio_drop:
    QUALIFICATION_DIAGNOSIS='gain-dependent phase rotation'
elif _fit_signature:
    QUALIFICATION_DIAGNOSIS='lineshape/fitting failure'
elif _receiver_signature:
    QUALIFICATION_DIAGNOSIS='receiver limitation'
elif QUALIFICATION_HYSTERESIS_K>0.01:
    QUALIFICATION_DIAGNOSIS='reference drift, warming or hysteresis'
elif _ratio_drop:
    QUALIFICATION_DIAGNOSIS='feedback-transfer nonlinearity (drive-independent)'
else:
    QUALIFICATION_DIAGNOSIS='K proportional within measured uncertainty'
_cause_scores = [
    ('analogue compression', 3*int(_ratio_drop)+2*int(_drive_signature)+int(_response_drop)+int(_external_compression)),
    ('gain-dependent phase rotation', 3*int(_phase_rotation)+int(_ratio_drop)),
    ('lineshape/fitting failure', 4*int(_fit_signature)),
    ('receiver limitation', 4*int(_receiver_signature)),
    ('reference drift/warming/hysteresis', 4*int(QUALIFICATION_HYSTERESIS_K>0.01)),
    ('feedback-transfer nonlinearity', 2*int(_ratio_drop and not _drive_signature and not _phase_rotation and not _fit_signature))]
RANKED_CAUSES = sorted([{'cause':name,'evidence_score':int(score)} for name,score in _cause_scores if score>0], key=lambda x:-x['evidence_score'])
if not RANKED_CAUSES: RANKED_CAUSES=[{'cause':'insufficient evidence or proportional response','evidence_score':0}]
_rolloff_rows=sorted({id(r):r for r,d in _valid_rows if r['gain']>0 and
    np.isfinite(r.get('k_small_signal',np.nan))}.values(), key=lambda r:r['gain'])
ROLLOFF_ONSET_GAIN=float('nan'); ROLLOFF_MAGNITUDE=float('nan')
for _row in _rolloff_rows:
    _ratio=_row['k_small_signal']/_row['gain']
    if np.isfinite(QUALIFICATION_K_PER_A) and _ratio < 0.95*QUALIFICATION_K_PER_A:
        ROLLOFF_ONSET_GAIN=_row['gain']; ROLLOFF_MAGNITUDE=1.0-_ratio/QUALIFICATION_K_PER_A; break
_moderate=[r for r in DIAG_SUMMARY if r.get('k_small_signal',r['k_vector'])<=0.55]
QUALIFICATION_ESTIMATORS_OK=bool(_moderate and all(r.get('agreement_ok',False) for r in _moderate))
def _drive_row_ok(_r):
    _ks=[d['k_paired'] for d in _r.get('drive_results',[])
         if d.get('receiver_repeat_ok') and np.isfinite(d.get('k_paired',np.nan))]
    return len(_ks)>=2 and np.ptp(_ks)<=0.015
QUALIFICATION_DRIVE_OK=bool(_moderate and all(_drive_row_ok(r) for r in _moderate))
_reliable=[r for r in DIAG_SUMMARY if r.get('converged') and r.get('agreement_ok')]
SAFE_FULL_MAP_GAIN=max([r['gain'] for r in _reliable]+[0.0])
SAFE_FULL_MAP_K=max([r.get('k_small_signal',r['k_vector']) for r in _reliable]+[0.0])
QUALIFICATION_SWEEP_SUITABLE=bool(QUALIFICATION_ESTIMATORS_OK and QUALIFICATION_DRIVE_OK and
    QUALIFICATION_HYSTERESIS_K<=0.01 and globals().get('DIAG_TEMPORAL_OK',False) and
    globals().get('DRIFT_VALID',False) and not _fit_signature and
    SAFE_FULL_MAP_GAIN>0.0 and not globals().get('HARD_STOP_REQUESTED',False) and
    (not RUN_HIGH_Q_EMULATOR_VALIDATION or globals().get('HIGH_Q_FILTER_VALID',False)) and
    globals().get('CRYOGENIC_BASELINE_READY',True))
HIGHEST_RELIABLE_Q=max([r['q'] for r in _reliable]+[float('nan')])
GENERATOR_RESOLUTION_HZ = FS_CLK/float(1 << 30)
Q_GRID_RESOLUTION_CEILING = F0/max(HIGH_Q_POINTS_PER_LW*GENERATOR_RESOLUTION_HZ,1e-30)
_reliable_records=[r.get('record') for r in _reliable if r.get('record') is not None]
_reliable_ppl=[r['measured'].get('receiver',{}).get(
    'points_per_linewidth',r['measured'].get('in_linewidth',0)) for r in _reliable_records]
MIN_RELIABLE_POINTS_PER_LW=min(_reliable_ppl) if _reliable_ppl else 0
HIGH_K_VECTOR_QUALIFIED=bool(any(r.get('converged') and
    r.get('k_small_signal',0)>=HIGH_K_PAIR_OPTIONAL and r.get('quadrature_available')
    for r in DIAG_SUMMARY))
RECOMMENDED_MAP_PHASE_RANGE_DEG=(0.0,360.0)
RECOMMENDED_MAP_MIN_POINTS_PER_LW=8
print('')
print('NON-PROPORTIONAL K(A) DIAGNOSIS: %s' % QUALIFICATION_DIAGNOSIS.upper())
print('  low-gain K/A             : %.4f' % QUALIFICATION_K_PER_A)
print('  maximum up/down dK       : %.4f' % QUALIFICATION_HYSTERESIS_K)
print('  roll-off onset/magnitude : A %s / %s' %
      (('%.4f'%ROLLOFF_ONSET_GAIN) if np.isfinite(ROLLOFF_ONSET_GAIN) else 'not resolved',
       ('%.1f%%'%(100*ROLLOFF_MAGNITUDE)) if np.isfinite(ROLLOFF_MAGNITUDE) else 'not resolved'))
print('  ranked explanations      : %s' % ', '.join(x['cause'] for x in RANKED_CAUSES))
print('  highest reliable limit   : A %.4f, K %.4f' % (SAFE_FULL_MAP_GAIN,SAFE_FULL_MAP_K))
print('  highest reliable Q       : %.1f' % HIGHEST_RELIABLE_Q)
print('  high-K vector qualified  : %s' % ('PASS' if HIGH_K_VECTOR_QUALIFIED else 'NOT YET'))
print('  minimum reliable pts/lw  : %d' % MIN_RELIABLE_POINTS_PER_LW)
print('  grid-resolution Q ceiling: %.3g (settling/SNR normally limit first)' % Q_GRID_RESOLUTION_CEILING)
print('  later map requirement    : full 0..360 deg; >=%d points/linewidth'
      % RECOMMENDED_MAP_MIN_POINTS_PER_LW)
print('  later full sweep suitable: %s' % ('PASS' if QUALIFICATION_SWEEP_SUITABLE else 'STOP'))

# Evidence-collection plots remain useful even when every high-gain fit is rejected.
_attempts = []
for _record in globals().get('DIAG_RECORDS', []):
    _attempts.extend(_record.get('attempts', []) or [_record])
if _attempts:
    fig, ax = plt.subplots(2, 2, figsize=(12.0, 8.0))
    _gains = sorted(set(float(r.get('gain', np.nan)) for r in _attempts if np.isfinite(r.get('gain',np.nan))))
    _drives = sorted(set(float(r.get('drive_frac', np.nan)) for r in _attempts if np.isfinite(r.get('drive_frac',np.nan))))
    _matrix = np.full((len(_drives), len(_gains)), np.nan)
    for iy, drv in enumerate(_drives):
        for ix, gain in enumerate(_gains):
            pts = [r for r in _attempts if abs(r.get('drive_frac',-1)-drv)<1e-12 and abs(r.get('gain',-1)-gain)<1e-12]
            if pts: _matrix[iy,ix] = np.mean([1.0 if r.get('valid') else 0.0 for r in pts])
    if _matrix.size:
        im=ax[0,0].imshow(_matrix, origin='lower', aspect='auto', vmin=0, vmax=1, cmap='RdYlGn')
        ax[0,0].set_xticks(range(len(_gains))); ax[0,0].set_xticklabels(['%.3g'%x for x in _gains])
        ax[0,0].set_yticks(range(len(_drives))); ax[0,0].set_yticklabels(['%.4f'%x for x in _drives])
        ax[0,0].set_xlabel('A'); ax[0,0].set_ylabel('probe drive'); ax[0,0].set_title('Gain/drive accepted fraction')
        fig.colorbar(im, ax=ax[0,0], label='valid fraction')
    for r in _attempts:
        ok=bool(r.get('valid')); reason=(r.get('stop_reason') or 'accepted')
        ax[0,1].plot(r.get('gain',np.nan), r.get('phase_deg',np.nan), 'o', mfc=(CLR_BLUE if ok else 'none'), mec=(CLR_BLUE if ok else CLR_RED))
        if not ok: ax[0,1].annotate(reason[:24], (r.get('gain',np.nan),r.get('phase_deg',np.nan)), fontsize=6, rotation=20)
    ax[0,1].set_xlabel('A'); ax[0,1].set_ylabel('phase / deg'); ax[0,1].set_title('Phase/gain failures (hollow = rejected)')
    _refs=[]
    for r in _attempts:
        for key in ('before','after'):
            if key in r: _refs.append(r[key])
    _refs=sorted(_refs,key=lambda x:x['time'])
    if _refs:
        _t0=_refs[0]['time']; _tm=np.asarray([(x['time']-_t0)/60.0 for x in _refs])
        _cent=np.asarray([x['fit']['f0'] for x in _refs]); _wid=np.asarray([x['fit']['lw'] for x in _refs])
        ax[1,0].plot(_tm,_cent-_cent[0],'o-',label='centre change / Hz')
        ax[1,0].plot(_tm,_wid-_wid[0],'s-',label='linewidth change / Hz')
        ax[1,0].legend(fontsize=8)
    ax[1,0].set_xlabel('chronological time / min'); ax[1,0].set_ylabel('change / Hz'); ax[1,0].set_title('Bracket references')
    _with_trace=[r for r in _attempts if 'measured' in r]
    for wanted, colour, label in ((True,CLR_BLUE,'highest valid'),(False,CLR_RED,'highest rejected')):
        candidates=[r for r in _with_trace if bool(r.get('valid'))==wanted]
        if candidates:
            tr=max(candidates,key=lambda r:r.get('gain',0))['measured']; x=(tr['f']-tr['fit']['f0'])/tr['fit']['lw']
            ax[1,1].plot(x,tr['p']/max(np.max(tr['p']),1e-30),'.-',ms=2,color=colour,label=label)
    ax[1,1].set_xlabel('(f-f0) / fitted linewidth'); ax[1,1].set_ylabel('normalised power'); ax[1,1].set_title('Highest valid and rejected lineshapes'); ax[1,1].legend(fontsize=8)
    plt.tight_layout(); plt.show()

# Residual shape and an independent width check for the highest valid trace.
_plot_records = [r for r in globals().get('DIAG_RECORDS', []) if r.get('valid')]
if _plot_records:
    _rr = max(_plot_records, key=lambda r: r.get('gain', 0.0))['measured']
    _x = (_rr['f'] - _rr['fit']['f0']) / _rr['fit']['lw']
    fig, (ar1, ar2) = plt.subplots(2, 1, figsize=(7.4, 5.4), sharex=True)
    ar1.plot(_x, _rr['p']/np.max(_rr['p']), '.', ms=3, label='measured')
    ar1.plot(_x, lorentzian(_rr['f'], _rr['fit'])/np.max(_rr['p']), '-',
             label='Lorentzian fit')
    _hp = _rr['fit_diagnostics']['half_power_lw']
    ar1.set_ylabel('normalised power'); ar1.legend(fontsize=8)
    ar1.set_title('Highest-gain accepted trace: fit %.2f kHz, half-power %s kHz'
                  % (_rr['fit']['lw']/1e3,
                     ('%.2f' % (_hp/1e3)) if _hp is not None else 'unavailable'))
    ar2.plot(_x, _rr['fit_diagnostics']['residual'], '.', ms=3)
    ar2.axhline(0.0, color='0.5', ls=':')
    ar2.set_xlabel('(f - f0) / fitted linewidth'); ar2.set_ylabel('residual / peak')
    plt.tight_layout(); plt.show()

# Optional manual VNA/signal-generator comparison; absence never gates the run.
_ext = EXTERNAL_MEASUREMENT
if np.isfinite(_ext.get('f0_hz',np.nan)) and np.isfinite(_ext.get('q',np.nan)):
    print('')
    print('MANUAL EXTERNAL COMPARISON: %s at %s K' %
          (_ext.get('instrument','external instrument'), _ext.get('temperature_k','?')))
    print('  centre: Red Pitaya %.6f MHz, external %.6f MHz, difference %+.1f Hz' %
          (F0/1e6,_ext['f0_hz']/1e6,F0-_ext['f0_hz']))
    print('  Q     : Red Pitaya %.2f, external %.2f, difference %+.2f' %
          (Q_SWEEP,_ext['q'],Q_SWEEP-_ext['q']))
    fig, ea=plt.subplots(1,2,figsize=(7.5,3.0))
    ea[0].errorbar([0,1],[F0,_ext['f0_hz']],yerr=[0,_ext.get('f0_unc_hz',0)],fmt='o')
    ea[1].errorbar([0,1],[Q_SWEEP,_ext['q']],yerr=[0,_ext.get('q_unc',0)],fmt='o')
    for a,title in zip(ea,('centre / Hz','Q')):
        a.set_xticks([0,1]); a.set_xticklabels(['Red Pitaya','external']); a.set_title(title)
    plt.tight_layout(); plt.show()
else:
    print('manual external VNA/signal-generator comparison: not supplied (informative only)')

BOUNDARY_TEXT = """

K=1 BOUNDARY CAMPAIGN
  highest diagnostic K : {k:.4f}
  K-estimator agreement: {agree}
  drive/receiver repeat: {conv}
  drift/scatter record  : {drift}
  physical loop gain    : {gainok}
  external safety checks: {external}
  first limiting factor: {limit}
  OUT1 protection margin: {margin:+.1f} dB
  interstage margin     : {imargin:+.1f} dB
  onset result         : {onset}
""".format(k=(highest['k_vector'] if highest else float('nan')),
           agree=('PASS' if DIAG_AGREEMENT_OK else 'STOP'),
           conv=('PASS' if DIAG_CONVERGENCE_OK else 'STOP'),
           drift=('PASS' if globals().get('DRIFT_VALID', False) else 'STOP'),
           gainok=('PASS' if transfer_ok else 'STOP'),
           external=('PASS' if HIGH_GAIN_CHAIN_CONFIRMED else 'STOP'),
           limit=FIRST_LIMITING_FACTOR, margin=protection_margin_db,
           imargin=interstage_margin_db,
           onset=(('gain %.5f +/- %.5f' % (ONSET_BRACKET['onset_gain'],
                                            ONSET_BRACKET['uncertainty_gain']))
                  if ONSET_BRACKET else 'NOT RUN - ' + BOUNDARY_STOP_REASON))
GAIN_PHASE_SWEEP_SUITABLE = bool(QUALIFICATION_SWEEP_SUITABLE)
SEO_REGION_SUITABLE = bool(GAIN_PHASE_SWEEP_SUITABLE and transfer_ok and
                           HIGH_GAIN_CHAIN_CONFIRMED)
print('')
print('SELF-CONTAINED CAMPAIGN VERDICT')
print('  repeated full gain/phase sweep : %s'
      % ('SUITABLE' if GAIN_PHASE_SWEEP_SUITABLE else 'NOT YET SUITABLE'))
print('  reliable approach to SEO region: %s'
      % ('SUITABLE' if SEO_REGION_SUITABLE else 'NOT YET SUITABLE'))
if not (QUALIFICATION_ESTIMATORS_OK and QUALIFICATION_DRIVE_OK):
    print('  limiting evidence: paired/vector/repeat/receiver convergence')
elif not globals().get('DRIFT_VALID', False):
    print('  limiting evidence: incomplete five minute qualification stability record')
elif not transfer_ok:
    print('  limiting evidence: measured K/A cannot reach K=1 inside A<=1.90')
elif not HIGH_GAIN_CHAIN_CONFIRMED:
    print('  SEO-only limitation: external power/spectrum/scope validation')
# Diagnostics are authoritative; rebuild the displayed/saved recommendation now.
REPORT_TEXT = _recommendation_text() + BOUNDARY_TEXT
print(BOUNDARY_TEXT)


In [ ]:
# --- Step 8e picture --------------------------------------------------------
if globals().get('sh_df') is None or len(sh_df) < 2:
    print('Step 8d/8e produced no data - nothing to plot.')
else:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.6, 4.0))
    a1.plot(sh_gain[_ord], sh_df[_ord] / 1e3, 'o-', color=CLR_BLUE)
    a1.plot(sh_gain[_ord], _slope * sh_gain[_ord] / 1e3, '--', color=CLR_GREY,
            label='%.0f Hz per unit gain' % _slope)
    a1.axhline(0.0, ls=':', color=CLR_GREY)
    a1.set_xlabel('signed feedback gain (negative = opposite phase)')
    a1.set_ylabel('centre shift / kHz')
    a1.set_title('(a)  range')
    a1.legend()

    _lwax = sh_df[_ord] / LW
    a2.plot(sh_gain[_ord], _lwax, 'o-', color=CLR_ORANGE)
    a2.axhline(0.0, ls=':', color=CLR_GREY)
    a2.set_xlabel('signed feedback gain')
    a2.set_ylabel('centre shift / linewidths')
    a2.set_title('(b)  in linewidths - the portable number')
    plt.show()

    print('Panel (b) is the fair comparison against a resonator of any Q: a shift of')
    print('one linewidth decouples as effectively at 40 MHz as at 740 kHz.')


---
# Step 9 — Close the loop on the instrument, and write down what was learned

**The self-test runs again.** If it passed at the start and passes now, nothing about the
instrument changed while the coil was being measured, so the numbers are about the coil. If it
passed then and fails now, the reason is in the board, not the physics.

**Then everything measured is written to a file**, so the next notebook does not have to be told
the answer by hand.


In [ ]:
# =============================================================================
#  STEP 9a - THE SELF-TEST, AGAIN
# =============================================================================
print('=' * 68)
print('STEP 9 - closing checks')
print('=' * 68)
print('re-running the instrument self-test...')
print('')
st1_after = instrument_selftest()
print('')
if st1_before['ok'] and st1_after['ok']:
    print('The instrument returned a known answer BEFORE and AFTER the session.')
    print('Nothing about the board changed while the coil was being measured.')
elif st1_before['ok'] and not st1_after['ok']:
    print('*** The self-test passed at the start and FAILS now. ***')
    print('Something changed during the session - a reload, a thermal problem, a')
    print('register left in an odd state. Treat every coil number below as')
    print('provisional until this passes again.')
else:
    print('*** The self-test was already failing. Nothing below is trustworthy. ***')


In [ ]:
# =============================================================================
#  STEP 9b - SUMMARY
# =============================================================================
def _g(name, default=None):
    """Fetch a result that an earlier step may have skipped."""
    v = globals().get(name, default)
    return default if v is None else v


_qa = globals().get('Q_ALT', None)
print('')
print('=' * 68)
print('BENCH COIL - SUMMARY')
print('=' * 68)
print('centre frequency          : %.6f MHz' % (F0 / 1e6))
print('linewidth                 : %.3f kHz' % (LW / 1e3))
print('quality factor, swept     : %.1f' % Q_SWEEP)
print('quality factor, alt. dec. : %s' % ('%.1f' % _qa if _qa else 'not obtained'))
print('drive used                : %.5f (linear: %s)' %
      (DRIVE_FRAC, globals().get('LINEAR_OK', 'not tested')))
_summary_drift = _g('DRIFT_HZ_PER_MIN', float('nan'))
print('drift                     : %s' %
      (('%+.1f Hz/min' % _summary_drift) if np.isfinite(_summary_drift)
       else 'not measured'))
print('narrowing phase           : %.1f deg (loop-specific)' % _g('PHI_NARROW', float('nan')))
print('shift phase               : %.1f deg (loop-specific)' % _g('PHI_SHIFT', float('nan')))
print('')
print(REPORT_TEXT)


In [ ]:
# =============================================================================
#  STEP 9c - WRITE RESULTS AND LEAVE THE HARDWARE SAFE
# =============================================================================
def _plain_record(record):
    """JSON-safe verification summary; full numeric traces go to the NPZ file."""
    out = {k: v for k, v in record.items()
           if k not in ('before', 'measured', 'after', 'baseline', 'attempts')}
    for key in ('before', 'measured', 'after'):
        if key in record:
            out[key] = {'time': float(record[key]['time']),
                        'drive_frac': float(record[key].get('drive_frac', record.get('drive_frac', float('nan')))),
                        'quality_error': record[key].get('quality_error', ''),
                        'fit': {name: float(record[key]['fit'][name])
                                for name in ('f0', 'lw', 'Q')}}
            for name in ('adc_peak_frac', 'in_linewidth', 'coefficients'):
                if name in record[key]:
                    value = record[key][name]
                    out[key][name] = (list(value) if isinstance(value, tuple) else value)
            if 'receiver' in record[key]:
                out[key]['receiver'] = dict(record[key]['receiver'])
            if 'settling_validation' in record[key]:
                out[key]['settling_validation'] = record[key]['settling_validation']
            if 'fit_diagnostics' in record[key]:
                out[key]['fit_diagnostics'] = {
                    name: record[key]['fit_diagnostics'].get(name)
                    for name in ('half_power_lw', 'half_power_disagreement',
                                 'structured_residual', 'floor_peak_amp',
                                 'residual_rms', 'residual_max',
                                 'complex_lw_disagreement', 'f0_uncertainty',
                                 'lw_uncertainty', 'q_uncertainty', 'jackknife_fits')}
                _cf = record[key]['fit_diagnostics'].get('complex_fit')
                if _cf is not None:
                    out[key]['fit_diagnostics']['complex_fit'] = {
                        name: float(_cf[name]) for name in ('f0', 'lw', 'Q', 'resid', 'pole_sign')}
    if 'attempts' in record:
        out['attempts'] = [_plain_record(attempt) for attempt in record['attempts']]
    if 'baseline' in record:
        out['baseline'] = {k: float(v) for k, v in record['baseline'].items()}
    return out


def _recommendation_json(record):
    return None if record is None else _plain_record(record)


def _plain_diagnostic(row):
    """Summary only; the embedded record and trace arrays belong in NPZ."""
    return {k: v for k, v in row.items() if k not in ('record',)}


def _json_safe(value):
    """Recursively convert numpy/Python values for the board's Python 3.5 JSON."""
    if isinstance(value, dict):
        return {str(k): _json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return [_json_safe(v) for v in value.tolist()]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


results = {
    'source_note': SOURCE_NOTE,
    'environment_profile': ENVIRONMENT_PROFILE,
    'external_measurement': dict(EXTERNAL_MEASUREMENT),
    'measured_utc': time.strftime('%Y-%m-%dT%H:%M:%S', time.gmtime()),
    'build_id': '0x%08X' % acq.build_id(),
    'coil': {'f0_hz': float(F0), 'linewidth_hz': float(LW),
             'q_swept': float(Q_SWEEP),
             'q_alt_dec': (float(_qa) if _qa else None)},
    'drive': {'frac': float(DRIVE_FRAC),
              'linear': bool(globals().get('LINEAR_OK', False)),
              'window': [float(x) for x in _g('DRIVE_WINDOW', (float('nan'),)*2)]},
    'stability': {'drift_hz_per_min': float(_g('DRIFT_HZ_PER_MIN', float('nan'))),
                  'scatter_hz': float(_g('SCATTER_HZ', float('nan'))),
                  'q_scatter_frac': float(_g('DRIFT_Q_SCATTER_FRAC', float('nan'))),
                  'observed_minutes': float(_g('DRIFT_OBSERVED_MINUTES', 0.0)),
                  'valid_for_boundary': bool(_g('DRIFT_VALID', False))},
    'receiver_validation': {'zero_if_tracking': True, 'fir_taps': int(FIR_TAPS),
        'points_per_linewidth_target': int(HIGH_Q_POINTS_PER_LW),
        'emulator_valid': bool(globals().get('HIGH_Q_FILTER_VALID', False)),
        'emulator_rows': globals().get('HIGH_Q_FILTER_ROWS', [])},
    'hardware': {'out1_pad_db': float(EXT_PAD_DB),
                 'interstage_pad_db': float(INTERSTAGE_PAD_DB),
                 'terminated_output': bool(TERMINATED_OUTPUT),
                 'measured_external_gain_db': float(EXT_GAIN_DB),
                 'gain_safety_model_db': float(EXT_GAIN_SAFETY_DB),
                 'high_gain_chain_confirmed': bool(
                     globals().get('HIGH_GAIN_CHAIN_CONFIRMED', False)),
                 'measured_gain_increase_db': float(
                     globals().get('MEASURED_GAIN_INCREASE_DB', float('nan'))),
                 'measured_second_input_dbm': float(
                     globals().get('MEASURED_SECOND_INPUT_DBM', float('nan'))),
                 'measured_final_output_dbm': float(
                     globals().get('MEASURED_FINAL_OUTPUT_DBM', float('nan'))),
                 'measured_worst_harmonic_dbc': float(
                     globals().get('MEASURED_WORST_HARMONIC_DBC', float('nan'))),
                 'measured_broadband_rise_db': float(
                     globals().get('MEASURED_BROADBAND_RISE_DB', float('nan'))),
                 'oscilloscope_spectrum_checks_required': True},
    'feedback_model': {
        'linewidth_model': 'B = B0 * (1 - K*cos(theta))',
        'q_model': 'Q = Q0 / (1 - K*cos(theta))',
        'shift_model': 'df = (B0/2) * K*sin(theta)',
        'k_per_gain_fitted': float(_g('k_slope', float('nan'))),
        'onset_gain_extrapolated': float(_g('A_ONSET', float('nan'))),
        'small_signal_fit': globals().get('K0_FIT', {'valid': False}),
        'measured_k_calibration': globals().get('K_CALIBRATION', {'valid': False}),
        'q_max_mapped_unverified': (float(np.nanmax(MAP_Q)) if 'MAP_Q' in globals() else None),
        'repeated_map_suitable': bool(globals().get('MAP_REPEAT_VALID', False)),
        'full_sweep_suitable': bool(globals().get('GAIN_PHASE_SWEEP_SUITABLE', False)),
        'seo_region_suitable': bool(globals().get('SEO_REGION_SUITABLE', False)),
        'qualification_diagnosis': globals().get('QUALIFICATION_DIAGNOSIS', ''),
        'ranked_causes': globals().get('RANKED_CAUSES', []),
        'rolloff_onset_gain': float(globals().get('ROLLOFF_ONSET_GAIN', float('nan'))),
        'rolloff_magnitude_fraction': float(globals().get('ROLLOFF_MAGNITUDE', float('nan'))),
        'highest_reliable_q': float(globals().get('HIGHEST_RELIABLE_Q', float('nan'))),
        'high_k_vector_qualified': bool(globals().get('HIGH_K_VECTOR_QUALIFIED', False)),
        'minimum_reliable_points_per_linewidth': int(globals().get('MIN_RELIABLE_POINTS_PER_LW', 0)),
        'generator_resolution_hz': float(globals().get('GENERATOR_RESOLUTION_HZ', float('nan'))),
        'grid_resolution_q_ceiling': float(globals().get('Q_GRID_RESOLUTION_CEILING', float('nan'))),
        'high_k_pair_optional_threshold': float(HIGH_K_PAIR_OPTIONAL),
        'qualification_k_per_a': float(globals().get('QUALIFICATION_K_PER_A', float('nan'))),
        'qualification_hysteresis_k': float(globals().get('QUALIFICATION_HYSTERESIS_K', float('nan'))),
        'safe_full_map_gain': float(globals().get('SAFE_FULL_MAP_GAIN', float('nan'))),
        'safe_full_map_k': float(globals().get('SAFE_FULL_MAP_K', float('nan'))),
        'map_repeat_cv95': float(globals().get('_map_cv95', float('nan'))),
        'map_reference_q_cv': float(globals().get('_ref_q_cv', float('nan'))),
        'shift_max_swept_hz': float(_g('SHIFT_MAX', float('nan'))),
    },
    'recommendations': {
        'maximum_q': _recommendation_json(Q_RECOMMENDATION),
        'maximum_shift': _recommendation_json(SHIFT_RECOMMENDATION),
        'stop_reason': FIRST_ORDER_STOP_REASON,
    },
    'verification': [_plain_record(r) for r in VERIFY_RECORDS],
    'boundary_campaign': {
        'diagnostics': [_plain_diagnostic(r)
                        for r in globals().get('DIAG_SUMMARY', [])],
        'measurements': [_plain_record(r) for r in globals().get('DIAG_RECORDS', [])],
        'agreement_ok': bool(globals().get('DIAG_AGREEMENT_OK', False)),
        'convergence_ok': bool(globals().get('DIAG_CONVERGENCE_OK', False)),
        'repeat_ok': bool(globals().get('DIAG_REPEAT_OK', False)),
        'drift_ok': bool(globals().get('DRIFT_VALID', False)),
        'full_campaign_temporal_ok': bool(globals().get('DIAG_TEMPORAL_OK', False)),
        'full_campaign_minutes': float(globals().get('DIAG_DRIFT_COVERAGE_MIN', 0.0)),
        'physical_gain_ok': bool(globals().get('transfer_ok', False)),
        'amplifier_safety_ok': bool(globals().get('SAFETY_GATE_OK', False)),
        'first_limiting_factor': globals().get('FIRST_LIMITING_FACTOR', ''),
        'safety_gate_ok': bool(globals().get('SAFETY_GATE_OK', False)),
        'stop_reason': globals().get('BOUNDARY_STOP_REASON', ''),
        'onset_bracket': globals().get('ONSET_BRACKET', None),
        'onset_measurements': globals().get('ONSET_RECORDS', []),
    },
    'selftest': {'before_ok': bool(st1_before['ok']), 'after_ok': bool(st1_after['ok'])},
}


def _pad_rows(rows):
    rows = [np.asarray(row, dtype=float) for row in rows]
    width = max([len(row) for row in rows] + [0])
    out = np.full((len(rows), width), np.nan)
    for i, row in enumerate(rows):
        out[i, :len(row)] = row
    return out


def _attempt_records(records):
    """Flatten retry wrappers so rejected attempts retain their raw traces."""
    out = []
    for record in records:
        attempts = record.get('attempts', [])
        out.extend(attempts if attempts else [record])
    return out


_DIAG_ATTEMPTS = _attempt_records(globals().get('DIAG_RECORDS', []))
CAMPAIGN_END_STATE = ('hard safety stop' if globals().get('HARD_STOP_REQUESTED', False)
                      else 'normal completion')
results['boundary_campaign']['campaign_end_state'] = CAMPAIGN_END_STATE
results['boundary_campaign']['hard_stop_record'] = _recommendation_json(
    globals().get('HARD_STOP_RECORD', None))
results['boundary_campaign']['attempt_count'] = len(_DIAG_ATTEMPTS)
REPORT_TEXT += ('\nCAMPAIGN COMPLETION\n  state: %s\n  attempted records: %d\n'
                % (CAMPAIGN_END_STATE, len(_DIAG_ATTEMPTS)))


_save_errors = []
try:
    try:
        _json_tmp = RESULTS_JSON + '.tmp'
        with open(_json_tmp, 'w') as fh:
            json.dump(_json_safe(results), fh, indent=2, sort_keys=True, allow_nan=False)
        os.replace(_json_tmp, RESULTS_JSON)
        print('results written to %s' % os.path.abspath(RESULTS_JSON))
    except Exception as exc:
        _save_errors.append('JSON: %s' % exc)

    try:
        _npz_target = 'bench_coil_6db_traces.npz'
        _npz_tmp = _npz_target + '.tmp.npz'
        np.savez_compressed(
        _npz_tmp,
        map_gain=np.asarray(globals().get('map_gain', []), dtype=float),
        map_phase_deg=np.asarray(globals().get('map_phi', []), dtype=float),
        map_q=np.asarray(globals().get('MAP_Q', []), dtype=float),
        map_shift_hz=np.asarray(globals().get('MAP_DF', []), dtype=float),
        map_oscillating=np.asarray(globals().get('MAP_OSC', []), dtype=bool),
        map_q_repeats=np.asarray(globals().get('MAP_Q_REPEATS', []), dtype=float),
        map_shift_repeats_hz=np.asarray(globals().get('MAP_DF_REPEATS', []), dtype=float),
        map_reference_q=np.asarray(globals().get('MAP_REFERENCE_Q', []), dtype=float),
        paired_gain=np.asarray(globals().get('gains_ok', []), dtype=float),
        paired_k=np.asarray(globals().get('ks', []), dtype=float),
        shift_gain=np.asarray(globals().get('sh_gain', []), dtype=float),
        shift_hz=np.asarray(globals().get('sh_df', []), dtype=float),
        lineshape_frequency_hz=_pad_rows(globals().get('spec_f', [])),
        lineshape_power=_pad_rows(globals().get('spec_p', [])),
        verify_frequency_hz=_pad_rows([
            r[key]['f'] for r in VERIFY_RECORDS for key in ('before', 'measured', 'after')
            if r.get('valid')]),
        diagnostic_gain=np.asarray([r.get('gain',np.nan) for r in globals().get('DIAG_SUMMARY', [])], dtype=float),
        diagnostic_k_q=np.asarray([r.get('k_q',np.nan) for r in globals().get('DIAG_SUMMARY', [])], dtype=float),
        diagnostic_k_paired=np.asarray([r.get('k_paired',np.nan) for r in globals().get('DIAG_SUMMARY', [])], dtype=float),
        diagnostic_k_vector=np.asarray([r.get('k_vector',np.nan) for r in globals().get('DIAG_SUMMARY', [])], dtype=float),
        diagnostic_k_complex=np.asarray([r.get('k_complex', np.nan) for r in globals().get('DIAG_SUMMARY', [])], dtype=float),
        diagnostic_k_joint=np.asarray([r.get('k_joint', np.nan) for r in globals().get('DIAG_SUMMARY', [])], dtype=float),
        diagnostic_k_small_signal=np.asarray([r.get('k_small_signal', np.nan) for r in globals().get('DIAG_SUMMARY', [])], dtype=float),
        diagnostic_q_small_signal=np.asarray([r.get('q_small_signal', np.nan) for r in globals().get('DIAG_SUMMARY', [])], dtype=float),
        diagnostic_phase_deg=np.asarray([r.get('phase_deg',np.nan) for r in globals().get('DIAG_SUMMARY', [])], dtype=float),
        diagnostic_frequency_hz=_pad_rows([
            r[key]['f'] for r in _DIAG_ATTEMPTS
            for key in ('before', 'measured', 'after') if key in r]),
        diagnostic_power=_pad_rows([
            r[key]['p'] for r in _DIAG_ATTEMPTS
            for key in ('before', 'measured', 'after') if key in r]),
        diagnostic_complex_real=_pad_rows([
            np.real(r[key]['z']) for r in _DIAG_ATTEMPTS
            for key in ('before', 'measured', 'after') if key in r]),
        diagnostic_complex_imag=_pad_rows([
            np.imag(r[key]['z']) for r in _DIAG_ATTEMPTS
            for key in ('before', 'measured', 'after') if key in r]),
        diagnostic_timestamp_s=np.asarray([
            r[key]['time'] for r in _DIAG_ATTEMPTS
            for key in ('before', 'measured', 'after') if key in r], dtype=float),
        diagnostic_fit_residual=_pad_rows([
            r[key]['fit_diagnostics']['residual']
            for r in _DIAG_ATTEMPTS
            for key in ('before', 'measured', 'after') if key in r]),
        verify_power=_pad_rows([
            r[key]['p'] for r in VERIFY_RECORDS for key in ('before', 'measured', 'after')
            if r.get('valid')]),
        verify_complex_real=_pad_rows([
            np.real(r[key]['z']) for r in VERIFY_RECORDS
            for key in ('before', 'measured', 'after') if r.get('valid')]),
        verify_complex_imag=_pad_rows([
            np.imag(r[key]['z']) for r in VERIFY_RECORDS
            for key in ('before', 'measured', 'after') if r.get('valid')]),
        )
        os.replace(_npz_tmp, _npz_target)
        print('raw traces written to %s' % os.path.abspath(_npz_target))
    except Exception as exc:
        _save_errors.append('NPZ: %s' % exc)

    try:
        _report_target = 'bench_coil_6db_report.md'
        _report_tmp = _report_target + '.tmp'
        with open(_report_tmp, 'w') as fh:
            fh.write('# Bench coil first-order optimisation\n\n' + REPORT_TEXT + '\n')
        os.replace(_report_tmp, _report_target)
        print('report written to %s' % os.path.abspath(_report_target))
    except Exception as exc:
        _save_errors.append('report: %s' % exc)
finally:
    gen.off()
    ps.off()
    ps.monitor_off()
    emu.disable()
    acq.stop()

if _save_errors:
    print('WARNING - one or more outputs failed independently:')
    for _err in _save_errors:
        print('  ' + _err)

print('generator off, feedback off, emulator disabled, acquisition stopped.')
print('The board is back to its power-on behaviour.')


---
# What to do with this

**If the two quality factors agree**, you have a characterised coil and `bench_coil_6db_results.json`
holds everything a later notebook needs — Stage 10's atlas can be pointed at it with
`EMULATION = False`.

**If they disagree**, in order of how often each is the culprit: the **fitted baseline** (if the
floor in the Step 3 plot is a sizeable fraction of the peak the width is inflated — more drive or
a shorter pickup lead fixes it); **something in the band** (compare the Step 2 survey with and
without the coil); **compression**, which Step 6 tests directly; and **the drive**, where Step 4b
says which values give a trustworthy fit and raising `N_AVG` widens the window from the bottom.

**Things deliberately not done here**, and why:

| | |
|---|---|
| Walking into self-oscillation | Saturates the input, and a saturated input still looks vaguely like a resonance. Worth doing, but as its own experiment with the level watched continuously |
| Varying the coupling | The capacitors are soldered on. Step 6's ladder is the substitute; the real test needs a second pair |
| Anything needing an ion | The dip, resistive cooling, ion temperature and the phase-sensitive technique all wait for the trap |
| A servo during the scan | Step 7 says whether one is needed |

**Run the self-test cell whenever a result surprises you.** Ten seconds, no recabling, and it
separates "the coil did something interesting" from "the instrument moved".
